# NIHONGA ? Phase 3: Identify Layers to Remove

**Phase registered:** `Phase 3 ? Identify Layers to Remove` (see [PLAN.md](PLAN.md#phase-3--identify-layers-to-remove)).

This phase will measure which transformer blocks appear redundant before any model is modified. We will begin conservatively, using the frozen Phase 2 evaluation conditions and locally saved artifacts so previously completed work is reused on reruns. Candidate importance signals include activation similarity across consecutive blocks, hidden-state cosine similarity, and the output change caused by bypassing one block. The resulting measurements will support selecting groups of layers for later pruning; this notebook will not prune or alter model weights.

This opening cell records the scope and safeguards for Phase 3.

## 1. Reuse the frozen Phase 2 model identity

Before inspecting any layers, we need to bind this analysis to the exact model used for the baseline. Phase 2 saved the model repository and immutable revision in `data/phase2/manifest.json` and repeated the evaluation settings in `data/phase2/protocol.json`. We will read those local files as the source of truth, then record their identities in the Phase 3 artifacts. This keeps the analysis reproducible and avoids a model download or any inference just to identify the baseline.

In [ ]:
# Read the frozen Phase 2 identity and persist Phase 3 provenance once
import hashlib
import json
from pathlib import Path

PHASE2_DIR = Path("data/phase2")
PHASE3_DIR = Path("data/phase3")
PHASE3_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = PHASE2_DIR / "manifest.json"
protocol_path = PHASE2_DIR / "protocol.json"
for required_path in (manifest_path, protocol_path):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required Phase 2 artifact is missing: {required_path}")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
model = manifest.get("model", {})
repo = model.get("repo") or protocol.get("MODEL_REPO")
revision = model.get("revision") or protocol.get("MODEL_REVISION")
if not repo or not revision:
    raise ValueError("Phase 2 artifacts do not identify both the model repository and immutable revision.")
if protocol.get("MODEL_REPO") not in (None, repo) or protocol.get("MODEL_REVISION") not in (None, revision):
    raise ValueError("Phase 2 manifest and protocol identify different model revisions.")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

expected = {"model_repo": repo, "model_revision": revision,
            "phase2_manifest_sha256": sha256_file(manifest_path),
            "phase2_protocol_sha256": sha256_file(protocol_path)}
provenance_path = PHASE3_DIR / "provenance.json"
if provenance_path.exists():
    provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
    if any(provenance.get(key) != value for key, value in expected.items()):
        raise RuntimeError(f"Existing provenance does not match current Phase 2 inputs: {provenance_path}")
    print(f"Reusing verified provenance: {provenance_path}")
else:
    provenance = {"version": "phase3-provenance-v1", **expected}
    provenance_path.write_text(json.dumps(provenance, indent=2) + "\n", encoding="utf-8")
    print(f"Saved provenance: {provenance_path}")

print(f"Model: {repo}@{revision}")
print(f"Baseline protocol: {protocol.get('PROTOCOL_VERSION', 'unspecified')}")

## 2. Inspect the saved transformer configuration

Layer importance measurements need the actual transformer depth and block layout for the pinned model. We first reuse `data/phase3/architecture.json` and `model_config.json`, checking their model identity, transformer component path, configuration contents, and depth summary. When the Modal preflight report is present, its verified source-configuration checksum must also match the architecture record. These checks require neither a local model cache nor a GPU or network call.

Only when both saved architecture files are absent do we inspect the exact pinned transformer configuration in the local Hugging Face cache and save the two artifacts. Incomplete or inconsistent saved artifacts stop this step without overwriting anything. The stored source checksum refers to the original Hugging Face configuration bytes; the locally formatted JSON snapshot is checked by its parsed contents rather than compared against that original byte checksum. Reading this metadata does not load model weights or execute inference.


In [1]:
# Reuse saved architecture metadata before inspecting the local model cache.
import hashlib
import json
import os
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
provenance = json.loads((PHASE3_DIR / "provenance.json").read_text(encoding="utf-8"))
repo = provenance["model_repo"]
revision = provenance["model_revision"]
config_snapshot = PHASE3_DIR / "model_config.json"
architecture_path = PHASE3_DIR / "architecture.json"
depth_keys = ("num_layers", "num_hidden_layers", "depth", "transformer_layers", "layers")

def describe_depth(config):
    fields = {key: config[key] for key in depth_keys
              if key in config and isinstance(config[key], (int, float, list, dict))}
    layers = config.get("num_layers")
    if type(layers) is not int or layers <= 0:
        raise ValueError("Transformer configuration must declare a positive integer num_layers.")
    return fields

if architecture_path.exists() or config_snapshot.exists():
    if not (architecture_path.is_file() and config_snapshot.is_file()):
        raise RuntimeError("Phase 3 architecture artifacts are incomplete; preserve and inspect them.")
    architecture_record = json.loads(architecture_path.read_text(encoding="utf-8"))
    config = json.loads(config_snapshot.read_text(encoding="utf-8"))
    depth_fields = describe_depth(config)
    expected_metadata = {
        "version": "phase3-architecture-config-v1",
        "model_repo": repo, "model_revision": revision,
        "config_relative_path": "transformer/config.json",
        "config": config, "depth_fields": depth_fields,
    }
    if any(architecture_record.get(key) != value for key, value in expected_metadata.items()):
        raise RuntimeError("Saved architecture metadata differs from provenance or the saved transformer config.")
    config_sha256 = architecture_record.get("config_sha256", "")
    if (not isinstance(config_sha256, str) or len(config_sha256) != 64
            or any(c not in "0123456789abcdef" for c in config_sha256)):
        raise RuntimeError("Saved source-configuration checksum is invalid.")
    preflight_path = PHASE3_DIR / "modal_preflight.json"
    if preflight_path.is_file():
        preflight = json.loads(preflight_path.read_text(encoding="utf-8"))
        expected = preflight.get("request", {})
        if any(expected.get(key) != value for key, value in {
                "model_repo": repo, "model_revision": revision,
                "config_sha256": config_sha256, "num_layers": config["num_layers"]}.items()):
            raise RuntimeError("Saved Modal preflight and architecture metadata disagree.")
        if preflight.get("status") != "passed":
            raise RuntimeError("Saved Modal preflight did not pass; inspect its recorded errors.")
    print(f"Reusing verified architecture artifacts in {PHASE3_DIR}; no local model cache needed.")
else:
    repo_cache_name = "models--" + repo.replace("/", "--")
    hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
    hub_cache = Path(os.environ.get("HF_HUB_CACHE", hf_home / "hub"))
    config_path = hub_cache / repo_cache_name / "snapshots" / revision / "transformer" / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(
            f"No saved architecture artifacts or pinned local transformer config ({config_path}). "
            "No download or Modal call was attempted."
        )
    config_bytes = config_path.read_bytes()
    config = json.loads(config_bytes)
    depth_fields = describe_depth(config)
    config_sha256 = hashlib.sha256(config_bytes).hexdigest()
    architecture_record = {
        "version": "phase3-architecture-config-v1",
        "model_repo": repo, "model_revision": revision,
        "config_relative_path": "transformer/config.json",
        "config_sha256": config_sha256, "depth_fields": depth_fields, "config": config,
    }
    # Exclusive creation avoids overwriting evidence if artifacts appeared in the meantime.
    with config_snapshot.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(config, indent=2, ensure_ascii=False) + "\n")
    with architecture_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(architecture_record, indent=2, ensure_ascii=False) + "\n")
    print(f"Saved pinned transformer config snapshot and architecture summary in {PHASE3_DIR}")

print(f"Config: {architecture_record['config_relative_path']} (source sha256 {config_sha256})")
print(f"Common depth fields: {depth_fields}")


Reusing verified architecture artifacts in data\phase3; no local model cache needed.
Config: transformer/config.json (source sha256 56ae3281c4e6c2d1aa3658252d187488071815fd79bef15808bb0205fc1a2241)
Common depth fields: {'num_layers': 32}


## 3. Freeze a reproducible layer-analysis probe set

Layer bypass and hidden-state comparisons must use identical prompts, seeds, resolutions, and sampling settings. We will derive a small diagnostic subset from the already frozen Phase 2 internal evaluation set, stratifying deterministically across its capability dimensions and difficulty levels. Each selected row will retain its original generation conditions and metadata. We will serialize the selected rows and their source-file hash under `data/phase3/`; reruns will reuse the saved subset and verify that it still matches the Phase 2 source. This selection step makes no model calls and does not alter the baseline evaluation set.

In [ ]:
# Build and persist a deterministic stratified probe set from Phase 2
import hashlib
import json
from collections import defaultdict
from pathlib import Path

PHASE2_EVAL_PATH = Path("data/phase2/internal_eval_set.jsonl")
PHASE3_DIR = Path("data/phase3")
PROBE_PATH = PHASE3_DIR / "layer_probe_set.jsonl"
PROBE_MANIFEST_PATH = PHASE3_DIR / "layer_probe_manifest.json"
PROBE_LIMIT = 64
PROBE_SALT = "nihonga-phase3-layer-probe-v1"
if not PHASE2_EVAL_PATH.is_file():
    raise FileNotFoundError(f"Frozen Phase 2 internal evaluation set not found: {PHASE2_EVAL_PATH}")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_sha256 = sha256_file(PHASE2_EVAL_PATH)
rows = [json.loads(line) for line in PHASE2_EVAL_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
if not rows:
    raise ValueError("Phase 2 internal evaluation set is empty.")
required_fields = {"eval_id", "prompt", "seed", "width", "height", "num_inference_steps", "true_cfg_scale"}
missing = sorted(required_fields.difference(rows[0]))
if missing:
    raise ValueError(f"Phase 2 evaluation rows are missing required conditions: {missing}")

groups = defaultdict(list)
for row in rows:
    key = (str(row.get("dimension", "unknown")), str(row.get("difficulty", "unknown")))
    groups[key].append(row)
for key in groups:
    groups[key].sort(key=lambda row: hashlib.sha256(f"{PROBE_SALT}:{row['eval_id']}".encode()).hexdigest())

selected = []
ordered_strata = sorted(groups)
while len(selected) < min(PROBE_LIMIT, len(rows)):
    added_this_round = False
    for key in ordered_strata:
        already_selected = sum(1 for row in selected if (str(row.get("dimension", "unknown")), str(row.get("difficulty", "unknown"))) == key)
        if already_selected < len(groups[key]):
            selected.append(groups[key][already_selected])
            added_this_round = True
            if len(selected) >= min(PROBE_LIMIT, len(rows)):
                break
    if not added_this_round:
        break

probe_manifest = {
    "version": "phase3-layer-probe-v1",
    "source_path": PHASE2_EVAL_PATH.as_posix(),
    "source_sha256": source_sha256,
    "selection_salt": PROBE_SALT,
    "selection_method": "sha256-ranked round-robin across (dimension, difficulty) strata",
    "requested_max_rows": PROBE_LIMIT,
    "selected_rows": len(selected),
    "strata_counts": {f"{dimension}|difficulty={difficulty}": sum(1 for row in selected if str(row.get("dimension", "unknown")) == dimension and str(row.get("difficulty", "unknown")) == difficulty) for dimension, difficulty in ordered_strata},
    "eval_ids": [row["eval_id"] for row in selected],
}

PHASE3_DIR.mkdir(parents=True, exist_ok=True)
if PROBE_PATH.exists() or PROBE_MANIFEST_PATH.exists():
    if not (PROBE_PATH.is_file() and PROBE_MANIFEST_PATH.is_file()):
        raise RuntimeError("Probe artifacts are incomplete; inspect them before rerunning.")
    saved_manifest = json.loads(PROBE_MANIFEST_PATH.read_text(encoding="utf-8"))
    saved_rows = [json.loads(line) for line in PROBE_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    if saved_manifest != probe_manifest or saved_rows != selected:
        raise RuntimeError("Saved probe set differs from the current frozen Phase 2 source or deterministic selection.")
    print(f"Reusing verified probe set: {len(saved_rows)} rows from {PROBE_PATH}")
else:
    PROBE_PATH.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in selected), encoding="utf-8")
    PROBE_MANIFEST_PATH.write_text(json.dumps(probe_manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"Saved deterministic probe set: {len(selected)} rows to {PROBE_PATH}")

print(f"Source SHA-256: {source_sha256}")
print(f"Strata represented: {len(ordered_strata)}; probe cap: {PROBE_LIMIT}")

## 4. Verify the Modal analysis runtime and cached model weights

Our local notebook coordinates the experiments and reads saved results; GPU analysis runs on Modal. We reuse Phase 2's pinned inference packages and `nihonga-hf-cache` volume, requesting `H100!` to prevent automatic GPU upgrades. The verified Qwen-Image-2.1 transformer has **32 blocks**, so pruning candidates must use that actual depth rather than the illustrative 60-block counts in PLAN.md.

This preflight checks CUDA and BF16 support, package versions, the pinned transformer configuration, and the presence of every weight shard referenced by the transformer, text encoder, and VAE indexes. It records file sizes without loading model weights or generating images. The weights cache is mounted read-only and Hugging Face networking is disabled inside the worker: missing files are reported, never downloaded.

We save the report to `nihonga-phase3-artifacts` before returning it and then save a local copy under `data/phase3/`. Rerunning with the same inputs reads the local report without any Modal call. If only the local copy is missing, we recover the remote report before allocating a GPU. A saved failed preflight also stops on reruns, preserving the evidence rather than silently repeating the work. This file inventory is a readiness check; loading and probing the model belongs to a separate reviewed step.


In [1]:
# Inspect the remote runtime once; reuse local or remote reports before scheduling a GPU.
import hashlib
import json
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
provenance = json.loads((PHASE3_DIR / "provenance.json").read_text(encoding="utf-8"))
architecture = json.loads((PHASE3_DIR / "architecture.json").read_text(encoding="utf-8"))
protocol_path = Path("data/phase2/protocol.json")
protocol_bytes = protocol_path.read_bytes()
if hashlib.sha256(protocol_bytes).hexdigest() != provenance["phase2_protocol_sha256"]:
    raise RuntimeError("Phase 2 protocol changed since Phase 3 provenance was frozen.")
protocol = json.loads(protocol_bytes)
if any(architecture[k] != provenance[k] for k in ("model_repo", "model_revision")):
    raise RuntimeError("Architecture and provenance identify different models.")

request = {
    "version": "phase3-modal-preflight-v1",
    "model_repo": provenance["model_repo"],
    "model_revision": provenance["model_revision"],
    "config_sha256": architecture["config_sha256"],
    "num_layers": architecture["config"]["num_layers"],
    "packages": protocol["GEN_PACKAGES"],
    "python_version": "3.13",
    "gpu_requested": "H100!",
    "weights_volume": "nihonga-hf-cache",
    "results_volume": "nihonga-phase3-artifacts",
}
request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
report_path = PHASE3_DIR / "modal_preflight.json"
remote_path = f"/preflight/{request_key}.json"

if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(f"Reusing local report: {report_path}; no Modal call.")
else:
    import modal

    results_volume = modal.Volume.from_name(request["results_volume"], create_if_missing=True)
    try:
        report = json.loads(b"".join(results_volume.read_file(remote_path)))
        print("Recovered saved remote report; no GPU scheduled.")
    except FileNotFoundError:
        app = modal.App("nihonga-phase3-preflight")
        weights_volume = modal.Volume.from_name(request["weights_volume"])
        image = (modal.Image.debian_slim(python_version=request["python_version"])
                 .apt_install("git").pip_install(*request["packages"])
                 .env({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}))

        @app.function(image=image, gpu=request["gpu_requested"], timeout=600,
                      max_containers=1, retries=0, serialized=True,
                      volumes={"/root/.cache/huggingface": weights_volume.with_mount_options(read_only=True),
                               "/phase3": results_volume})
        def inspect_runtime(expected, key):
            import hashlib
            import importlib.metadata
            import json
            import platform
            from datetime import datetime, timezone
            from pathlib import Path
            import torch
            from diffusers import QwenImage21Pipeline, QwenImage21Transformer2DModel

            saved = Path("/phase3/preflight") / f"{key}.json"
            if saved.exists():
                return json.loads(saved.read_text())
            errors = []
            snapshot = (Path("/root/.cache/huggingface/hub") /
                        ("models--" + expected["model_repo"].replace("/", "--")) /
                        "snapshots" / expected["model_revision"])
            config_path = snapshot / "transformer/config.json"
            if not config_path.is_file():
                errors.append("Pinned transformer configuration is missing.")
            else:
                config_bytes = config_path.read_bytes()
                if hashlib.sha256(config_bytes).hexdigest() != expected["config_sha256"]:
                    errors.append("Transformer configuration checksum differs from local architecture.")
                if json.loads(config_bytes).get("num_layers") != expected["num_layers"]:
                    errors.append("Transformer depth differs from local architecture.")

            inventory = {}
            for component in ("transformer", "text_encoder", "vae"):
                directory = snapshot / component
                indexes = sorted(directory.glob("*.safetensors.index.json"))
                referenced = set()
                for index in indexes:
                    referenced.update(json.loads(index.read_text())["weight_map"].values())
                if not indexes:
                    referenced.update(p.name for p in directory.glob("*.safetensors"))
                if not referenced:
                    errors.append(f"No safetensors weights found for {component}.")
                files = []
                for name in sorted(referenced):
                    weight_path = directory / name
                    present = weight_path.is_file() and weight_path.stat().st_size > 0
                    files.append({"name": name, "present": present,
                                  "bytes": weight_path.stat().st_size if present else 0})
                    if not present:
                        errors.append(f"Missing or empty weight shard: {component}/{name}")
                inventory[component] = {"indexes": [p.name for p in indexes], "files": files,
                                        "total_bytes": sum(f["bytes"] for f in files)}

            available = torch.cuda.is_available()
            gpu = {}
            if available:
                props = torch.cuda.get_device_properties(0)
                gpu = {"name": props.name, "total_bytes": props.total_memory,
                       "compute_capability": [props.major, props.minor],
                       "bf16_supported": torch.cuda.is_bf16_supported()}
                if "H100" not in props.name:
                    errors.append("Allocated GPU is not the requested H100.")
                if not gpu["bf16_supported"]:
                    errors.append("GPU does not support BF16.")
            else:
                errors.append("CUDA is unavailable.")
            measured = {"request": expected, "request_key": key,
                        "recorded_utc": datetime.now(timezone.utc).isoformat(),
                        "python": platform.python_version(), "cuda": torch.version.cuda,
                        "packages": {p: importlib.metadata.version(p) for p in
                                     ("torch", "torchvision", "transformers", "diffusers", "accelerate", "pillow")},
                        "gpu": gpu, "weights": inventory,
                        "status": "failed" if errors else "passed", "errors": errors}
            saved.parent.mkdir(parents=True, exist_ok=True)
            temporary = saved.with_suffix(".tmp")
            temporary.write_text(json.dumps(measured, indent=2) + "\n")
            temporary.replace(saved)
            results_volume.commit()
            return measured

        with modal.enable_output(), app.run():
            report = inspect_runtime.remote(request, request_key)

    if report.get("request") != request or report.get("request_key") != request_key:
        raise RuntimeError("Remote report does not match this preflight request.")
    temporary = report_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    temporary.replace(report_path)
    print(f"Saved local report: {report_path}")

if report.get("request") != request or report.get("request_key") != request_key:
    raise RuntimeError("Saved preflight belongs to different inputs; preserve it and investigate.")
if report["status"] != "passed":
    raise RuntimeError("Saved preflight failed: " + "; ".join(report["errors"]))
print(f"Preflight passed: {report['gpu']['name']}; {request['num_layers']} transformer blocks.")
for component, entry in report["weights"].items():
    print(f"  {component}: {len(entry['files'])} weight files, {entry['total_bytes'] / 2**30:.2f} GiB")
print("No model weights loaded; no inference performed.")


✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-fMfrEBMfn3TYr6NUxGmVA5
Building image im-lNwhzktfTuxuQOM89s52pa

=> Step 0: FROM base

=> Step 1: ENV HF_HUB_OFFLINE=1

=> Step 2: ENV TRANSFORMERS_OFFLINE=1
Saving image...
Image saved, took 1.10s

Built image im-lNwhzktfTuxuQOM89s52pa in 1.74s


✓ Created objects.
└── 🔨 Created function inspect_runtime.
Stopping app - local entrypoint completed.
✓ App completed. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-fMfrEBMfn3TYr6NUxGmVA5
Saved local report: data\phase3\modal_preflight.json
Preflight passed: NVIDIA H100 80GB HBM3; 32 transformer blocks.
  transformer: 2 weight files, 13.25 GiB
  text_encoder: 4 weight files, 16.33 GiB
  vae: 1 weight files, 1.26 GiB
No model weights loaded; no inference performed.


## 5. Load the pinned model on Modal and generate one smoke image

The remote preflight verified files and hardware; this step confirms the complete pipeline can actually load and generate an image. We use the same pinned BF16 model and inference packages as Phase 2 on an H100 80GB, with the existing weights volume mounted read-only and Hugging Face offline mode enabled. We check the loaded transformer block count and scheduler configuration against saved metadata, then generate the first frozen probe prompt using its original seed, resolution, step count, guidance scale, and Phase 2 noise device. No layers are removed.

We record model-load time, first-image generation time, and peak CUDA memory allocated and reserved. The first-image timing includes initialization and warmup, so it is a readiness measurement rather than a steady-state speed benchmark. CUDA memory statistics describe the PyTorch allocator and are not total device utilization.

The worker saves a single durable packet containing the lossless PNG and measurements to `nihonga-phase3-artifacts` before returning; the notebook saves the image and report under `data/phase3/smoke/`. A matching local result is verified and reused without any Modal call. Missing local files are recovered from the remote packet before requesting a GPU. Failed runs are also saved and stop on reruns, preventing automatic repeats of expensive work. The probe-file hash and generation settings are part of the result identity so incompatible artifacts cannot be silently reused.


In [1]:
# Load and exercise the pinned pipeline once on Modal, preserving every completed result.
import base64
import hashlib
import json
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
preflight = json.loads((PHASE3_DIR / "modal_preflight.json").read_text(encoding="utf-8"))
provenance = json.loads((PHASE3_DIR / "provenance.json").read_text(encoding="utf-8"))
architecture = json.loads((PHASE3_DIR / "architecture.json").read_text(encoding="utf-8"))
protocol_bytes = Path("data/phase2/protocol.json").read_bytes()
if hashlib.sha256(protocol_bytes).hexdigest() != provenance["phase2_protocol_sha256"]:
    raise RuntimeError("Phase 2 protocol changed since provenance was frozen.")
protocol = json.loads(protocol_bytes)
if preflight["status"] != "passed":
    raise RuntimeError("The saved Modal preflight must pass before loading the model.")
for key in ("model_repo", "model_revision"):
    if provenance[key] != preflight["request"][key] or provenance[key] != architecture[key]:
        raise RuntimeError("Model identities differ across Phase 3 artifacts.")
if architecture["config_sha256"] != preflight["request"]["config_sha256"]:
    raise RuntimeError("Architecture checksum differs from the verified preflight.")
probe_bytes = (PHASE3_DIR / "layer_probe_set.jsonl").read_bytes()
probe_manifest = json.loads((PHASE3_DIR / "layer_probe_manifest.json").read_text(encoding="utf-8"))
rows = [json.loads(line) for line in probe_bytes.decode("utf-8").splitlines() if line.strip()]
if [r["eval_id"] for r in rows] != probe_manifest["eval_ids"]:
    raise RuntimeError("Probe IDs differ from the frozen manifest.")
row = rows[0]
baseline_info_bytes = Path("data/phase2/generator_info.json").read_bytes()
baseline_info = json.loads(baseline_info_bytes)
request = {
    "version": "phase3-model-smoke-v1", "model_repo": provenance["model_repo"],
    "model_revision": provenance["model_revision"], "dtype": protocol["DTYPE"],
    "noise_device": protocol["NOISE_DEVICE"], "config_sha256": architecture["config_sha256"],
    "num_layers": architecture["config"]["num_layers"], "row": row,
    "probe_sha256": hashlib.sha256(probe_bytes).hexdigest(),
    "baseline_info_sha256": hashlib.sha256(baseline_info_bytes).hexdigest(),
    "scheduler_class": baseline_info["scheduler_class"],
    "scheduler_config": baseline_info["scheduler_config"],
    "runtime": preflight["request"],
}
request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
smoke_dir = PHASE3_DIR / "smoke"
smoke_dir.mkdir(exist_ok=True)
report_path = smoke_dir / "model_smoke.json"
image_path = smoke_dir / "model_smoke.png"
remote_path = f"/smoke/{request_key}/result.json"

def validate_smoke_report(report):
    if report.get("request") != request or report.get("request_key") != request_key:
        raise RuntimeError("Saved smoke test belongs to different inputs; preserve and inspect it.")

report = None
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    validate_smoke_report(report)
    if report["status"] == "failed":
        raise RuntimeError("Saved smoke test failed; no retry scheduled: " + report["error"])
    if image_path.is_file():
        if hashlib.sha256(image_path.read_bytes()).hexdigest() != report["image_sha256"]:
            raise RuntimeError("Saved smoke image checksum differs; preserve and inspect it.")
        print("Reusing verified local smoke image and report; no Modal call.")
    else:
        report = None  # Recover a missing image from the durable remote packet.

if report is None:
    import modal

    runtime = request["runtime"]
    results_volume = modal.Volume.from_name(runtime["results_volume"])
    try:
        packet = json.loads(b"".join(results_volume.read_file(remote_path)))
        print("Recovered remote smoke result; no GPU scheduled.")
    except FileNotFoundError:
        # A local report proves the worker already ran. Never recompute if its backup is missing.
        if report_path.exists() or image_path.exists():
            raise RuntimeError("Local smoke artifacts are incomplete and remote backup is missing; inspect before retrying.")
        app = modal.App("nihonga-phase3-model-smoke")
        weights_volume = modal.Volume.from_name(runtime["weights_volume"])
        image = (modal.Image.debian_slim(python_version=runtime["python_version"])
                 .apt_install("git").pip_install(*runtime["packages"])
                 .env({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}))

        @app.function(image=image, gpu=runtime["gpu_requested"], timeout=1800,
                      max_containers=1, retries=0, serialized=True,
                      volumes={"/root/.cache/huggingface": weights_volume.with_mount_options(read_only=True),
                               "/phase3": results_volume})
        def run_model_smoke(expected, key):
            import base64
            import hashlib
            import importlib.metadata
            import io
            import json
            import time
            from datetime import datetime, timezone
            from pathlib import Path
            import torch
            from diffusers import QwenImage21Pipeline

            saved = Path("/phase3/smoke") / key / "result.json"
            if saved.is_file():
                return json.loads(saved.read_text())
            measured = {"request": expected, "request_key": key,
                        "recorded_utc": datetime.now(timezone.utc).isoformat(),
                        "status": "failed"}
            packet = {"report": measured}
            try:
                measured["gpu"] = torch.cuda.get_device_name(0)
                if "H100" not in measured["gpu"]:
                    raise RuntimeError("Allocated GPU differs from requested H100.")
                measured["packages"] = {p: importlib.metadata.version(p) for p in
                                         ("torch", "diffusers", "transformers", "accelerate", "pillow")}
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
                start = time.perf_counter()
                pipe = QwenImage21Pipeline.from_pretrained(
                    expected["model_repo"], revision=expected["model_revision"],
                    dtype=getattr(torch, expected["dtype"]), local_files_only=True).to("cuda")
                pipe.set_progress_bar_config(disable=True)
                torch.cuda.synchronize()
                measured["load_seconds"] = time.perf_counter() - start
                measured["num_layers_loaded"] = len(pipe.transformer.transformer_blocks)
                if measured["num_layers_loaded"] != expected["num_layers"]:
                    raise RuntimeError("Loaded block count differs from verified architecture.")
                if type(pipe.scheduler).__name__ != expected["scheduler_class"]:
                    raise RuntimeError("Scheduler class differs from Phase 2.")
                scheduler_config = json.loads(json.dumps(dict(pipe.scheduler.config), default=str))
                if scheduler_config != expected["scheduler_config"]:
                    raise RuntimeError("Scheduler configuration differs from Phase 2.")
                measured["scheduler_class"] = type(pipe.scheduler).__name__
                measured["load_peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
                measured["load_peak_reserved_bytes"] = torch.cuda.max_memory_reserved()
                print(f"Loaded {measured['num_layers_loaded']} blocks in {measured['load_seconds']:.1f}s.")
                torch.cuda.reset_peak_memory_stats()
                r = expected["row"]
                generator = torch.Generator(expected["noise_device"]).manual_seed(r["seed"])
                torch.cuda.synchronize()
                start = time.perf_counter()
                with torch.inference_mode():
                    generated = pipe(prompt=r["prompt"], width=r["width"], height=r["height"],
                                     num_inference_steps=r["num_inference_steps"],
                                     true_cfg_scale=r["true_cfg_scale"], generator=generator).images[0]
                torch.cuda.synchronize()
                measured["generation_seconds"] = time.perf_counter() - start
                measured["generation_peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
                measured["generation_peak_reserved_bytes"] = torch.cuda.max_memory_reserved()
                measured["image_size"] = list(generated.size)
                if measured["image_size"] != [r["width"], r["height"]]:
                    raise RuntimeError("Generated image dimensions differ from the frozen probe.")
                buffer = io.BytesIO()
                generated.save(buffer, format="PNG")
                png = buffer.getvalue()
                measured["image_sha256"] = hashlib.sha256(png).hexdigest()
                measured["status"] = "passed"
                packet["png_base64"] = base64.b64encode(png).decode("ascii")
            except Exception as error:
                measured["error"] = f"{type(error).__name__}: {error}"
            # Persist image and metadata together before returning, including failed runs.
            saved.parent.mkdir(parents=True, exist_ok=True)
            temporary = saved.with_suffix(".tmp")
            temporary.write_text(json.dumps(packet, indent=2) + "\n")
            temporary.replace(saved)
            results_volume.commit()
            return packet

        with modal.enable_output(), app.run():
            packet = run_model_smoke.remote(request, request_key)

    report = packet["report"]
    validate_smoke_report(report)
    if report["status"] == "passed":
        png = base64.b64decode(packet["png_base64"], validate=True)
        if hashlib.sha256(png).hexdigest() != report["image_sha256"]:
            raise RuntimeError("Remote smoke image checksum differs from its report.")
        if image_path.exists() and image_path.read_bytes() != png:
            raise RuntimeError("Existing local smoke image differs from remote backup.")
        if not image_path.exists():
            temporary = image_path.with_suffix(".tmp")
            temporary.write_bytes(png)
            temporary.replace(image_path)
    if report_path.exists():
        if json.loads(report_path.read_text(encoding="utf-8")) != report:
            raise RuntimeError("Existing local smoke report differs from remote backup.")
    else:
        temporary = report_path.with_suffix(".tmp")
        temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        temporary.replace(report_path)

if report["status"] != "passed":
    raise RuntimeError("Smoke test failed and was saved: " + report["error"])
print(f"Smoke test passed: {report['gpu']}, {report['num_layers_loaded']} blocks.")
print(f"Load: {report['load_seconds']:.1f}s; first image: {report['generation_seconds']:.1f}s.")
print(f"Generation peak: {report['generation_peak_allocated_bytes'] / 2**30:.2f} GiB allocated, "
      f"{report['generation_peak_reserved_bytes'] / 2**30:.2f} GiB reserved.")
print(f"Saved image: {image_path}; report: {report_path}")
print("First-image timing includes warmup; this is a readiness test, not a latency benchmark.")


✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-yRIK2pJRn186qRdKzH8SZf
✓ Created objects.
└── 🔨 Created function run_model_smoke.

Loading pipeline components...:  40%|████      | 2/5 [00:02<00:02,  1.04it/s]


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]


Loading weights:   4%|▎         | 28/750 [00:00<00:03, 219.21it/s]


Loading weights:   7%|▋         | 50/750 [00:01<00:16, 42.59it/s] 


Loading weights:   8%|▊         | 62/750 [00:01<00:13, 51.72it/s]


Loading weights:  10%|▉         | 73/750 [00:01<00:12, 56.40it/s]


Loading weights:  11%|█         | 83/750 [00:01<00:10, 63.66it/s]


Loading weights:  12%|█▏        | 93/750 [00:01<00:09, 68.04it/s]


Loading weights:  14%|█▎        | 103/750 [00:01<00:09, 71.53it/s]


Loading weights:  15%|█▍        | 112/750 [00:01<00:09, 66.33it/s]


Loading weights:  17%|█▋        | 125/750 [00:01<00:08, 76.28it/s]


Loading weights:  18%|█▊        | 134/750 [00:02<00:09, 67.80it/s]


Loading weights:  1

## 6. Measure image-token activation similarity on a five-prompt pilot

We first validate our measurement hooks on one frozen prompt from each of the five capability groups. Each prompt retains its Phase 2 seed, dimensions, 40-step schedule, guidance scale, and CPU noise generator. We use the pinned BF16 teacher on Modal H100, without changing or removing any block. Forward hooks compare the hidden states immediately before and after each of the 32 transformer blocks at denoising steps 0, 19, and 39, recording the actual model timestep and prefix-cache mode.

Qwen-Image-2.1 processes the text prefix with the image tokens on its first forward, then reuses prefix KV caches and processes only image queries. The pinned implementation appends target image tokens to the sequence, so we compare only those trailing tokens at every sampled step. This avoids mixing text tokens into early measurements. For each block we compute FP32 cosine similarity over the flattened image-token states, the mean and p10/p50/p90 of per-token cosine, input/output RMS, and relative RMS change `||h_out - h_in|| / max(||h_in||, epsilon)`. High similarity and small changes are redundancy signals, not evidence that a block can safely be removed.

We collect all image tokens without sampling, reduce hidden states on the GPU, and retain only scalar statistics. Each prompt produces 96 layer-stage records and a lossless image; the five-prompt pilot produces 480 records. For the prompt shared with the smoke test we also record whether its instrumented image exactly matches the earlier uninstrumented PNG. This comparison is a diagnostic because separate GPU runs may have numerical variation. Timing and memory include the hooks and must not be used as model speed benchmarks.

The worker checkpoints each prompt's measurements and image together to the persistent results volume before moving to the next prompt. We save local copies under `data/phase3/activations/pilot/` and a summary of layer-stage means across the five prompts. Reruns verify and reuse local results and the saved summary, making no Modal calls when all artifacts exist; missing local results are recovered from the remote packets before scheduling only unfinished prompts. Saved failures stop instead of automatically repeating GPU work. This small pilot verifies instrumentation before expanding coverage to the full frozen probe set.


In [1]:
# Pilot image-token activation measurements; checkpoint each prompt before returning.
import base64
import hashlib
import json
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
smoke = json.loads((PHASE3_DIR / "smoke/model_smoke.json").read_text(encoding="utf-8"))
if smoke["status"] != "passed":
    raise RuntimeError("The saved model smoke test must pass first.")
provenance = json.loads((PHASE3_DIR / "provenance.json").read_text(encoding="utf-8"))
architecture = json.loads((PHASE3_DIR / "architecture.json").read_text(encoding="utf-8"))
protocol_bytes = Path("data/phase2/protocol.json").read_bytes()
if hashlib.sha256(protocol_bytes).hexdigest() != provenance["phase2_protocol_sha256"]:
    raise RuntimeError("Phase 2 protocol changed since provenance was frozen.")
protocol = json.loads(protocol_bytes)
for key in ("model_repo", "model_revision"):
    if provenance[key] != smoke["request"][key] or architecture[key] != provenance[key]:
        raise RuntimeError("Saved model identities differ.")
if architecture["config_sha256"] != smoke["request"]["config_sha256"]:
    raise RuntimeError("Saved architecture differs from the smoke test.")
probe_bytes = (PHASE3_DIR / "layer_probe_set.jsonl").read_bytes()
if hashlib.sha256(probe_bytes).hexdigest() != smoke["request"]["probe_sha256"]:
    raise RuntimeError("Frozen probe set changed since the model smoke test.")
rows = [json.loads(line) for line in probe_bytes.decode("utf-8").splitlines() if line.strip()]
pilot_rows = []
seen_dimensions = set()
for row in rows:
    if row["dimension"] not in seen_dimensions:
        pilot_rows.append(row)
        seen_dimensions.add(row["dimension"])
if seen_dimensions != {"aesthetics", "alignment", "creative", "real_world", "visual_quality"}:
    raise RuntimeError("The frozen probe set does not cover all five capability groups.")
if any(r["true_cfg_scale"] != 1.0 or r["num_inference_steps"] < 3 for r in pilot_rows):
    raise RuntimeError("This pilot requires guidance scale 1.0 and at least three steps.")

metric_names = ["flattened_cosine", "token_cosine_mean", "token_cosine_p10",
                "token_cosine_p50", "token_cosine_p90", "relative_rms_change",
                "input_rms", "output_rms"]
request = {
    "version": "phase3-activation-pilot-v1", "model_repo": provenance["model_repo"],
    "model_revision": provenance["model_revision"], "dtype": protocol["DTYPE"],
    "noise_device": protocol["NOISE_DEVICE"], "config_sha256": architecture["config_sha256"],
    "num_layers": architecture["config"]["num_layers"], "runtime": smoke["request"]["runtime"],
    "scheduler_class": smoke["request"]["scheduler_class"],
    "scheduler_config": smoke["request"]["scheduler_config"],
    "probe_sha256": hashlib.sha256(probe_bytes).hexdigest(),
    "selection": "first frozen probe per capability group, preserving probe order",
    "pilot_rows": pilot_rows, "step_selection": "0, (num_inference_steps - 1) // 2, num_inference_steps - 1",
    "token_scope": "all target image tokens; exclude cached text and condition prefixes",
    "metric_names": metric_names, "reduction_dtype": "float32", "epsilon": 1e-12,
    "reference_smoke": {"eval_id": smoke["request"]["row"]["eval_id"],
                        "image_sha256": smoke["image_sha256"]},
}
request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
pilot_dir = PHASE3_DIR / "activations/pilot"
pilot_dir.mkdir(parents=True, exist_ok=True)
manifest_path = pilot_dir / "request.json"
manifest = {"request": request, "request_key": request_key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Saved activation pilot uses different inputs; preserve and inspect it.")
else:
    with manifest_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(manifest, indent=2) + "\n")

def validate_activation_report(report, row):
    if report.get("request_key") != request_key or report.get("row") != row:
        raise RuntimeError("Saved activation result belongs to different inputs.")
    if report["status"] == "failed":
        raise RuntimeError("Saved activation run failed; no retry scheduled: " + report["error"])
    steps = [0, (row["num_inference_steps"] - 1) // 2, row["num_inference_steps"] - 1]
    expected_pairs = {(step, layer) for step in steps for layer in range(request["num_layers"])}
    observed_pairs = {(m["step_index"], m["layer_index"]) for m in report["measurements"]}
    if observed_pairs != expected_pairs or len(report["measurements"]) != len(expected_pairs):
        raise RuntimeError("Saved activation measurements are incomplete or duplicated.")

def save_activation_packet(packet, row):
    report = packet["report"]
    if report.get("request_key") != request_key or report.get("row") != row:
        raise RuntimeError("Remote activation packet belongs to different inputs.")
    report_path = pilot_dir / f"{row['eval_id']}.json"
    image_path = pilot_dir / f"{row['eval_id']}.png"
    if report["status"] == "passed":
        validate_activation_report(report, row)
        png = base64.b64decode(packet["png_base64"], validate=True)
        if hashlib.sha256(png).hexdigest() != report["image_sha256"]:
            raise RuntimeError("Remote activation image checksum differs from its report.")
        if image_path.exists() and image_path.read_bytes() != png:
            raise RuntimeError("Existing local activation image differs from its backup.")
        if not image_path.exists():
            temporary = image_path.with_suffix(".tmp")
            temporary.write_bytes(png)
            temporary.replace(image_path)
    if report_path.exists():
        if json.loads(report_path.read_text(encoding="utf-8")) != report:
            raise RuntimeError("Existing local activation report differs from its backup.")
    else:
        temporary = report_path.with_suffix(".tmp")
        temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        temporary.replace(report_path)
    validate_activation_report(report, row)
    return report

reports = {}
missing_rows = []
for row in pilot_rows:
    report_path = pilot_dir / f"{row['eval_id']}.json"
    image_path = pilot_dir / f"{row['eval_id']}.png"
    if report_path.exists():
        report = json.loads(report_path.read_text(encoding="utf-8"))
        validate_activation_report(report, row)
        if image_path.is_file():
            if hashlib.sha256(image_path.read_bytes()).hexdigest() != report["image_sha256"]:
                raise RuntimeError("Saved activation image checksum differs from its report.")
            reports[row["eval_id"]] = report
            continue
    missing_rows.append(row)

if missing_rows:
    import modal

    runtime = request["runtime"]
    results_volume = modal.Volume.from_name(runtime["results_volume"])
    pending_rows = []
    for row in missing_rows:
        remote_path = f"/activations/{request_key}/{row['eval_id']}.json"
        try:
            packet = json.loads(b"".join(results_volume.read_file(remote_path)))
            reports[row["eval_id"]] = save_activation_packet(packet, row)
            print(f"Recovered {row['eval_id']} from remote backup; no GPU work.")
        except FileNotFoundError:
            if any((pilot_dir / f"{row['eval_id']}.{suffix}").exists() for suffix in ("json", "png")):
                raise RuntimeError("Local activation artifacts are incomplete and their remote backup is missing.")
            pending_rows.append(row)

    if pending_rows:
        app = modal.App("nihonga-phase3-activation-pilot")
        weights_volume = modal.Volume.from_name(runtime["weights_volume"])
        image = (modal.Image.debian_slim(python_version=runtime["python_version"])
                 .apt_install("git").pip_install(*runtime["packages"])
                 .env({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}))

        @app.function(image=image, gpu=runtime["gpu_requested"], timeout=1800,
                      max_containers=1, retries=0, serialized=True,
                      volumes={"/root/.cache/huggingface": weights_volume.with_mount_options(read_only=True),
                               "/phase3": results_volume})
        def measure_activations(expected, key, jobs):
            import base64
            import hashlib
            import importlib.metadata
            import inspect
            import io
            import json
            import math
            import time
            from datetime import datetime, timezone
            from pathlib import Path
            import torch
            from diffusers import QwenImage21Pipeline

            root = Path("/phase3/activations") / key
            root.mkdir(parents=True, exist_ok=True)
            packets = []
            pipe = None
            load_error = None
            runtime_info = {"gpu": torch.cuda.get_device_name(0),
                            "packages": {p: importlib.metadata.version(p) for p in
                                         ("torch", "diffusers", "transformers", "accelerate", "pillow")}}
            try:
                if "H100" not in runtime_info["gpu"]:
                    raise RuntimeError("Allocated GPU is not the requested H100.")
                start = time.perf_counter()
                pipe = QwenImage21Pipeline.from_pretrained(
                    expected["model_repo"], revision=expected["model_revision"],
                    dtype=getattr(torch, expected["dtype"]), local_files_only=True).to("cuda")
                pipe.set_progress_bar_config(disable=True)
                pipe.transformer.eval()
                if (len(pipe.transformer.transformer_blocks) != expected["num_layers"]
                        or not pipe.transformer.config.causal_condition):
                    raise RuntimeError("Loaded architecture is incompatible with target-token probing.")
                if (type(pipe.scheduler).__name__ != expected["scheduler_class"]
                        or json.loads(json.dumps(dict(pipe.scheduler.config), default=str)) != expected["scheduler_config"]):
                    raise RuntimeError("Loaded scheduler differs from Phase 2.")
                source = Path(inspect.getfile(type(pipe.transformer)))
                runtime_info["transformer_source_sha256"] = hashlib.sha256(source.read_bytes()).hexdigest()
                torch.cuda.synchronize()
                runtime_info["load_seconds"] = time.perf_counter() - start
                print(f"Loaded {expected['num_layers']} blocks; measuring {len(jobs)} pilot prompts.")
            except Exception as error:
                load_error = f"{type(error).__name__}: {error}"

            for row in jobs:
                saved = root / f"{row['eval_id']}.json"
                if saved.is_file():
                    packets.append(json.loads(saved.read_text(encoding="utf-8")))
                    continue
                report = {"request_key": key, "row": row, "runtime": runtime_info,
                          "recorded_utc": datetime.now(timezone.utc).isoformat(), "status": "failed"}
                packet = {"report": report}
                hooks = []
                measurements = []
                pending_stats = []
                state = {"calls": 0}
                stage_by_step = {0: "early", (row["num_inference_steps"] - 1) // 2: "middle",
                                 row["num_inference_steps"] - 1: "final"}

                def begin_forward(module, args, kwargs):
                    step = state["calls"]
                    state["calls"] += 1
                    state["step"] = step
                    if step in stage_by_step:
                        state["target_tokens"] = math.prod(kwargs["img_shapes"][0][-1])
                        state["timestep"] = float(kwargs["timestep"].detach().float().cpu().reshape(-1)[0])
                        state["kv_cache_mode"] = kwargs.get("kv_cache_mode")

                def block_hook(layer_index):
                    def capture(module, args, kwargs, output):
                        step = state["step"]
                        if step not in stage_by_step:
                            return None
                        hidden = kwargs["hidden_states"]
                        count = state["target_tokens"]
                        if (not isinstance(output, torch.Tensor) or output.shape != hidden.shape
                                or hidden.ndim != 3 or hidden.shape[0] != 1 or hidden.shape[1] < count):
                            raise RuntimeError("Unexpected transformer block input/output layout.")
                        mode = state["kv_cache_mode"]
                        if mode != ("extract" if step == 0 else "cached"):
                            raise RuntimeError("Unexpected prefix KV-cache mode.")
                        # The pinned model appends target tokens, and trims the prefix on cached steps.
                        # All operations below read inputs and outputs; hooks never replace or mutate them.
                        x = hidden[:, -count:, :].detach().float()
                        y = output[:, -count:, :].detach().float()
                        eps = expected["epsilon"]
                        x_sq = x.square().sum()
                        y_sq = y.square().sum()
                        norm_x = torch.linalg.vector_norm(x, dim=-1)
                        norm_y = torch.linalg.vector_norm(y, dim=-1)
                        dot_per_token = (x * y).sum(dim=-1)
                        token_cos = dot_per_token / (norm_x * norm_y).clamp_min(eps)
                        quantiles = torch.quantile(token_cos.flatten(),
                                                 torch.tensor([0.1, 0.5, 0.9], device=x.device))
                        stats = torch.stack([
                            dot_per_token.sum() / (x_sq * y_sq).sqrt().clamp_min(eps),
                            token_cos.mean(), quantiles[0], quantiles[1], quantiles[2],
                            (y - x).square().sum().sqrt() / x_sq.sqrt().clamp_min(eps),
                            (x_sq / x.numel()).sqrt(), (y_sq / y.numel()).sqrt()])
                        pending_stats.append(stats)
                        measurements.append({"step_index": step, "stage": stage_by_step[step],
                                             "timestep": state["timestep"], "layer_index": layer_index,
                                             "target_tokens": count, "block_sequence_tokens": hidden.shape[1],
                                             "hidden_dim": hidden.shape[2], "kv_cache_mode": mode})
                        return None
                    return capture

                try:
                    if load_error is not None:
                        raise RuntimeError(load_error)
                    hooks.append(pipe.transformer.register_forward_pre_hook(begin_forward, with_kwargs=True))
                    for index, block in enumerate(pipe.transformer.transformer_blocks):
                        hooks.append(block.register_forward_hook(block_hook(index), with_kwargs=True))
                    torch.cuda.reset_peak_memory_stats()
                    generator = torch.Generator(expected["noise_device"]).manual_seed(row["seed"])
                    torch.cuda.synchronize()
                    start = time.perf_counter()
                    with torch.inference_mode():
                        generated = pipe(prompt=row["prompt"], width=row["width"], height=row["height"],
                                         num_inference_steps=row["num_inference_steps"],
                                         true_cfg_scale=row["true_cfg_scale"], generator=generator,
                                         use_kv_cache=True).images[0]
                    torch.cuda.synchronize()
                    report["instrumented_generation_seconds"] = time.perf_counter() - start
                    report["peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
                    report["peak_reserved_bytes"] = torch.cuda.max_memory_reserved()
                    if (state["calls"] != row["num_inference_steps"]
                            or len(measurements) != 3 * expected["num_layers"]):
                        raise RuntimeError("Unexpected forward count or missing activation measurements.")
                    values = torch.stack(pending_stats).cpu().tolist()
                    if not all(math.isfinite(value) for stats in values for value in stats):
                        raise RuntimeError("Activation measurements contain nonfinite values.")
                    for measurement, stats in zip(measurements, values):
                        measurement.update(dict(zip(expected["metric_names"], stats)))
                    report["measurements"] = measurements
                    buffer = io.BytesIO()
                    generated.save(buffer, format="PNG")
                    png = buffer.getvalue()
                    report["image_sha256"] = hashlib.sha256(png).hexdigest()
                    report["image_size"] = list(generated.size)
                    if report["image_size"] != [row["width"], row["height"]]:
                        raise RuntimeError("Generated image dimensions differ from the probe.")
                    if row["eval_id"] == expected["reference_smoke"]["eval_id"]:
                        report["matches_uninstrumented_smoke_png"] = (
                            report["image_sha256"] == expected["reference_smoke"]["image_sha256"])
                    report["status"] = "passed"
                    packet["png_base64"] = base64.b64encode(png).decode("ascii")
                except Exception as error:
                    report["error"] = f"{type(error).__name__}: {error}"
                finally:
                    for hook in hooks:
                        hook.remove()
                    pending_stats.clear()
                temporary = saved.with_suffix(".tmp")
                temporary.write_text(json.dumps(packet, indent=2) + "\n", encoding="utf-8")
                temporary.replace(saved)
                results_volume.commit()
                packets.append(packet)
                print(f"Saved {row['dimension']}: {row['eval_id']} ({report['status']}).")
                if report["status"] == "failed":
                    break
            return packets

        with modal.enable_output(), app.run():
            packets = measure_activations.remote(request, request_key, pending_rows)
        row_by_id = {row["eval_id"]: row for row in pilot_rows}
        for packet in packets:
            eval_id = packet["report"]["row"]["eval_id"]
            reports[eval_id] = save_activation_packet(packet, row_by_id[eval_id])
else:
    print("Reusing all verified local activation pilot results; no Modal call.")

if len(reports) != len(pilot_rows):
    raise RuntimeError("Activation pilot is incomplete; inspect saved results before continuing.")
summary_path = pilot_dir / "summary.json"
report_hashes = {row["eval_id"]: hashlib.sha256(
    (pilot_dir / f"{row['eval_id']}.json").read_bytes()).hexdigest() for row in pilot_rows}
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary["request_key"] != request_key or summary["report_hashes"] != report_hashes:
        raise RuntimeError("Saved activation summary differs from its input reports.")
    print("Reusing saved activation summary.")
else:
    from statistics import fmean

    measurements = [dict(m, eval_id=row["eval_id"], dimension=row["dimension"])
                    for row in pilot_rows for m in reports[row["eval_id"]]["measurements"]]
    layer_stage_means = []
    for stage in ("early", "middle", "final"):
        for layer in range(request["num_layers"]):
            values = [m for m in measurements if m["stage"] == stage and m["layer_index"] == layer]
            layer_stage_means.append({"stage": stage, "layer_index": layer, "n_prompts": len(values),
                                      **{name: fmean(m[name] for m in values) for name in metric_names}})
    summary = {"request_key": request_key, "report_hashes": report_hashes,
               "n_prompts": len(pilot_rows), "n_measurements": len(measurements),
               "layer_stage_means": layer_stage_means,
               "scope": "instrumentation pilot; similarity alone does not establish pruning safety"}
    temporary = summary_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    temporary.replace(summary_path)

print(f"Saved {summary['n_measurements']} measurements: {summary['n_prompts']} prompts x "
      f"{request['num_layers']} blocks x 3 denoising stages.")
print("Mean image-token cosine / relative RMS change across five prompts:")
for entry in summary["layer_stage_means"]:
    if entry["layer_index"] in (0, 8, 16, 24, request["num_layers"] - 1):
        print(f"  {entry['stage']:6s} block {entry['layer_index']:2d}: "
              f"cosine={entry['token_cosine_mean']:.4f}, change={entry['relative_rms_change']:.4f}")
reference_report = reports[request["reference_smoke"]["eval_id"]]
print("Instrumented PNG matches prior uninstrumented smoke PNG:",
      reference_report["matches_uninstrumented_smoke_png"])
print(f"Results: {pilot_dir}; raw hidden-state tensors were reduced on GPU, not retained.")
print("These are diagnostic signals; layer bypass and image-quality checks are still required before pruning.")


✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-UzaUscoX2cU2wAYK8MLpoz
✓ Created objects.
└── 🔨 Created function measure_activations.

Loading pipeline components...:  20%|██        | 1/5 [00:00<00:02,  1.69it/s]


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/750 [00:00<04:29,  2.78it/s]


Loading weights:   0%|          | 2/750 [00:02<12:48,  1.03s/it]


Loading weights:   0%|          | 3/750 [00:03<12:17,  1.01it/s]


Loading weights:   1%|          | 4/750 [00:03<09:29,  1.31it/s]


Loading weights:   2%|▏         | 13/750 [00:03<01:40,  7.36it/s]


Loading weights:   3%|▎         | 19/750 [00:03<01:01, 11.93it/s]


Loading weights:   4%|▎         | 28/750 [00:03<00:35, 20.48it/s]


Loading weights:   5%|▍         | 37/750 [00:03<00:24, 29.12it/s]


Loading weights:   6%|▌         | 43/750 [00:03<00:21, 33.21it/s]


Loading weights:   7%|▋         | 51/750 [00:03<00:17, 41.03it/s]


Loading weights:   8%|▊   

## Interpretation of the activation pilot results

The measurement pipeline completed successfully for all five prompts, one from each capability group. Each prompt contributed 32 blocks × 3 denoising stages = 96 records, giving **480 records in total**. Block indices are zero-based: block 0 is the first block and block 31 is the last. “Passed” means generation and measurement checks succeeded; it is not an image-quality score. The messages about stopping the app and terminating the runner mark normal completion of the Modal run.

**How to read the two printed metrics.** We compare a block's incoming and outgoing hidden states on the generated image tokens, excluding the text prefix. Each printed value is an average across the five prompts:

- **Cosine similarity** is the mean of the per-token cosines between input and output feature vectors. A value near 1 means their directions are similar; a value near 0 means little directional alignment. It does not measure image quality or the similarity between two generated images.
- **Relative RMS change** is `||h_out - h_in|| / ||h_in||`, with a small denominator safeguard. A value of 0.10 means the update's norm is about 10% of the input norm. A value of 8 means the update's norm is about eight times the input norm; it does not mean the output is eight times better or eight times larger. This metric complements cosine because feature vectors can retain their direction while changing substantially in magnitude.

The table reproduces the selected blocks printed above. Each entry is **cosine / relative RMS change**. These are three positions in the actual denoising schedule; their model timesteps are recorded in the per-prompt reports.

| Block index | Early: step 0 | Middle: step 19 | Final: step 39 |
|---|---:|---:|---:|
| 0 | 0.1060 / 8.0751 | 0.4537 / 10.4572 | 0.6791 / 6.6391 |
| 8 | 0.9906 / 0.1661 | 0.9893 / 0.1690 | 0.9902 / 0.1744 |
| 16 | 0.9879 / 0.1895 | 0.9789 / 0.2216 | 0.9822 / 0.1914 |
| 24 | 0.9964 / 0.0847 | 0.9920 / 0.1275 | 0.9795 / 0.2334 |
| 31 | 0.5758 / 4.2262 | 0.8201 / 1.7268 | 0.7905 / 1.5455 |

**What the selected blocks show.** Block 0 makes a large transformation at every sampled stage. Its early cosine is only 0.1060, and its relative change ranges from 6.6391 to 10.4572. Block 31 also changes the representation substantially, especially early in denoising, where its cosine is 0.5758 and its relative change is 4.2262. These observations argue for caution when considering either boundary block for removal; they do not establish its causal importance to image quality.

The displayed interior blocks, 8, 16, and 24, preserve feature directions much more closely, with cosines around 0.98–1.00 and considerably smaller relative changes. They therefore show stronger redundancy signals in this pilot. However, small residual updates can still carry essential information, and multiple small updates can accumulate. High similarity alone does not justify removing a block or a group of blocks.

**The behavior depends on denoising stage.** For example, block 24 changes from cosine 0.9964 and relative change 0.0847 early to cosine 0.9795 and relative change 0.2334 at the final step. A block that appears nearly unchanged early can make a larger relative contribution during detail refinement. Averaging all stages into a single score could hide this behavior.

**Instrumentation check and limits.** The prompt shared with the earlier smoke test produced an exactly matching PNG with the hooks installed. This supports that the instrumentation preserved the output for that checked prompt. It does not validate every prompt or prove that the teacher is invariant across separate GPU runs.

The results establish that the hooks work and expose meaningful differences between blocks and stages. They remain a small diagnostic pilot: only five prompts were used, and the 480 records are correlated observations, not 480 independent tests. The printed table also shows only five of the 32 blocks. Establishing pruning safety requires broader prompt coverage and controlled bypass measurements, including checks of generated-image regressions.

All five generated images, per-prompt scalar statistics, and the aggregate summary are saved under `data/phase3/activations/pilot/`; the remote prompt packets also contain their images and statistics. Raw hidden-state tensors were reduced on the GPU and were not retained. The saved statistics support these comparisons without rerunning inference, but cannot reconstruct hidden states for later bridge training.


## 7. Expand activation measurements to the full 64-prompt probe set

The five-prompt pilot verified the hooks and matched the earlier smoke-test image. We now apply the same image-token measurements to all 64 frozen prompts, preserving their original seeds, resolutions, schedules, guidance settings, pinned BF16 model, and H100 runtime. We reuse the five completed pilot reports and images from their original files rather than regenerate them. Only the remaining 59 prompts require inference. Coverage is 64 prompts ? 32 blocks ? 3 denoising stages = **6,144 layer-stage records**.

To keep the validated measurement implementation consistent, this cell reads only the worker function definition from the pilot code cell. It does not execute the pilot cell. A checked AST transformation changes its result delivery from one accumulated list to one streamed packet per completed prompt; the forward hooks, token selection, and metric reductions are unchanged. We hash both worker definitions and freeze the full request, including pilot artifact checksums and the entire probe set. The remote worker saves that request and commits each prompt packet before returning it, and the local notebook immediately saves each received image and report.

Results for the 59 additional prompts are stored under `data/phase3/activations/full/`. Matching local results are verified and reused without Modal calls. Missing local results are recovered from the remote volume before scheduling only unfinished prompts. Failed reports stop without automatic GPU retries. The model loads once for all unfinished prompts, and every completed prompt remains checkpointed if the run is interrupted.

The saved summary combines all 64 reports, with separate means for each block and denoising stage, each capability group, and each difficulty level. It also records p10/p50/p90 and extreme values across prompts for mean token cosine and relative RMS change, plus the IDs of the prompts with the lowest cosine and largest update. These prompt-level percentiles differ from the within-image token percentiles collected by each hook. Group sample counts are included because subgroup estimates have less coverage than the overall mean. Cached reruns reuse this summary without recalculating statistics. This pass measures activation behavior; it does not bypass blocks or decide which layers to remove.


In [1]:
# Extend the validated activation recipe to all frozen probes; reuse every completed prompt.
import ast
import base64
import hashlib
import json
import math
import textwrap
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
pilot_dir = PHASE3_DIR / "activations/pilot"
full_dir = PHASE3_DIR / "activations/full"
full_dir.mkdir(parents=True, exist_ok=True)
pilot_manifest = json.loads((pilot_dir / "request.json").read_text(encoding="utf-8"))
pilot_request = pilot_manifest["request"]
pilot_key = pilot_manifest["request_key"]
if hashlib.sha256(json.dumps(pilot_request, sort_keys=True).encode()).hexdigest() != pilot_key:
    raise RuntimeError("Pilot request checksum is invalid.")
pilot_summary = json.loads((pilot_dir / "summary.json").read_text(encoding="utf-8"))
if pilot_summary["request_key"] != pilot_key:
    raise RuntimeError("Pilot summary belongs to another request.")
provenance = json.loads((PHASE3_DIR / "provenance.json").read_text(encoding="utf-8"))
architecture = json.loads((PHASE3_DIR / "architecture.json").read_text(encoding="utf-8"))
protocol_bytes = Path("data/phase2/protocol.json").read_bytes()
if hashlib.sha256(protocol_bytes).hexdigest() != provenance["phase2_protocol_sha256"]:
    raise RuntimeError("Phase 2 protocol changed since Phase 3 provenance was frozen.")
protocol = json.loads(protocol_bytes)
for key in ("model_repo", "model_revision"):
    if provenance[key] != pilot_request[key] or architecture[key] != provenance[key]:
        raise RuntimeError("Model identities differ across saved artifacts.")
if (architecture["config_sha256"] != pilot_request["config_sha256"]
        or architecture["config"]["num_layers"] != pilot_request["num_layers"]
        or protocol["DTYPE"] != pilot_request["dtype"]
        or protocol["NOISE_DEVICE"] != pilot_request["noise_device"]
        or protocol["GEN_PACKAGES"] != pilot_request["runtime"]["packages"]):
    raise RuntimeError("Architecture or inference settings differ from the validated pilot.")
probe_bytes = (PHASE3_DIR / "layer_probe_set.jsonl").read_bytes()
if hashlib.sha256(probe_bytes).hexdigest() != pilot_request["probe_sha256"]:
    raise RuntimeError("Frozen probe set changed since the activation pilot.")
rows = [json.loads(line) for line in probe_bytes.decode("utf-8").splitlines() if line.strip()]
probe_manifest = json.loads((PHASE3_DIR / "layer_probe_manifest.json").read_text(encoding="utf-8"))
if len(rows) != 64 or [r["eval_id"] for r in rows] != probe_manifest["eval_ids"]:
    raise RuntimeError("The full pass requires the original 64-row frozen probe set.")
if any(r["true_cfg_scale"] != 1.0 or r["num_inference_steps"] < 3 for r in rows):
    raise RuntimeError("The validated hooks require guidance scale 1.0 and at least three steps.")
row_by_id = {r["eval_id"]: r for r in rows}
pilot_rows = {r["eval_id"]: r for r in pilot_request["pilot_rows"]}
if any(row_by_id.get(eval_id) != row for eval_id, row in pilot_rows.items()):
    raise RuntimeError("Pilot generation conditions differ from the frozen probe set.")

# Read only the existing worker definition, not the executable pilot cell.
# AST changes below replace list accumulation with yields; the hooks and metrics are unchanged.
notebook_path = Path("phase3-identify-layers-to-remove.ipynb")
notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
pilot_cells = [c for c in notebook["cells"] if c.get("id") == "phase3-activation-pilot"]
if len(pilot_cells) != 1:
    raise RuntimeError("Cannot locate the validated pilot code cell.")
pilot_code = "".join(pilot_cells[0]["source"])
definitions = [node for node in ast.walk(ast.parse(pilot_code))
               if isinstance(node, ast.FunctionDef) and node.name == "measure_activations"]
if len(definitions) != 1:
    raise RuntimeError("Cannot uniquely locate the validated measurement worker.")
definition = definitions[0]
worker_source = textwrap.dedent("".join(
    pilot_code.splitlines(keepends=True)[definition.lineno - 1:definition.end_lineno]))

class StreamCompletedPackets(ast.NodeTransformer):
    def __init__(self):
        self.yields = 0
        self.removed_lists = 0
        self.changed_returns = 0

    def visit_Assign(self, node):
        if (len(node.targets) == 1 and isinstance(node.targets[0], ast.Name)
                and node.targets[0].id == "packets" and isinstance(node.value, ast.List)):
            self.removed_lists += 1
            return None
        return self.generic_visit(node)

    def visit_Expr(self, node):
        call = node.value
        if (isinstance(call, ast.Call) and isinstance(call.func, ast.Attribute)
                and isinstance(call.func.value, ast.Name) and call.func.value.id == "packets"
                and call.func.attr == "append" and len(call.args) == 1):
            self.yields += 1
            return ast.copy_location(ast.Expr(value=ast.Yield(value=call.args[0])), node)
        return self.generic_visit(node)

    def visit_Return(self, node):
        if isinstance(node.value, ast.Name) and node.value.id == "packets":
            self.changed_returns += 1
            return ast.copy_location(ast.Return(value=None), node)
        return self.generic_visit(node)

transform = StreamCompletedPackets()
worker_tree = transform.visit(ast.parse(worker_source))
if (transform.yields, transform.removed_lists, transform.changed_returns) != (2, 1, 1):
    raise RuntimeError("Pilot worker structure changed; inspect before extending it.")
worker_tree.body[0].name = "stream_full_activations"
# The remote request is persisted alongside prompt packets for recoverable provenance.
remote_manifest_code = '''
remote_manifest = root / "request.json"
payload = {"request": expected, "request_key": key}
if remote_manifest.exists():
    if json.loads(remote_manifest.read_text(encoding="utf-8")) != payload:
        raise RuntimeError("Remote full-pass manifest differs from the request.")
else:
    temporary_manifest = remote_manifest.with_suffix(".tmp")
    temporary_manifest.write_text(json.dumps(payload, indent=2) + "\\n", encoding="utf-8")
    temporary_manifest.replace(remote_manifest)
    results_volume.commit()
'''
worker_body = worker_tree.body[0].body
root_positions = [i for i, node in enumerate(worker_body)
                  if isinstance(node, ast.Expr) and isinstance(node.value, ast.Call)
                  and isinstance(node.value.func, ast.Attribute)
                  and isinstance(node.value.func.value, ast.Name)
                  and node.value.func.value.id == "root" and node.value.func.attr == "mkdir"]
if len(root_positions) != 1:
    raise RuntimeError("Cannot locate the worker checkpoint directory.")
worker_body[root_positions[0] + 1:root_positions[0] + 1] = ast.parse(remote_manifest_code).body
worker_tree = ast.fix_missing_locations(worker_tree)
stream_source = ast.unparse(worker_tree) + "\n"

pilot_artifacts = {}
for eval_id in pilot_rows:
    report_path = pilot_dir / f"{eval_id}.json"
    image_path = pilot_dir / f"{eval_id}.png"
    digest = hashlib.sha256(report_path.read_bytes()).hexdigest()
    if digest != pilot_summary["report_hashes"][eval_id]:
        raise RuntimeError("Pilot report changed since its saved summary.")
    report = json.loads(report_path.read_text(encoding="utf-8"))
    if (report["request_key"] != pilot_key or report["row"] != pilot_rows[eval_id]
            or report["status"] != "passed"
            or hashlib.sha256(image_path.read_bytes()).hexdigest() != report["image_sha256"]):
        raise RuntimeError("Pilot artifact identity or image verification failed.")
    pilot_artifacts[eval_id] = {"report_sha256": digest, "image_sha256": report["image_sha256"]}

request = {k: v for k, v in pilot_request.items() if k not in ("version", "selection", "pilot_rows")}
request.update({"version": "phase3-activation-full-v1", "selection": "all 64 frozen probes, original order",
                "probe_rows": rows, "reused_pilot_request_key": pilot_key,
                "pilot_artifacts": pilot_artifacts,
                "pilot_worker_source_sha256": hashlib.sha256(worker_source.encode()).hexdigest(),
                "stream_worker_source_sha256": hashlib.sha256(stream_source.encode()).hexdigest()})
request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
manifest = {"request": request, "request_key": request_key}
manifest_path = full_dir / "request.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Saved full activation pass uses different inputs; preserve and inspect it.")
else:
    with manifest_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(manifest, indent=2) + "\n")

def validate_full_report(report, row):
    source_key = pilot_key if row["eval_id"] in pilot_rows else request_key
    if report.get("request_key") != source_key or report.get("row") != row:
        raise RuntimeError("Activation report belongs to different inputs.")
    if report["status"] != "passed":
        raise RuntimeError("Saved activation run failed; no retry scheduled: " + report["error"])
    steps = [0, (row["num_inference_steps"] - 1) // 2, row["num_inference_steps"] - 1]
    expected_pairs = {(step, layer) for step in steps for layer in range(request["num_layers"])}
    observed_pairs = {(m["step_index"], m["layer_index"]) for m in report["measurements"]}
    if observed_pairs != expected_pairs or len(report["measurements"]) != len(expected_pairs):
        raise RuntimeError("Activation records are missing or duplicated.")
    for measurement in report["measurements"]:
        if not all(math.isfinite(measurement[name]) for name in request["metric_names"]):
            raise RuntimeError("Activation records contain nonfinite metrics.")

def save_full_packet(packet, row):
    report = packet["report"]
    if report.get("request_key") != request_key or report.get("row") != row:
        raise RuntimeError("Remote activation packet belongs to different inputs.")
    report_path = full_dir / f"{row['eval_id']}.json"
    image_path = full_dir / f"{row['eval_id']}.png"
    if report["status"] == "passed":
        validate_full_report(report, row)
        png = base64.b64decode(packet["png_base64"], validate=True)
        if hashlib.sha256(png).hexdigest() != report["image_sha256"]:
            raise RuntimeError("Remote activation image checksum differs from its report.")
        if image_path.exists() and image_path.read_bytes() != png:
            raise RuntimeError("Existing local activation image differs from remote backup.")
        if not image_path.exists():
            temporary = image_path.with_suffix(".tmp")
            temporary.write_bytes(png)
            temporary.replace(image_path)
    if report_path.exists():
        if json.loads(report_path.read_text(encoding="utf-8")) != report:
            raise RuntimeError("Existing local report differs from remote backup.")
    else:
        temporary = report_path.with_suffix(".tmp")
        temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        temporary.replace(report_path)
    validate_full_report(report, row)
    return report

reports = {}
missing_rows = []
for row in rows:
    source_dir = pilot_dir if row["eval_id"] in pilot_rows else full_dir
    report_path = source_dir / f"{row['eval_id']}.json"
    image_path = source_dir / f"{row['eval_id']}.png"
    if report_path.is_file():
        report = json.loads(report_path.read_text(encoding="utf-8"))
        validate_full_report(report, row)
        if image_path.is_file():
            if hashlib.sha256(image_path.read_bytes()).hexdigest() != report["image_sha256"]:
                raise RuntimeError("Saved activation image checksum differs from its report.")
            reports[row["eval_id"]] = report
            continue
    missing_rows.append(row)
print(f"Reusing {len(reports)}/64 local results, including all five pilot probes.")

if missing_rows:
    import modal

    runtime = request["runtime"]
    results_volume = modal.Volume.from_name(runtime["results_volume"])
    pending_rows = []
    remote_directory = f"/activations/{request_key}"
    try:
        remote_files = {Path(entry.path).name for entry in results_volume.listdir(remote_directory, recursive=False)}
    except (FileNotFoundError, modal.exception.NotFoundError):
        remote_files = set()
    for row in missing_rows:
        remote_path = f"{remote_directory}/{row['eval_id']}.json"
        try:
            if f"{row['eval_id']}.json" not in remote_files:
                raise FileNotFoundError(remote_path)
            packet = json.loads(b"".join(results_volume.read_file(remote_path)))
            reports[row["eval_id"]] = save_full_packet(packet, row)
            print(f"Recovered {row['eval_id']} from remote backup; no GPU work.")
        except FileNotFoundError:
            if any((full_dir / f"{row['eval_id']}.{suffix}").exists() for suffix in ("json", "png")):
                raise RuntimeError("Local activation artifacts are incomplete and remote backup is missing.")
            pending_rows.append(row)
    if pending_rows:
        print(f"Scheduling only {len(pending_rows)} unfinished prompts on one H100.")
        app = modal.App("nihonga-phase3-full-activations")
        weights_volume = modal.Volume.from_name(runtime["weights_volume"])
        image = (modal.Image.debian_slim(python_version=runtime["python_version"])
                 .apt_install("git").pip_install(*runtime["packages"])
                 .env({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}))
        # This defines only the validated worker with streaming returns; it never executes the pilot cell.
        exec(compile(stream_source, "phase3_full_activation_worker", "exec"), globals())
        stream_full_activations = app.function(
            image=image, gpu=runtime["gpu_requested"], timeout=1800,
            max_containers=1, serialized=True,
            volumes={"/root/.cache/huggingface": weights_volume.with_mount_options(read_only=True),
                     "/phase3": results_volume})(stream_full_activations)
        with modal.enable_output(), app.run():
            for packet in stream_full_activations.remote_gen(request, request_key, pending_rows):
                eval_id = packet["report"]["row"]["eval_id"]
                reports[eval_id] = save_full_packet(packet, row_by_id[eval_id])
                print(f"Local checkpoint: {len(reports)}/64 prompts saved.")
else:
    print("All 64 verified local results are present; no Modal call.")
if len(reports) != 64:
    raise RuntimeError("Full activation pass is incomplete; saved prompt checkpoints remain available.")

input_reports = {}
for row in rows:
    source_dir = pilot_dir if row["eval_id"] in pilot_rows else full_dir
    report_path = source_dir / f"{row['eval_id']}.json"
    input_reports[row["eval_id"]] = {"path": report_path.as_posix(),
                                    "sha256": hashlib.sha256(report_path.read_bytes()).hexdigest(),
                                    "source": "pilot" if row["eval_id"] in pilot_rows else "full"}
summary_path = full_dir / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary["request_key"] != request_key or summary["input_reports"] != input_reports:
        raise RuntimeError("Saved full summary differs from its source reports.")
    print("Reusing saved full activation summary; no statistics recomputed.")
else:
    from collections import Counter, defaultdict
    from statistics import fmean

    def distribution(values):
        ordered = sorted(values)
        def percentile(q):
            position = (len(ordered) - 1) * q
            lower = math.floor(position)
            upper = math.ceil(position)
            return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)
        return {"min": ordered[0], "p10": percentile(0.1), "p50": percentile(0.5),
                "p90": percentile(0.9), "max": ordered[-1]}

    buckets = defaultdict(list)
    for row in rows:
        for measurement in reports[row["eval_id"]]["measurements"]:
            annotated = dict(measurement, eval_id=row["eval_id"])
            for grouping, group in (("overall", "all"), ("dimension", row["dimension"]),
                                    ("difficulty", str(row["difficulty"]))):
                buckets[grouping, group, measurement["stage"], measurement["layer_index"]].append(annotated)
    aggregates = {"overall": [], "dimension": [], "difficulty": []}
    for (grouping, group, stage, layer), values in sorted(buckets.items()):
        aggregates[grouping].append({
            "group": group, "stage": stage, "layer_index": layer, "n_prompts": len(values),
            "means": {name: fmean(m[name] for m in values) for name in request["metric_names"]},
            "token_cosine_mean_distribution": distribution([m["token_cosine_mean"] for m in values]),
            "relative_rms_change_distribution": distribution([m["relative_rms_change"] for m in values]),
            "lowest_cosine_eval_id": min(values, key=lambda m: m["token_cosine_mean"])["eval_id"],
            "largest_change_eval_id": max(values, key=lambda m: m["relative_rms_change"])["eval_id"],
        })
    summary = {"request_key": request_key, "input_reports": input_reports,
               "n_prompts": len(rows), "n_reused_pilot": len(pilot_rows),
               "n_full_pass_prompts": len(rows) - len(pilot_rows),
               "n_measurements": sum(len(report["measurements"]) for report in reports.values()),
               "dimension_counts": dict(Counter(row["dimension"] for row in rows)),
               "difficulty_counts": dict(Counter(str(row["difficulty"]) for row in rows)),
               "aggregates": aggregates,
               "percentile_scope": "distribution across prompts of per-prompt scalar metrics; linear interpolation",
               "scope": "activation diagnostics; no blocks bypassed or selected for pruning"}
    temporary = summary_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    temporary.replace(summary_path)

print(f"Full pass complete: {summary['n_prompts']} prompts, {summary['n_measurements']} measurements; "
      f"{summary['n_reused_pilot']} pilot probes reused.")
print("Capability coverage:", summary["dimension_counts"])
print("Difficulty coverage:", summary["difficulty_counts"])
print("Overall mean image-token cosine / relative RMS change:")
for stage in ("early", "middle", "final"):
    for entry in summary["aggregates"]["overall"]:
        if entry["stage"] == stage and entry["layer_index"] in (0, 8, 16, 24, request["num_layers"] - 1):
            mean = entry["means"]
            print(f"  {stage:6s} block {entry['layer_index']:2d}: "
                  f"cosine={mean['token_cosine_mean']:.4f}, change={mean['relative_rms_change']:.4f}")
print(f"Saved summary: {summary_path}; pilot files retained at {pilot_dir}.")
print("Capability and difficulty summaries include prompt percentiles and worst-case IDs; similarity alone does not establish pruning safety.")


Reusing 5/64 local results, including all five pilot probes.
Scheduling only 59 unfinished prompts on one H100.
✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-yNw6TvJQwKI5Fwp4rRh9Ob
✓ Created objects.
└── 🔨 Created function stream_full_activations.

Loading pipeline components...:  20%|██        | 1/5 [00:01<00:04,  1.03s/it]


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]


Loading weights:   1%|▏         | 11/750 [00:00<00:07, 103.47it/s]


Loading weights:   3%|▎         | 22/750 [00:00<00:07, 92.15it/s] 


Loading weights:   4%|▍         | 32/750 [00:00<00:08, 81.85it/s]


Loading weights:   5%|▌         | 41/750 [00:00<00:08, 80.26it/s]


Loading weights:   7%|▋         | 50/750 [00:00<00:09, 76.12it/s]


Loading weights:   8%|▊         | 58/750 [00:00<00:09, 73.92it/s]


Loading weights:   9%|▉         | 66/750 [00:00<00:09, 69.92it/s]


Loading weights:  10%|█         | 76/750 [00:00<00:08, 75.07it/s]


Loading weights:  11%|█▏        | 85/75

## Analysis of the full activation pass

The full pass contains **64 prompts, 32 blocks, and three denoising stages: 6,144 records**. The five pilot probes were reused unchanged. Capability coverage is 15 aesthetics, 15 alignment, 14 creative, 10 real-world, and 10 visual-quality prompts. Difficulty levels 1–4 each contain 13 prompts; level 5 contains 12. These are diagnostic samples from our frozen internal evaluation set, not a Qwen-Image-Bench quality evaluation.

The broad pattern seen in the pilot persists: some blocks make small residual updates with very high input/output directional similarity, while the boundary blocks transform the representation substantially. The full summary also reveals important differences within the interior of the network.

**The strongest consistently small-update signals occur in blocks 2–5.** Looking across all 32 blocks, these four have the smallest maximum stage-mean relative changes. Their mean updates remain approximately 5.5%–10.9% of the input norm across the sampled stages, with mean token cosines above 0.995. They merit controlled single-block bypass tests. These measurements do not show what would happen after removing them, and the apparent similarity of adjacent blocks does not establish that the entire 2–5 region can be removed together.

The table gives mean relative RMS change across the 64 prompts at steps 0, 19, and 39. A value of 0.10 means the update norm is approximately 10% of the input norm. The final column is the **lowest of the three stage-mean token cosines**, not the lowest individual prompt or token cosine. Block indices are zero-based.

| Block index | Early change | Middle change | Final change | Lowest stage-mean cosine |
|---|---:|---:|---:|---:|
| 0 | 8.1140 | 10.2556 | 6.1345 | 0.1051 |
| 2 | 0.0868 | 0.0554 | 0.0813 | 0.9976 |
| 3 | 0.1088 | 0.0747 | 0.0800 | 0.9971 |
| 4 | 0.0846 | 0.0932 | 0.0957 | 0.9970 |
| 5 | 0.0992 | 0.1004 | 0.1003 | 0.9958 |
| 8 | 0.1698 | 0.1739 | 0.1761 | 0.9887 |
| 16 | 0.1973 | 0.2266 | 0.1940 | 0.9781 |
| 23 | 0.0819 | 0.1192 | 0.2006 | 0.9800 |
| 24 | 0.0907 | 0.1318 | 0.2379 | 0.9787 |
| 30 | 0.6830 | 0.5304 | 0.5399 | 0.8048 |
| 31 | 4.0371 | 1.7325 | 1.5529 | 0.5471 |

**The initial printed examples were useful diagnostics, but they do not constitute a ranking.** Blocks 8 and 16 also retain high directional similarity, although their relative updates are larger than those of blocks 2–5. Block 0 has mean update ratios of 6.13–10.26 and block 31 has ratios of 1.55–4.04. Block 30 also has comparatively large changes, approximately 0.53–0.68. Large relative updates can reflect representation scaling as well as changes in direction; they are not image-quality scores or direct measurements of pruning damage.

**Late-denoising behavior can reverse an early redundancy impression.** Block 23 has the smallest mean update at the early sampled step, 0.0819, but its final-step update is 0.2006. Block 24 similarly rises from 0.0907 early to 0.2379 at the final step, about 2.6 times larger, while its mean cosine falls from 0.9960 to 0.9787. Ranking blocks using only early denoising, or averaging away timestep differences, would understate these final-stage changes. The selected step indices correspond to actual scheduler timesteps recorded in each report; they are not necessarily equally spaced noise levels.

**The small-update pattern is shared across the sampled capability and difficulty groups.** For blocks 2–5, every capability-group/stage mean relative change is below 0.110, and every difficulty-group/stage mean is below 0.113. This supports the consistency of the observed signal within this probe set. Each group still contains only 10–15 or 12–13 prompts, so these averages cannot exclude damage to rare concepts or individual difficult examples. The overall mean also gives somewhat more weight to the more numerous capability groups; capability summaries should be examined separately.

**Prompt variation matters even when the mean looks attractive.** At the early step, block 2 has a mean relative change of 0.0868, a prompt-level p90 of 0.0920, and a maximum of 0.1060. Block 3 is more variable there: mean 0.1088, p90 0.1266, maximum 0.1396. For a larger-update example, block 29 has an early mean of 0.3423 but a maximum of 0.6992 on `internal-p1-0277430ca05c`. The saved worst-case IDs let us inspect the specific prompts behind these tails. These percentiles describe variation across prompt-level measurements; they differ from the within-image token percentiles recorded by each hook.

**What these results establish.** We have reproducible, checkpointed measurements that support prioritizing blocks for causal tests. High cosine can coexist with an important change in magnitude, and a small update can still affect features that the downstream network amplifies. All measurements were collected along the intact teacher's trajectories and cover only three of its 40 denoising steps. They describe image-token behavior; the text prefix was excluded even though its transformations and caches can also be affected by architectural pruning. Controlled bypass comparisons on identical inputs and checks of generated-image regressions are therefore needed to establish removal safety. No blocks have been bypassed, pruned, or selected for removal by this pass.

The 59 new images and reports, frozen full request, and summary are saved under `data/phase3/activations/full/`. The summary references the five original pilot reports under `data/phase3/activations/pilot/`. Per-prompt packets are also persisted on Modal. Local cached reruns were verified to make no Modal calls, recompute no summary statistics, and rewrite no artifacts. The retained measurements are scalar reductions; raw hidden-state tensors were not saved.


## 8. Convert activation evidence into a shortlist for individual bypass tests

We can now decide which blocks to test first using the saved 64-prompt measurements. This step runs entirely locally. It verifies the full summary, model identity, and every source-report checksum, then ranks all 32 blocks by their worst capability-group or difficulty-group p90 relative RMS change across early, middle, and final denoising. Lower values indicate smaller updates even in the higher-change portion of those sampled groups. Ties are resolved by the maximum observed prompt-level relative change and then the block index. This transparent ordering avoids an arbitrary weighted combination of metrics. Group estimates remain noisy and correlated, and this score does not measure image-quality damage.

We retain the stage-level distributions, minimum observed prompt-mean token cosine, and IDs behind the largest updates and lowest cosines, so the ranking does not discard important exceptions. The group p90 values describe variation across prompts, while the minimum cosine is the smallest prompt-mean token cosine across the three measured stages. Boundary blocks remain in the complete ranking but are excluded from the initial conservative test batch. We select four interior blocks for independent bypass tests, retain two reserve candidates, and identify the interior block with the largest ranking signal as a diagnostic comparison. That comparison block is not assumed to damage image quality before it is tested.

The four-candidate batch size is an experimental choice, not a pruning budget or quality threshold. Consecutive candidates must first be tested separately; their individual results cannot predict the effect of removing them together. Activations were measured on the intact teacher's trajectories and exclude the text prefix, so this shortlist determines test priority only. Layer-removal decisions remain pending controlled output comparisons and generated-image regression measurements.

The full ranking, frozen selection policy, candidate order, reserve candidates, comparison block, and source identities are saved in `data/phase3/bypass_shortlist.json`. Matching reruns read the saved result without recomputing scores, calling Modal, or loading any model. Changed inputs stop instead of silently replacing the decision artifact.


In [1]:
# Rank saved activation signals locally and freeze a shortlist for individual bypass tests.
import hashlib
import json
import math
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
summary_path = PHASE3_DIR / "activations/full/summary.json"
summary_bytes = summary_path.read_bytes()
summary = json.loads(summary_bytes)
manifest = json.loads((PHASE3_DIR / "activations/full/request.json").read_text(encoding="utf-8"))
architecture = json.loads((PHASE3_DIR / "architecture.json").read_text(encoding="utf-8"))
if summary["request_key"] != manifest["request_key"]:
    raise RuntimeError("Full activation summary and manifest identify different runs.")
model = manifest["request"]
if any(architecture[key] != model[key] for key in ("model_repo", "model_revision", "config_sha256")):
    raise RuntimeError("Architecture differs from the full activation run.")
num_layers = architecture["config"]["num_layers"]
if summary["n_prompts"] != 64 or summary["n_measurements"] != 64 * num_layers * 3:
    raise RuntimeError("The full activation pass is incomplete.")
for source in summary["input_reports"].values():
    if hashlib.sha256(Path(source["path"]).read_bytes()).hexdigest() != source["sha256"]:
        raise RuntimeError("A source activation report differs from the saved full summary.")

ranking_request = {
    "version": "phase3-activation-bypass-shortlist-v1",
    "summary_sha256": hashlib.sha256(summary_bytes).hexdigest(),
    "activation_request_key": summary["request_key"],
    "model_repo": model["model_repo"], "model_revision": model["model_revision"],
    "num_layers": num_layers,
    "sort_keys": ["worst_subgroup_p90_relative_change", "max_observed_relative_change", "layer_index"],
    "subgroups": ["dimension", "difficulty"], "stages": ["early", "middle", "final"],
    "boundary_layers": [0, num_layers - 1], "initial_candidate_count": 4, "reserve_count": 2,
    "control_rule": "highest-ranked update signal among interior blocks (last in ascending ranking)",
    "decision_scope": "which individual blocks to bypass-test; no approval for removal or joint pruning",
}
ranking_key = hashlib.sha256(json.dumps(ranking_request, sort_keys=True).encode()).hexdigest()
output_path = PHASE3_DIR / "bypass_shortlist.json"
if output_path.exists():
    result = json.loads(output_path.read_text(encoding="utf-8"))
    if result.get("request") != ranking_request or result.get("request_key") != ranking_key:
        raise RuntimeError("Saved shortlist belongs to different inputs; preserve and inspect it.")
    print("Reusing saved ranking and shortlist; no scores recomputed and no model calls.")
else:
    ranking = []
    for layer in range(num_layers):
        overall = [entry for entry in summary["aggregates"]["overall"] if entry["layer_index"] == layer]
        subgroups = [dict(entry, grouping=grouping) for grouping in ranking_request["subgroups"]
                    for entry in summary["aggregates"][grouping] if entry["layer_index"] == layer]
        if ({entry["stage"] for entry in overall} != set(ranking_request["stages"])
                or len(overall) != 3 or any(entry["n_prompts"] != 64 for entry in overall)):
            raise RuntimeError(f"Overall stage coverage is incomplete for block {layer}.")
        for grouping, counts in (("dimension", summary["dimension_counts"]),
                                 ("difficulty", summary["difficulty_counts"])):
            entries = [entry for entry in subgroups if entry["grouping"] == grouping]
            expected = {(str(group), stage) for group in counts for stage in ranking_request["stages"]}
            if ({(entry["group"], entry["stage"]) for entry in entries} != expected
                    or len(entries) != len(expected)
                    or any(entry["n_prompts"] != counts[entry["group"]] for entry in entries)):
                raise RuntimeError(f"Subgroup stage coverage is incomplete for block {layer}.")
        worst = max(subgroups, key=lambda entry: entry["relative_rms_change_distribution"]["p90"])
        largest = max(overall, key=lambda entry: entry["relative_rms_change_distribution"]["max"])
        lowest = min(overall, key=lambda entry: entry["token_cosine_mean_distribution"]["min"])
        ranking.append({
            "layer_index": layer, "is_boundary": layer in ranking_request["boundary_layers"],
            "worst_subgroup_p90_relative_change": worst["relative_rms_change_distribution"]["p90"],
            "worst_subgroup": {key: worst[key] for key in ("grouping", "group", "stage", "n_prompts")},
            "worst_overall_stage_p90_relative_change": max(
                entry["relative_rms_change_distribution"]["p90"] for entry in overall),
            "max_observed_relative_change": largest["relative_rms_change_distribution"]["max"],
            "largest_change_prompt": {"eval_id": largest["largest_change_eval_id"], "stage": largest["stage"]},
            "min_observed_prompt_mean_token_cosine": lowest["token_cosine_mean_distribution"]["min"],
            "lowest_cosine_prompt": {"eval_id": lowest["lowest_cosine_eval_id"], "stage": lowest["stage"]},
            "per_stage": {entry["stage"]: {"means": entry["means"],
                "relative_change_distribution": entry["relative_rms_change_distribution"],
                "token_cosine_mean_distribution": entry["token_cosine_mean_distribution"]} for entry in overall},
        })
    if any(not math.isfinite(entry[key]) for entry in ranking for key in ranking_request["sort_keys"]):
        raise RuntimeError("Ranking signals contain nonfinite values.")
    ranking.sort(key=lambda entry: tuple(entry[key] for key in ranking_request["sort_keys"]))
    for position, entry in enumerate(ranking, 1):
        entry["rank"] = position
    interior = [entry for entry in ranking if not entry["is_boundary"]]
    count = ranking_request["initial_candidate_count"]
    result = {
        "request": ranking_request, "request_key": ranking_key, "ranking": ranking,
        "candidate_layers": [entry["layer_index"] for entry in interior[:count]],
        "reserve_layers": [entry["layer_index"] for entry in interior[count:count + ranking_request["reserve_count"]]],
        "diagnostic_control_layer": interior[-1]["layer_index"],
        "removal_decision": "pending controlled bypass and generated-image regression measurements",
        "interpretation": "low relative activation updates are a test-priority heuristic, not causal importance",
    }
    temporary = output_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    temporary.replace(output_path)

print("Rank  Block  Worst subgroup p90  Maximum change  Lowest prompt cosine")
for entry in result["ranking"][:8]:
    print(f"{entry['rank']:4d}  {entry['layer_index']:5d}  "
          f"{entry['worst_subgroup_p90_relative_change']:18.4f}  "
          f"{entry['max_observed_relative_change']:14.4f}  "
          f"{entry['min_observed_prompt_mean_token_cosine']:20.4f}")
print("Individual bypass-test order:", result["candidate_layers"])
print("Reserve candidates:", result["reserve_layers"])
print("Larger-update diagnostic control:", result["diagnostic_control_layer"])
print("Removal decision:", result["removal_decision"])
print(f"Saved ranking, evidence and selection policy: {output_path}")


Rank  Block  Worst subgroup p90  Maximum change  Lowest prompt cosine
   1      2              0.0973          0.1060                0.9972
   2      4              0.1034          0.1090                0.9964
   3      5              0.1144          0.1161                0.9945
   4      3              0.1344          0.1396                0.9961
   5      1              0.1435          0.1681                0.9947
   6      6              0.1717          0.1727                0.9937
   7      9              0.1844          0.1994                0.9838
   8      8              0.1906          0.2047                0.9863
Individual bypass-test order: [2, 4, 5, 3]
Reserve candidates: [1, 6]
Larger-update diagnostic control: 30
Removal decision: pending controlled bypass and generated-image regression measurements
Saved ranking, evidence and selection policy: data\phase3\bypass_shortlist.json


## What the shortlist results mean

**Our decision is to test blocks 2, 4, 5, and 3 individually, in that order.** We have not established that any of them can be removed safely. We chose them because they make the smallest relatively consistent changes to the model's internal image representation in the measurements we collected. Indices start at zero, so block 2 is the third transformer block.

A block receives a large array of numbers representing the image being denoised and returns an updated array. We measured the size and direction of that update. A small numerical update suggests that bypassing the block might have a smaller effect, but the update could still contain essential information about an object, a letter, or a spatial relationship.

**The columns answer different questions:**

- **Worst subgroup p90 update:** for each capability group and difficulty group at each sampled denoising stage, we find the 90th percentile of the relative update size across its prompts. We then keep the largest of those values. This penalizes blocks whose updates grow in particular groups or stages. It is a descriptive sample statistic, not a 90% confidence statement or a probability that pruning will succeed.
- **Largest observed update:** the largest relative update we measured for that block over all 64 prompts and three stages. This shows an extreme that the p90 statistic can hide.
- **Lowest prompt-mean cosine:** the lowest mean directional similarity we observed for any prompt/stage. Values close to 1 mean the block's input and output feature vectors point in similar directions. A value of 0.9972 does not mean 99.72% of image quality is preserved.

Update percentages are relative to the norm of the block's input representation. They are not percentages of pixels changed, image error, or model parameters removed.

| Test order | Block index | Worst subgroup p90 update | Largest observed update | Lowest prompt-mean cosine |
|---|---:|---:|---:|---:|
| 1 | 2 | 9.7% | 10.6% | 0.9972 |
| 2 | 4 | 10.3% | 10.9% | 0.9964 |
| 3 | 5 | 11.4% | 11.6% | 0.9945 |
| 4 | 3 | 13.4% | 14.0% | 0.9961 |

**Block 2 provides a concrete example.** Its worst subgroup is the ten visual-quality prompts at the early denoising stage. The p90 relative update there is approximately 9.7% of the input norm. Across all prompt/stage measurements for block 2, the largest update is approximately 10.6%, and the lowest prompt-mean cosine is 0.9972. Together, these observations make it a reasonable first block to test. They provide no measurement of how much the generated image would change if we actually bypassed it.

**Why is the order 2, 4, 5, 3?** The primary ranking uses the worst subgroup p90 update, so smaller values come first: 9.7%, 10.3%, 11.4%, and 13.4%. Block 3 has a slightly higher lowest cosine than block 5, but its higher-change cases have larger relative updates. Cosine is retained as supporting evidence; it is not the primary ranking key.

**The reserve candidates are blocks 1 and 6.** They come next in the same ranking if we need a broader experimental comparison. **Block 30 is the diagnostic comparison block:** it has the largest update-ranking signal among interior blocks. Testing it gives us an example with a much stronger activation change signal. We still have to measure its actual output effect; a large activation change does not guarantee a large quality regression.

**How this leads to a removal decision.** An individual bypass experiment feeds the original model and the bypassed model identical latents, prompt conditioning, and timesteps, then measures the change in their velocity predictions. Generated-image checks assess the practical effect on capabilities and quality. Small prediction differences and acceptable image regressions would support considering that block for removal or replacement with a bridge. Large differences would argue for keeping it or providing a stronger replacement. Removing several adjacent blocks together requires a separate interaction test because their effects can accumulate.

The current artifact, `data/phase3/bypass_shortlist.json`, therefore records a test order and its evidence. The removal decision remains pending. No model weights or architecture were changed, and this ranking used only saved local data.


### What “velocity” means in these experiments

**Velocity is the model's prediction of how the current noisy image representation changes as the noise level changes.** It is a tensor, with one predicted change for each latent component. The latent is a compressed representation of an image, not the final RGB pixels.

At each of our 40 denoising steps, the model receives the noisy latent, prompt conditioning and timestep. Its 32 transformer blocks process those inputs, then its final output layer produces the predicted velocity. The scheduler uses that prediction to update the latent. The neural network predicts the velocity; the scheduler determines the step size.

Our saved configuration uses `FlowMatchEulerDiscreteScheduler` with deterministic Euler updates:

$$
x_{\mathrm{next}} = x_{\mathrm{current}} + (\sigma_{\mathrm{next}}-\sigma_{\mathrm{current}})\,v_{\mathrm{predicted}}.
$$

Here $x$ is the latent and $\sigma$ represents its noise level. Generation proceeds toward lower noise levels, so the difference in sigma is negative. “Velocity” is a rate with respect to this noise coordinate, not a speed in seconds or the raw difference between two transformer blocks.

For a simplified example with one latent number:

```text
Current latent value:  0.80
Current sigma:        0.60
Next sigma:           0.50
Predicted velocity:   2.00

Next value = 0.80 + (0.50 - 0.60) * 2.00 = 0.60
```

In our bypass experiments, the intact teacher and bypassed model receive exactly the same saved noisy latent, prompt conditioning and timestep. We compare their final velocity predictions:

$$
\mathrm{relative\ error} =
\frac{\lVert v_{\mathrm{bypass}}-v_{\mathrm{teacher}}\rVert_2}
{\max(\lVert v_{\mathrm{teacher}}\rVert_2,10^{-12})}.
$$

The norm is computed over all target image-token prediction components in FP32. Multiplying by 100 expresses the error as a percentage. For example, block 5's 3.69% mean final-stage error means the norm of the prediction difference is about 3.69% of the teacher prediction's norm, averaged across five prompts. It does not mean an image loses 3.69% of its quality.

For one identical input and the same Euler step size, the update difference is the velocity difference multiplied by that step size. Across full generation, an altered prediction changes the latent supplied to later steps, so effects may accumulate or interact. This is why the pilot velocity comparisons support candidate assessment but do not establish finished-image quality.

The update equation is implemented in `diffusers/schedulers/scheduling_flow_match_euler_discrete.py`; the transformer output head is in `diffusers/models/transformers/transformer_qwenimage21.py`. See the [official scheduler documentation](https://huggingface.co/docs/diffusers/api/schedulers/flow_match_euler_discrete) for the distinction between model output and the scheduler's update.


## 9. Test block 2 with paired velocity predictions on the five pilot prompts

We test the first shortlisted block by replacing its forward computation with the identity function during three diagnostic transformer calls per prompt. The teacher remains intact for the trajectory that supplies the test inputs. The five frozen pilot prompts preserve their original seeds, resolutions, 40-step schedule, guidance and pinned BF16 H100 runtime. We capture the teacher's latent input, conditioning, output velocity, model timestep and scheduler sigma at steps 0, 19 and 39. Existing activation artifacts contain scalar statistics rather than these tensors, so this step needs a new teacher trajectory to obtain reusable paired targets.

Prefix caches require special care. The intact teacher's cached text features can change when a block is skipped, even though our previous measurements considered only image tokens. We save the teacher's prefix KV tensors, then independently prefill a fresh bypass cache at the first captured input. The same bypass cache is used at the middle and final captured inputs. The skipped block performs no computation and writes no cache entries; downstream blocks recompute their own prefix features. Every teacher/bypass comparison uses identical latent inputs and prompt conditioning.

We also replay the intact teacher on the captured inputs to measure the numerical discrepancy introduced by capture and replay. The velocity comparison records relative L2 error `||v_bypass - v_teacher|| / max(||v_teacher||, epsilon)`, RMSE, flattened cosine, and teacher RMS, all reduced in FP32. The intact replay control is reported alongside the bypass error. These are causal measurements of a single skipped block at teacher states, not image-quality scores or a full bypassed sampling trajectory.

The worker persists each prompt's shared teacher tensors, all control predictions, generated teacher PNG, bypass predictions, bypass prefix-cache tensors and report to the Modal results volume. Local copies are saved under `data/phase3/bypass/teacher_targets/` and `data/phase3/bypass/block_02_pilot/`. Shared teacher targets omit the candidate layer from their identity so later candidate tests can reuse them. Prompt results are checkpointed before returning, missing local artifacts are recovered before GPU scheduling, and completed local reruns make no Modal calls or recompute the summary. Saved failures stop rather than retry expensive work automatically. The temporary forward replacement is restored in a `finally` block; model weights are never changed.


In [1]:
# Paired velocity comparisons for one block; persist shared teacher inputs and all predictions.
import base64
import hashlib
import json
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
shortlist_bytes = (PHASE3_DIR / "bypass_shortlist.json").read_bytes()
shortlist = json.loads(shortlist_bytes)
full_manifest = json.loads((PHASE3_DIR / "activations/full/request.json").read_text(encoding="utf-8"))
pilot_manifest = json.loads((PHASE3_DIR / "activations/pilot/request.json").read_text(encoding="utf-8"))
model = full_manifest["request"]
if shortlist["request"]["activation_request_key"] != full_manifest["request_key"]:
    raise RuntimeError("Shortlist and activation run identify different measurements.")
layer = shortlist["candidate_layers"][0]
rows = pilot_manifest["request"]["pilot_rows"]
if layer != 2 or len(rows) != 5:
    raise RuntimeError("This first bypass pilot expects block 2 and the five validated pilot prompts.")
source_reports = {}
for row in rows:
    path = PHASE3_DIR / "activations/pilot" / f"{row['eval_id']}.json"
    source_reports[row["eval_id"]] = json.loads(path.read_text(encoding="utf-8"))
    if hashlib.sha256(path.read_bytes()).hexdigest() != model["pilot_artifacts"][row["eval_id"]]["report_sha256"]:
        raise RuntimeError("A pilot activation report changed after the full pass.")

target_request = {
    "version": "phase3-paired-teacher-targets-v1",
    **{key: model[key] for key in ("model_repo", "model_revision", "config_sha256", "dtype",
                                  "noise_device", "num_layers", "runtime", "scheduler_class", "scheduler_config")},
    "rows": rows, "probe_sha256": model["probe_sha256"],
    "stages": ["early", "middle", "final"], "step_rule": "0, (n - 1) // 2, n - 1",
    "transformer_source_sha256": next(iter(source_reports.values()))["runtime"]["transformer_source_sha256"],
    "teacher_image_sha256": {row["eval_id"]: source_reports[row["eval_id"]]["image_sha256"] for row in rows},
    "cache_policy": "save teacher prefix KV tensors; rebuild a separate bypass prefix cache at the early step",
    "control": "replay intact teacher on captured inputs before bypassing; save control predictions and errors",
}
target_key = hashlib.sha256(json.dumps(target_request, sort_keys=True).encode()).hexdigest()
request = {"version": "phase3-block-bypass-pilot-v1", "target_request": target_request,
           "target_key": target_key, "bypassed_layer": layer,
           "shortlist_sha256": hashlib.sha256(shortlist_bytes).hexdigest(),
           "metric_definition": "velocity relative L2 error, RMSE and flattened cosine in FP32; epsilon=1e-12"}
request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = PHASE3_DIR / f"bypass/block_{layer:02d}_pilot"
targets_dir = PHASE3_DIR / "bypass/teacher_targets" / target_key
run_dir.mkdir(parents=True, exist_ok=True)
targets_dir.mkdir(parents=True, exist_ok=True)
manifest_path = run_dir / "request.json"
manifest = {"request": request, "request_key": request_key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Saved bypass pilot uses different inputs; preserve and inspect it.")
else:
    with manifest_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(manifest, indent=2) + "\n")

def validate_report(report, row):
    if report.get("request_key") != request_key or report.get("row") != row:
        raise RuntimeError("Bypass report belongs to different inputs.")
    if report["status"] != "passed":
        raise RuntimeError("Saved bypass run failed; no automatic retry: " + report["error"])
    if [m["stage"] for m in report["measurements"]] != target_request["stages"]:
        raise RuntimeError("Bypass report does not cover all three denoising stages.")

def verify_files(report, row):
    paths = [(run_dir / f"{row['eval_id']}.pt", report["outputs_sha256"]),
             (targets_dir / f"{row['eval_id']}.pt", report["targets_sha256"]),
             (targets_dir / f"{row['eval_id']}.png", report["teacher_image_sha256"])]
    for path, digest in paths:
        if not path.is_file():
            return False
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Saved bypass artifact checksum differs: {path}")
    return True

reports = {}
missing_rows = []
for row in rows:
    path = run_dir / f"{row['eval_id']}.json"
    if path.exists():
        report = json.loads(path.read_text(encoding="utf-8"))
        validate_report(report, row)
        if verify_files(report, row):
            reports[row["eval_id"]] = report
            continue
    missing_rows.append(row)

if missing_rows:
    import modal
    runtime = target_request["runtime"]
    results_volume = modal.Volume.from_name(runtime["results_volume"])
    remote_run = f"/bypass/runs/{request_key}"
    remote_targets = f"/bypass/targets/{target_key}"

    def receive_report(report, row):
        path = run_dir / f"{row['eval_id']}.json"
        if report.get("request_key") != request_key or report.get("row") != row:
            raise RuntimeError("Remote bypass report belongs to different inputs.")
        if report["status"] == "passed":
            files = [(f"{remote_run}/{row['eval_id']}.pt", run_dir / f"{row['eval_id']}.pt", report["outputs_sha256"]),
                     (f"{remote_targets}/{row['eval_id']}.pt", targets_dir / f"{row['eval_id']}.pt", report["targets_sha256"]),
                     (f"{remote_targets}/{row['eval_id']}.png", targets_dir / f"{row['eval_id']}.png", report["teacher_image_sha256"])]
            for remote, local, digest in files:
                if local.exists():
                    if hashlib.sha256(local.read_bytes()).hexdigest() != digest:
                        raise RuntimeError(f"Existing local artifact differs: {local}")
                    continue
                temporary = local.with_suffix(".tmp")
                with temporary.open("wb") as handle:
                    for chunk in results_volume.read_file(remote):
                        handle.write(chunk)
                if hashlib.sha256(temporary.read_bytes()).hexdigest() != digest:
                    raise RuntimeError("Downloaded bypass artifact checksum differs.")
                temporary.replace(local)
        if path.exists():
            if json.loads(path.read_text(encoding="utf-8")) != report:
                raise RuntimeError("Local bypass report differs from remote backup.")
        else:
            temporary = path.with_suffix(".tmp")
            temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
            temporary.replace(path)
        validate_report(report, row)
        return report

    try:
        remote_names = {Path(entry.path).name for entry in results_volume.listdir(remote_run, recursive=False)}
    except (FileNotFoundError, modal.exception.NotFoundError):
        remote_names = set()
    pending = []
    for row in missing_rows:
        if f"{row['eval_id']}.json" in remote_names:
            report = json.loads(b"".join(results_volume.read_file(f"{remote_run}/{row['eval_id']}.json")))
            reports[row["eval_id"]] = receive_report(report, row)
        else:
            if (run_dir / f"{row['eval_id']}.json").exists() or (run_dir / f"{row['eval_id']}.pt").exists():
                raise RuntimeError("Incomplete local bypass artifacts have no remote result backup.")
            pending.append(row)

    if pending:
        app = modal.App("nihonga-phase3-block2-bypass")
        weights_volume = modal.Volume.from_name(runtime["weights_volume"])
        image = (modal.Image.debian_slim(python_version=runtime["python_version"])
                 .apt_install("git").pip_install(*runtime["packages"])
                 .env({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}))

        @app.function(image=image, gpu=runtime["gpu_requested"], timeout=1800, max_containers=1,
                      serialized=True, volumes={"/root/.cache/huggingface": weights_volume.with_mount_options(read_only=True),
                                                "/phase3": results_volume})
        def paired_bypass(expected, key, jobs):
            import hashlib
            import inspect
            import json
            import math
            import time
            from datetime import datetime, timezone
            from pathlib import Path
            import torch
            from diffusers import QwenImage21Pipeline
            from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

            target_config = expected["target_request"]
            root = Path("/phase3/bypass/runs") / key
            shared = Path("/phase3/bypass/targets") / expected["target_key"]
            root.mkdir(parents=True, exist_ok=True)
            shared.mkdir(parents=True, exist_ok=True)

            def save_json(path, value):
                temporary = path.with_suffix(".tmp")
                temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
                temporary.replace(path)

            def save_tensors(path, value):
                temporary = path.with_suffix(".tmp")
                torch.save(value, temporary)
                temporary.replace(path)

            def cpu(value):
                if isinstance(value, torch.Tensor):
                    return value.detach().cpu().clone()
                if isinstance(value, dict):
                    return {k: cpu(v) for k, v in value.items()}
                if isinstance(value, (list, tuple)):
                    return type(value)(cpu(v) for v in value)
                if value is None or isinstance(value, (str, int, float, bool)):
                    return value
                raise TypeError(f"Unsupported captured value: {type(value).__name__}")

            def gpu(value):
                if isinstance(value, torch.Tensor):
                    return value.to("cuda")
                if isinstance(value, dict):
                    return {k: gpu(v) for k, v in value.items()}
                if isinstance(value, (list, tuple)):
                    return type(value)(gpu(v) for v in value)
                return value

            def cache_tensors(cache):
                return [{"k": cpu(item.k), "v": cpu(item.v)} for item in cache.layer_caches]

            def restore_cache(values):
                cache = QwenImage21KVCache(len(values))
                for item, saved in zip(cache.layer_caches, values):
                    if saved["k"] is not None:
                        item.store(gpu(saved["k"]), gpu(saved["v"]))
                return cache

            def metrics(prediction, reference):
                x, y = prediction.float(), reference.float()
                error = x - y
                norm = torch.linalg.vector_norm(y).clamp_min(1e-12)
                return {"relative_l2_error": float(torch.linalg.vector_norm(error) / norm),
                        "rmse": float(error.square().mean().sqrt()),
                        "cosine": float((x * y).sum() / (torch.linalg.vector_norm(x) * norm).clamp_min(1e-12)),
                        "teacher_rms": float(y.square().mean().sqrt())}

            save_json(root / "request.json", {"request": expected, "request_key": key})
            save_json(shared / "request.json", {"request": target_config, "request_key": expected["target_key"]})
            results_volume.commit()
            pipe = None
            load_error = None
            try:
                if "H100" not in torch.cuda.get_device_name(0):
                    raise RuntimeError("Allocated GPU differs from requested H100.")
                pipe = QwenImage21Pipeline.from_pretrained(target_config["model_repo"],
                    revision=target_config["model_revision"], dtype=getattr(torch, target_config["dtype"]),
                    local_files_only=True).to("cuda")
                pipe.set_progress_bar_config(disable=True)
                pipe.transformer.eval()
                source = Path(inspect.getfile(type(pipe.transformer)))
                if hashlib.sha256(source.read_bytes()).hexdigest() != target_config["transformer_source_sha256"]:
                    raise RuntimeError("Transformer implementation differs from the activation pilot.")
                if (len(pipe.transformer.transformer_blocks) != target_config["num_layers"]
                        or not pipe.transformer.config.causal_condition
                        or type(pipe.scheduler).__name__ != target_config["scheduler_class"]
                        or json.loads(json.dumps(dict(pipe.scheduler.config), default=str)) != target_config["scheduler_config"]):
                    raise RuntimeError("Loaded architecture or scheduler differs from the saved baseline.")
            except Exception as error:
                load_error = f"{type(error).__name__}: {error}"

            for row in jobs:
                report_path = root / f"{row['eval_id']}.json"
                if report_path.exists():
                    yield json.loads(report_path.read_text(encoding="utf-8"))
                    continue
                report = {"request_key": key, "row": row, "bypassed_layer": expected["bypassed_layer"],
                          "recorded_utc": datetime.now(timezone.utc).isoformat(), "status": "failed"}
                hooks = []
                block = None
                original_forward = None
                had_own_forward = False
                try:
                    if load_error:
                        raise RuntimeError(load_error)
                    target_path = shared / f"{row['eval_id']}.pt"
                    target_meta_path = shared / f"{row['eval_id']}.json"
                    teacher_png = shared / f"{row['eval_id']}.png"
                    if target_meta_path.exists():
                        target_meta = json.loads(target_meta_path.read_text(encoding="utf-8"))
                        if (target_meta["target_key"] != expected["target_key"] or target_meta["row"] != row
                                or hashlib.sha256(target_path.read_bytes()).hexdigest() != target_meta["targets_sha256"]
                                or hashlib.sha256(teacher_png.read_bytes()).hexdigest() != target_meta["image_sha256"]):
                            raise RuntimeError("Shared teacher artifacts differ from their saved identity.")
                        bundle = torch.load(target_path, map_location="cpu", weights_only=True)
                        report["teacher_targets_reused"] = True
                    else:
                        if target_path.exists() or teacher_png.exists():
                            raise RuntimeError("Shared teacher artifacts are incomplete; preserve and inspect them.")
                        capture = {"calls": 0, "cache": None, "inputs": [], "velocities": []}
                        stage_by_step = {0: "early", (row["num_inference_steps"] - 1) // 2: "middle",
                                         row["num_inference_steps"] - 1: "final"}
                        def before_forward(module, args, kwargs):
                            step = capture["calls"]
                            capture["calls"] += 1
                            capture["selected"] = step in stage_by_step
                            if capture["selected"]:
                                mode = "extract" if step == 0 else "cached"
                                if kwargs["kv_cache_mode"] != mode:
                                    raise RuntimeError("Unexpected teacher cache mode.")
                                capture["cache"] = kwargs["kv_cache"]
                                capture["inputs"].append({"stage": stage_by_step[step], "step_index": step,
                                    "sigma": float(pipe.scheduler.sigmas[step]),
                                    "target_tokens": math.prod(kwargs["img_shapes"][0][-1]),
                                    "kwargs": cpu({k: v for k, v in kwargs.items() if k != "kv_cache"})})
                        def after_forward(module, args, kwargs, output):
                            if capture["selected"]:
                                count = capture["inputs"][-1]["target_tokens"]
                                capture["velocities"].append(cpu(output[0][:, -count:]))
                        hooks = [pipe.transformer.register_forward_pre_hook(before_forward, with_kwargs=True),
                                 pipe.transformer.register_forward_hook(after_forward, with_kwargs=True)]
                        with torch.inference_mode():
                            generated = pipe(prompt=row["prompt"], width=row["width"], height=row["height"],
                                num_inference_steps=row["num_inference_steps"], true_cfg_scale=row["true_cfg_scale"],
                                generator=torch.Generator(target_config["noise_device"]).manual_seed(row["seed"]),
                                use_kv_cache=True).images[0]
                        for hook in hooks:
                            hook.remove()
                        hooks = []
                        if capture["calls"] != row["num_inference_steps"] or len(capture["velocities"]) != 3:
                            raise RuntimeError("Teacher capture missed selected steps.")
                        teacher_cache = cache_tensors(capture["cache"])
                        bundle = {"row": row, "target_key": expected["target_key"],
                                  "inputs": capture["inputs"], "teacher_velocities": capture["velocities"],
                                  "teacher_prefix_cache": teacher_cache, "control_velocities": [], "control_metrics": []}
                        replay_cache = QwenImage21KVCache(target_config["num_layers"])
                        with torch.inference_mode(), pipe.transformer.cache_context("cond"):
                            for item, reference in zip(bundle["inputs"], bundle["teacher_velocities"]):
                                kwargs = gpu(item["kwargs"])
                                kwargs["kv_cache"] = replay_cache if item["stage"] == "early" else restore_cache(teacher_cache)
                                prediction = pipe.transformer(**kwargs)[0][:, -item["target_tokens"]:]
                                bundle["control_metrics"].append(metrics(prediction, gpu(reference)))
                                bundle["control_velocities"].append(cpu(prediction))
                        save_tensors(target_path, bundle)
                        temporary_png = teacher_png.with_suffix(".tmp")
                        generated.save(temporary_png, format="PNG")
                        temporary_png.replace(teacher_png)
                        target_meta = {"target_key": expected["target_key"], "row": row,
                            "targets_sha256": hashlib.sha256(target_path.read_bytes()).hexdigest(),
                            "image_sha256": hashlib.sha256(teacher_png.read_bytes()).hexdigest()}
                        save_json(target_meta_path, target_meta)
                        results_volume.commit()
                        report["teacher_targets_reused"] = False
                        del capture

                    block = pipe.transformer.transformer_blocks[expected["bypassed_layer"]]
                    had_own_forward = "forward" in block.__dict__
                    original_forward = block.forward
                    bypass_calls = {"count": 0}
                    def identity_forward(*args, **kwargs):
                        bypass_calls["count"] += 1
                        return kwargs["hidden_states"] if "hidden_states" in kwargs else args[0]
                    block.forward = identity_forward
                    student_cache = QwenImage21KVCache(target_config["num_layers"])
                    predictions = []
                    measurements = []
                    with torch.inference_mode(), pipe.transformer.cache_context("cond"):
                        for item, reference, control in zip(bundle["inputs"], bundle["teacher_velocities"], bundle["control_metrics"]):
                            kwargs = gpu(item["kwargs"])
                            kwargs["kv_cache"] = student_cache
                            prediction = pipe.transformer(**kwargs)[0][:, -item["target_tokens"]:]
                            predictions.append(cpu(prediction))
                            measured = metrics(prediction, gpu(reference))
                            if not all(math.isfinite(value) for value in measured.values()):
                                raise RuntimeError("Bypass metrics contain nonfinite values.")
                            measurements.append({"stage": item["stage"], "step_index": item["step_index"],
                                "sigma": item["sigma"], "target_tokens": item["target_tokens"],
                                "bypass": measured, "intact_replay_control": control})
                    if bypass_calls["count"] != 3 or student_cache.get_layer(expected["bypassed_layer"]).k is not None:
                        raise RuntimeError("The selected block was not consistently bypassed.")
                    outputs_path = root / f"{row['eval_id']}.pt"
                    save_tensors(outputs_path, {"row": row, "request_key": key, "bypassed_layer": expected["bypassed_layer"],
                        "predictions": predictions, "bypass_prefix_cache": cache_tensors(student_cache)})
                    report.update({"status": "passed", "measurements": measurements,
                        "targets_sha256": target_meta["targets_sha256"], "teacher_image_sha256": target_meta["image_sha256"],
                        "outputs_sha256": hashlib.sha256(outputs_path.read_bytes()).hexdigest(),
                        "teacher_png_matches_activation": target_meta["image_sha256"] == target_config["teacher_image_sha256"][row["eval_id"]]})
                except Exception as error:
                    report["error"] = f"{type(error).__name__}: {error}"
                finally:
                    for hook in hooks:
                        hook.remove()
                    if block is not None and original_forward is not None:
                        if had_own_forward:
                            block.forward = original_forward
                        else:
                            del block.forward
                save_json(report_path, report)
                results_volume.commit()
                print(f"Block {expected['bypassed_layer']} bypass: {row['dimension']} ({report['status']}).")
                yield report
                if report["status"] != "passed":
                    break

        with modal.enable_output(), app.run():
            by_id = {row["eval_id"]: row for row in rows}
            for report in paired_bypass.remote_gen(request, request_key, pending):
                row = by_id[report["row"]["eval_id"]]
                reports[row["eval_id"]] = receive_report(report, row)
                print(f"Saved paired inputs, predictions and report: {len(reports)}/5 prompts.")
else:
    print("Reusing all five verified local bypass results; no Modal call.")

if len(reports) != 5:
    raise RuntimeError("The bypass pilot is incomplete; completed prompt artifacts remain saved.")
summary_path = run_dir / "summary.json"
report_hashes = {row["eval_id"]: hashlib.sha256((run_dir / f"{row['eval_id']}.json").read_bytes()).hexdigest() for row in rows}
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary["request_key"] != request_key or summary["report_hashes"] != report_hashes:
        raise RuntimeError("Saved bypass summary differs from its source reports.")
else:
    from statistics import fmean
    stages = []
    for stage in target_request["stages"]:
        values = [m for report in reports.values() for m in report["measurements"] if m["stage"] == stage]
        stages.append({"stage": stage, "n_prompts": len(values),
            "mean_relative_l2_error": fmean(m["bypass"]["relative_l2_error"] for m in values),
            "max_relative_l2_error": max(m["bypass"]["relative_l2_error"] for m in values),
            "mean_velocity_cosine": fmean(m["bypass"]["cosine"] for m in values),
            "max_intact_replay_relative_error": max(m["intact_replay_control"]["relative_l2_error"] for m in values)})
    summary = {"request_key": request_key, "bypassed_layer": layer, "report_hashes": report_hashes,
               "n_prompts": 5, "n_comparisons": 15, "stages": stages,
               "teacher_png_matches_activation_count": sum(r["teacher_png_matches_activation"] for r in reports.values()),
               "scope": "paired velocity pilot on teacher trajectories; no bypassed full-trajectory image evaluation"}
    temporary = summary_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    temporary.replace(summary_path)
print(f"Block {layer} bypass pilot: 5 prompts, 15 paired velocity comparisons.")
for stage in summary["stages"]:
    print(f"  {stage['stage']:6s}: mean error={100*stage['mean_relative_l2_error']:.3f}%, "
          f"max error={100*stage['max_relative_l2_error']:.3f}%, cosine={stage['mean_velocity_cosine']:.6f}, "
          f"intact replay max error={100*stage['max_intact_replay_relative_error']:.6f}%")
print("Teacher PNG matches prior activation PNG:", summary["teacher_png_matches_activation_count"], "/5")
print(f"Saved reports and bypass tensors: {run_dir}")
print(f"Saved reusable teacher inputs, prefix caches, control predictions and velocities: {targets_dir}")
print("These percentages measure velocity error, not image-quality loss; the removal decision remains pending.")


✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-oQYDAyssSkFwyHldSHaTze
✓ Created objects.
└── 🔨 Created function paired_bypass.

Loading pipeline components...:  20%|██        | 1/5 [00:00<00:02,  1.61it/s]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Loading checkpoint shards:  50%|█████     | 1/2 [00:07<00:07,  7.28s/it]


Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.93s/it]

Loading pipeline components...:  60%|██████    | 3/5 [00:12<00:08,  4.01s/it]


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]


Loading weights:   2%|▏         | 12/750 [00:00<00:08, 87.40it/s]


Loading weights:   3%|▎         | 21/750 [00:00<00:11, 65.64it/s]


Loading weights:   4%|▍         | 30/750 [00:00<00:09, 73.44it/s]


Loading weights:   5%|▌         | 38/750 [00:00<00:10, 70.34it/s]


Loading weights:   7%|▋         | 56/750 [00:00<00:07, 96.44it/s]


Loading weights:   9%|▉         | 66/750 [00:00<00:07, 87.94it/s]


Loa

## What the block 2 bypass experiment tells us

**Block 2 is not behaving like an identity function at the model output.** Skipping it changed the model's predicted denoising velocity by an average of about 16.4% early, 4.1% in the middle, and 19.0% at the final sampled step. Its small hidden-state update made it a sensible first candidate to test, but downstream computations and changes in cached prefix features can propagate those differences. Activation similarity does not bound the final prediction error.

The model's velocity is the numerical update it predicts for the noisy latent representation of an image. A velocity error of 16% means `||v_bypass - v_teacher|| / ||v_teacher||` is approximately 0.16. It does not mean that image quality fell by 16%, or that 16% of pixels changed. We evaluated five prompts, one per capability group, at three teacher-generated latent states per prompt.

| Denoising stage | Mean velocity error | Largest velocity error | Mean velocity cosine | Intact replay error |
|---|---:|---:|---:|---:|
| Early | 16.41% | 19.41% | 0.986416 | 0.000000% |
| Middle | 4.15% | 4.74% | 0.999242 | 0.000000% |
| Final | 18.96% | 21.40% | 0.982153 | 0.000000% |

**The intact replay control validates the pairing.** Replaying the original model on the captured inputs produced zero relative error for all 15 comparisons. The saved replay and reference velocity tensors were also checked to be exactly equal. We therefore have no observed capture/replay discrepancy in these comparisons. Bypassing block 2 changed the predictions on those same inputs, with a separately rebuilt prefix cache. The skipped block's cache remained empty while every retained block populated its own cache.

**The effect is strongly timestep-dependent.** The middle-stage error is much smaller than the early or final error. The final-stage maximum reaches 21.40%, so averaging all stages into one number would conceal an important response at the end of denoising. We still measured only steps 0, 19 and 39 of a 40-step schedule; behavior at unsampled steps is unknown.

Relative errors also depend on the teacher velocity's magnitude. Across these prompts, mean RMSE was 0.2687 early, 0.0590 in the middle, and 0.2154 at the final step. Mean teacher RMS was 1.6323, 1.4362, and 1.1468, respectively. The final relative error is higher than the early relative error despite a smaller mean absolute RMSE; the smaller teacher velocity magnitude contributes to that pattern. These units refer to latent velocities, not pixel intensity.

**Why cosine alone would be misleading.** Mean velocity cosines remain close to 1, especially in the middle stage. Similar directions can still accompany differences in magnitude or changes to a small but important subset of features. The relative-error metric exposes changes that a cosine-only assessment could miss.

**Teacher image repeatability needs a separate note.** Four regenerated intact-teacher PNGs matched the earlier activation-run PNGs byte for byte. The real-world prompt `internal-p1-425ec4e55730` differed: its mean absolute RGB channel difference was 0.1866 on the 0–255 scale, its maximum channel difference was 100, and 28.9% of pixels changed in at least one channel. The exact source of this cross-run variation has not been established. These are repeatability statistics, not perceptual-quality scores. The paired velocity comparisons use the newly captured teacher targets and their exact replay controls, rather than assuming that the earlier run is bit-identical. Pixel diagnostics are saved in `teacher_image_repeatability.json` beside the bypass reports.

**The removal decision remains open.** This pilot provides evidence of a measurable output effect from bypassing block 2. It does not justify calling the block dispensable, quantify image-quality damage, or establish an acceptance threshold. The captured inputs belong to the intact teacher's trajectories; a bypassed model generating a complete image can follow a different trajectory and accumulate different errors. Comparisons with other shortlisted blocks, broader prompt coverage, and generated-image regression checks are needed before choosing a removal or bridge configuration. The observed prediction errors also provide concrete targets for the planned bridge training and architecture healing.

All captured latents, conditioning, teacher prefix caches, teacher velocities, replay predictions, bypass predictions, and bypass prefix caches are saved as tensor artifacts, with checksums in the reports. The teacher targets are shared across candidate tests so they do not have to be generated again. Completed local reruns were verified to make no Modal calls, recompute no summary statistics, and rewrite no experiment artifacts. No bypassed full-trajectory image has been generated in this step, and no model weights were modified.


### Step 10 — Compare block 4 with block 2 on identical saved teacher inputs

We temporarily replace block 4 with identity and measure the resulting velocity prediction at the early, middle and final denoising stages for the same five pilot prompts. Identity passes hidden states through unchanged; this is a diagnostic of what happens when the block is skipped. The learned bridge in Phase 4 will instead approximate the missing block's update.

We reuse the saved teacher inputs, velocities and exact replay controls from the block-2 experiment. Teacher regeneration is disabled. The bypass builds its own prefix cache with block 4 skipped, so downstream conditioning reflects the intervention. We save all predictions, caches, reports and the paired comparison locally and on Modal. Completed reruns verify checksums and read cached summaries without GPU calls. These measurements concern prediction error on teacher trajectories; they do not measure finished-image quality or learned bridge performance.


In [1]:
# Paired velocity comparisons for one block; persist shared teacher inputs and all predictions.
import base64
import hashlib
import json
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
shortlist_bytes = (PHASE3_DIR / "bypass_shortlist.json").read_bytes()
shortlist = json.loads(shortlist_bytes)
full_manifest = json.loads((PHASE3_DIR / "activations/full/request.json").read_text(encoding="utf-8"))
pilot_manifest = json.loads((PHASE3_DIR / "activations/pilot/request.json").read_text(encoding="utf-8"))
model = full_manifest["request"]
if shortlist["request"]["activation_request_key"] != full_manifest["request_key"]:
    raise RuntimeError("Shortlist and activation run identify different measurements.")
layer = shortlist["candidate_layers"][1]
rows = pilot_manifest["request"]["pilot_rows"]
if layer != 4 or len(rows) != 5:
    raise RuntimeError("This second bypass pilot expects block 4 and the five validated pilot prompts.")
source_reports = {}
for row in rows:
    path = PHASE3_DIR / "activations/pilot" / f"{row['eval_id']}.json"
    source_reports[row["eval_id"]] = json.loads(path.read_text(encoding="utf-8"))
    if hashlib.sha256(path.read_bytes()).hexdigest() != model["pilot_artifacts"][row["eval_id"]]["report_sha256"]:
        raise RuntimeError("A pilot activation report changed after the full pass.")

target_request = {
    "version": "phase3-paired-teacher-targets-v1",
    **{key: model[key] for key in ("model_repo", "model_revision", "config_sha256", "dtype",
                                  "noise_device", "num_layers", "runtime", "scheduler_class", "scheduler_config")},
    "rows": rows, "probe_sha256": model["probe_sha256"],
    "stages": ["early", "middle", "final"], "step_rule": "0, (n - 1) // 2, n - 1",
    "transformer_source_sha256": next(iter(source_reports.values()))["runtime"]["transformer_source_sha256"],
    "teacher_image_sha256": {row["eval_id"]: source_reports[row["eval_id"]]["image_sha256"] for row in rows},
    "cache_policy": "save teacher prefix KV tensors; rebuild a separate bypass prefix cache at the early step",
    "control": "replay intact teacher on captured inputs before bypassing; save control predictions and errors",
}
target_key = hashlib.sha256(json.dumps(target_request, sort_keys=True).encode()).hexdigest()

baseline_dir = PHASE3_DIR / "bypass/block_02_pilot"
baseline_manifest = json.loads((baseline_dir / "request.json").read_text(encoding="utf-8"))
baseline_summary_bytes = (baseline_dir / "summary.json").read_bytes()
baseline_summary = json.loads(baseline_summary_bytes)
if baseline_manifest["request"]["target_key"] != target_key:
    raise RuntimeError("Block 4 must use precisely the same teacher inputs as block 2.")
baseline_reports = {}
for row in rows:
    path = baseline_dir / f"{row['eval_id']}.json"
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != baseline_summary["report_hashes"][row["eval_id"]]:
        raise RuntimeError("Block-2 comparison source changed.")
    report = json.loads(raw)
    if report["status"] != "passed" or report["request_key"] != baseline_manifest["request_key"]:
        raise RuntimeError("Block-2 comparison source is invalid.")
    baseline_reports[row["eval_id"]] = report
    for extension, digest in [("pt", report["targets_sha256"]), ("png", report["teacher_image_sha256"])]:
        path = PHASE3_DIR / "bypass/teacher_targets" / target_key / f"{row['eval_id']}.{extension}"
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError("Shared local teacher cache is missing or changed; no regeneration allowed.")

request = {"version": "phase3-block-bypass-pilot-v1", "require_cached_teacher": True, "target_request": target_request,
           "target_key": target_key, "bypassed_layer": layer,
           "shortlist_sha256": hashlib.sha256(shortlist_bytes).hexdigest(),
           "metric_definition": "velocity relative L2 error, RMSE and flattened cosine in FP32; epsilon=1e-12"}
request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = PHASE3_DIR / f"bypass/block_{layer:02d}_pilot"
targets_dir = PHASE3_DIR / "bypass/teacher_targets" / target_key
run_dir.mkdir(parents=True, exist_ok=True)
targets_dir.mkdir(parents=True, exist_ok=True)
manifest_path = run_dir / "request.json"
manifest = {"request": request, "request_key": request_key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Saved bypass pilot uses different inputs; preserve and inspect it.")
else:
    with manifest_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(manifest, indent=2) + "\n")

def validate_report(report, row):
    if report.get("request_key") != request_key or report.get("row") != row:
        raise RuntimeError("Bypass report belongs to different inputs.")
    if report.get("teacher_targets_reused") is not True:
        raise RuntimeError("Block 4 unexpectedly regenerated teacher targets.")
    if report["status"] != "passed":
        raise RuntimeError("Saved bypass run failed; no automatic retry: " + report["error"])
    if [m["stage"] for m in report["measurements"]] != target_request["stages"]:
        raise RuntimeError("Bypass report does not cover all three denoising stages.")

def verify_files(report, row):
    paths = [(run_dir / f"{row['eval_id']}.pt", report["outputs_sha256"]),
             (targets_dir / f"{row['eval_id']}.pt", report["targets_sha256"]),
             (targets_dir / f"{row['eval_id']}.png", report["teacher_image_sha256"])]
    for path, digest in paths:
        if not path.is_file():
            return False
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Saved bypass artifact checksum differs: {path}")
    return True

reports = {}
missing_rows = []
for row in rows:
    path = run_dir / f"{row['eval_id']}.json"
    if path.exists():
        report = json.loads(path.read_text(encoding="utf-8"))
        validate_report(report, row)
        if verify_files(report, row):
            reports[row["eval_id"]] = report
            continue
    missing_rows.append(row)

if missing_rows:
    import modal
    runtime = target_request["runtime"]
    results_volume = modal.Volume.from_name(runtime["results_volume"])
    remote_run = f"/bypass/runs/{request_key}"
    remote_targets = f"/bypass/targets/{target_key}"

    def receive_report(report, row):
        path = run_dir / f"{row['eval_id']}.json"
        if report.get("request_key") != request_key or report.get("row") != row:
            raise RuntimeError("Remote bypass report belongs to different inputs.")
        if report["status"] == "passed":
            files = [(f"{remote_run}/{row['eval_id']}.pt", run_dir / f"{row['eval_id']}.pt", report["outputs_sha256"]),
                     (f"{remote_targets}/{row['eval_id']}.pt", targets_dir / f"{row['eval_id']}.pt", report["targets_sha256"]),
                     (f"{remote_targets}/{row['eval_id']}.png", targets_dir / f"{row['eval_id']}.png", report["teacher_image_sha256"])]
            for remote, local, digest in files:
                if local.exists():
                    if hashlib.sha256(local.read_bytes()).hexdigest() != digest:
                        raise RuntimeError(f"Existing local artifact differs: {local}")
                    continue
                temporary = local.with_suffix(".tmp")
                with temporary.open("wb") as handle:
                    for chunk in results_volume.read_file(remote):
                        handle.write(chunk)
                if hashlib.sha256(temporary.read_bytes()).hexdigest() != digest:
                    raise RuntimeError("Downloaded bypass artifact checksum differs.")
                temporary.replace(local)
        if path.exists():
            if json.loads(path.read_text(encoding="utf-8")) != report:
                raise RuntimeError("Local bypass report differs from remote backup.")
        else:
            temporary = path.with_suffix(".tmp")
            temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
            temporary.replace(path)
        validate_report(report, row)
        return report

    try:
        remote_names = {Path(entry.path).name for entry in results_volume.listdir(remote_run, recursive=False)}
    except (FileNotFoundError, modal.exception.NotFoundError):
        remote_names = set()
    pending = []
    for row in missing_rows:
        if f"{row['eval_id']}.json" in remote_names:
            report = json.loads(b"".join(results_volume.read_file(f"{remote_run}/{row['eval_id']}.json")))
            reports[row["eval_id"]] = receive_report(report, row)
        else:
            if (run_dir / f"{row['eval_id']}.json").exists() or (run_dir / f"{row['eval_id']}.pt").exists():
                raise RuntimeError("Incomplete local bypass artifacts have no remote result backup.")
            pending.append(row)

    if pending:
        app = modal.App("nihonga-phase3-block4-bypass")
        weights_volume = modal.Volume.from_name(runtime["weights_volume"])
        image = (modal.Image.debian_slim(python_version=runtime["python_version"])
                 .apt_install("git").pip_install(*runtime["packages"])
                 .env({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}))

        @app.function(image=image, gpu=runtime["gpu_requested"], timeout=1800, max_containers=1,
                      serialized=True, volumes={"/root/.cache/huggingface": weights_volume.with_mount_options(read_only=True),
                                                "/phase3": results_volume})
        def paired_bypass(expected, key, jobs):
            import hashlib
            import inspect
            import json
            import math
            import time
            from datetime import datetime, timezone
            from pathlib import Path
            import torch
            from diffusers import QwenImage21Pipeline
            from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

            target_config = expected["target_request"]
            root = Path("/phase3/bypass/runs") / key
            shared = Path("/phase3/bypass/targets") / expected["target_key"]
            root.mkdir(parents=True, exist_ok=True)
            shared.mkdir(parents=True, exist_ok=True)

            def save_json(path, value):
                temporary = path.with_suffix(".tmp")
                temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
                temporary.replace(path)

            def save_tensors(path, value):
                temporary = path.with_suffix(".tmp")
                torch.save(value, temporary)
                temporary.replace(path)

            def cpu(value):
                if isinstance(value, torch.Tensor):
                    return value.detach().cpu().clone()
                if isinstance(value, dict):
                    return {k: cpu(v) for k, v in value.items()}
                if isinstance(value, (list, tuple)):
                    return type(value)(cpu(v) for v in value)
                if value is None or isinstance(value, (str, int, float, bool)):
                    return value
                raise TypeError(f"Unsupported captured value: {type(value).__name__}")

            def gpu(value):
                if isinstance(value, torch.Tensor):
                    return value.to("cuda")
                if isinstance(value, dict):
                    return {k: gpu(v) for k, v in value.items()}
                if isinstance(value, (list, tuple)):
                    return type(value)(gpu(v) for v in value)
                return value

            def cache_tensors(cache):
                return [{"k": cpu(item.k), "v": cpu(item.v)} for item in cache.layer_caches]

            def restore_cache(values):
                cache = QwenImage21KVCache(len(values))
                for item, saved in zip(cache.layer_caches, values):
                    if saved["k"] is not None:
                        item.store(gpu(saved["k"]), gpu(saved["v"]))
                return cache

            def metrics(prediction, reference):
                x, y = prediction.float(), reference.float()
                error = x - y
                norm = torch.linalg.vector_norm(y).clamp_min(1e-12)
                return {"relative_l2_error": float(torch.linalg.vector_norm(error) / norm),
                        "rmse": float(error.square().mean().sqrt()),
                        "cosine": float((x * y).sum() / (torch.linalg.vector_norm(x) * norm).clamp_min(1e-12)),
                        "teacher_rms": float(y.square().mean().sqrt())}

            save_json(root / "request.json", {"request": expected, "request_key": key})
            save_json(shared / "request.json", {"request": target_config, "request_key": expected["target_key"]})
            results_volume.commit()
            pipe = None
            load_error = None
            try:
                if "H100" not in torch.cuda.get_device_name(0):
                    raise RuntimeError("Allocated GPU differs from requested H100.")
                pipe = QwenImage21Pipeline.from_pretrained(target_config["model_repo"],
                    revision=target_config["model_revision"], dtype=getattr(torch, target_config["dtype"]),
                    local_files_only=True).to("cuda")
                pipe.set_progress_bar_config(disable=True)
                pipe.transformer.eval()
                source = Path(inspect.getfile(type(pipe.transformer)))
                if hashlib.sha256(source.read_bytes()).hexdigest() != target_config["transformer_source_sha256"]:
                    raise RuntimeError("Transformer implementation differs from the activation pilot.")
                if (len(pipe.transformer.transformer_blocks) != target_config["num_layers"]
                        or not pipe.transformer.config.causal_condition
                        or type(pipe.scheduler).__name__ != target_config["scheduler_class"]
                        or json.loads(json.dumps(dict(pipe.scheduler.config), default=str)) != target_config["scheduler_config"]):
                    raise RuntimeError("Loaded architecture or scheduler differs from the saved baseline.")
            except Exception as error:
                load_error = f"{type(error).__name__}: {error}"

            for row in jobs:
                report_path = root / f"{row['eval_id']}.json"
                if report_path.exists():
                    yield json.loads(report_path.read_text(encoding="utf-8"))
                    continue
                report = {"request_key": key, "row": row, "bypassed_layer": expected["bypassed_layer"],
                          "recorded_utc": datetime.now(timezone.utc).isoformat(), "status": "failed"}
                hooks = []
                block = None
                original_forward = None
                had_own_forward = False
                try:
                    if load_error:
                        raise RuntimeError(load_error)
                    target_path = shared / f"{row['eval_id']}.pt"
                    target_meta_path = shared / f"{row['eval_id']}.json"
                    teacher_png = shared / f"{row['eval_id']}.png"
                    if target_meta_path.exists():
                        target_meta = json.loads(target_meta_path.read_text(encoding="utf-8"))
                        if (target_meta["target_key"] != expected["target_key"] or target_meta["row"] != row
                                or hashlib.sha256(target_path.read_bytes()).hexdigest() != target_meta["targets_sha256"]
                                or hashlib.sha256(teacher_png.read_bytes()).hexdigest() != target_meta["image_sha256"]):
                            raise RuntimeError("Shared teacher artifacts differ from their saved identity.")
                        bundle = torch.load(target_path, map_location="cpu", weights_only=True)
                        report["teacher_targets_reused"] = True
                    else:
                        raise RuntimeError("Saved teacher targets are required; teacher regeneration is disabled.")

                    block = pipe.transformer.transformer_blocks[expected["bypassed_layer"]]
                    had_own_forward = "forward" in block.__dict__
                    original_forward = block.forward
                    bypass_calls = {"count": 0}
                    def identity_forward(*args, **kwargs):
                        bypass_calls["count"] += 1
                        return kwargs["hidden_states"] if "hidden_states" in kwargs else args[0]
                    block.forward = identity_forward
                    student_cache = QwenImage21KVCache(target_config["num_layers"])
                    predictions = []
                    measurements = []
                    with torch.inference_mode(), pipe.transformer.cache_context("cond"):
                        for item, reference, control in zip(bundle["inputs"], bundle["teacher_velocities"], bundle["control_metrics"]):
                            kwargs = gpu(item["kwargs"])
                            kwargs["kv_cache"] = student_cache
                            prediction = pipe.transformer(**kwargs)[0][:, -item["target_tokens"]:]
                            predictions.append(cpu(prediction))
                            measured = metrics(prediction, gpu(reference))
                            if not all(math.isfinite(value) for value in measured.values()):
                                raise RuntimeError("Bypass metrics contain nonfinite values.")
                            measurements.append({"stage": item["stage"], "step_index": item["step_index"],
                                "sigma": item["sigma"], "target_tokens": item["target_tokens"],
                                "bypass": measured, "intact_replay_control": control})
                    if bypass_calls["count"] != 3 or student_cache.get_layer(expected["bypassed_layer"]).k is not None:
                        raise RuntimeError("The selected block was not consistently bypassed.")
                    outputs_path = root / f"{row['eval_id']}.pt"
                    save_tensors(outputs_path, {"row": row, "request_key": key, "bypassed_layer": expected["bypassed_layer"],
                        "predictions": predictions, "bypass_prefix_cache": cache_tensors(student_cache)})
                    report.update({"status": "passed", "measurements": measurements,
                        "targets_sha256": target_meta["targets_sha256"], "teacher_image_sha256": target_meta["image_sha256"],
                        "outputs_sha256": hashlib.sha256(outputs_path.read_bytes()).hexdigest(),
                        "teacher_png_matches_activation": target_meta["image_sha256"] == target_config["teacher_image_sha256"][row["eval_id"]]})
                except Exception as error:
                    report["error"] = f"{type(error).__name__}: {error}"
                finally:
                    for hook in hooks:
                        hook.remove()
                    if block is not None and original_forward is not None:
                        if had_own_forward:
                            block.forward = original_forward
                        else:
                            del block.forward
                save_json(report_path, report)
                results_volume.commit()
                print(f"Block {expected['bypassed_layer']} bypass: {row['dimension']} ({report['status']}).")
                yield report
                if report["status"] != "passed":
                    break

        with modal.enable_output(), app.run():
            by_id = {row["eval_id"]: row for row in rows}
            for report in paired_bypass.remote_gen(request, request_key, pending):
                row = by_id[report["row"]["eval_id"]]
                reports[row["eval_id"]] = receive_report(report, row)
                print(f"Saved paired inputs, predictions and report: {len(reports)}/5 prompts.")
else:
    print("Reusing all five verified local bypass results; no Modal call.")

if len(reports) != 5:
    raise RuntimeError("The bypass pilot is incomplete; completed prompt artifacts remain saved.")
summary_path = run_dir / "summary.json"
report_hashes = {row["eval_id"]: hashlib.sha256((run_dir / f"{row['eval_id']}.json").read_bytes()).hexdigest() for row in rows}
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary["request_key"] != request_key or summary["report_hashes"] != report_hashes:
        raise RuntimeError("Saved bypass summary differs from its source reports.")
else:
    from statistics import fmean
    stages = []
    for stage in target_request["stages"]:
        values = [m for report in reports.values() for m in report["measurements"] if m["stage"] == stage]
        stages.append({"stage": stage, "n_prompts": len(values),
            "mean_relative_l2_error": fmean(m["bypass"]["relative_l2_error"] for m in values),
            "max_relative_l2_error": max(m["bypass"]["relative_l2_error"] for m in values),
            "mean_velocity_cosine": fmean(m["bypass"]["cosine"] for m in values),
            "max_intact_replay_relative_error": max(m["intact_replay_control"]["relative_l2_error"] for m in values)})
    summary = {"request_key": request_key, "bypassed_layer": layer, "report_hashes": report_hashes,
               "n_prompts": 5, "n_comparisons": 15, "stages": stages,
               "teacher_png_matches_activation_count": sum(r["teacher_png_matches_activation"] for r in reports.values()),
               "scope": "paired velocity pilot on teacher trajectories; no bypassed full-trajectory image evaluation"}
    temporary = summary_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    temporary.replace(summary_path)
print(f"Block {layer} bypass pilot: 5 prompts, 15 paired velocity comparisons.")
for stage in summary["stages"]:
    print(f"  {stage['stage']:6s}: mean error={100*stage['mean_relative_l2_error']:.3f}%, "
          f"max error={100*stage['max_relative_l2_error']:.3f}%, cosine={stage['mean_velocity_cosine']:.6f}, "
          f"intact replay max error={100*stage['max_intact_replay_relative_error']:.6f}%")
print("Teacher PNG matches prior activation PNG:", summary["teacher_png_matches_activation_count"], "/5")
print(f"Saved reports and bypass tensors: {run_dir}")
print(f"Saved reusable teacher inputs, prefix caches, control predictions and velocities: {targets_dir}")
print("These percentages measure velocity error, not image-quality loss; the removal decision remains pending.")

comparison_path = run_dir / "comparison_with_block_02.json"
comparison_identity = {"target_key": target_key, "block_02_summary_sha256": hashlib.sha256(baseline_summary_bytes).hexdigest(),
                       "block_04_report_hashes": report_hashes, "block_04_request_key": request_key}
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    if comparison["identity"] != comparison_identity:
        raise RuntimeError("Saved comparison differs from its source results.")
else:
    paired = []
    for row in rows:
        a, b = baseline_reports[row["eval_id"]], reports[row["eval_id"]]
        if a["targets_sha256"] != b["targets_sha256"]:
            raise RuntimeError("The two candidates used different teacher targets.")
        for first, second in zip(a["measurements"], b["measurements"]):
            if (first["stage"] != second["stage"] or first["step_index"] != second["step_index"]
                    or first["intact_replay_control"] != second["intact_replay_control"]):
                raise RuntimeError("Candidate comparisons differ in steps or controls.")
            paired.append({"eval_id": row["eval_id"], "dimension": row["dimension"], "stage": first["stage"],
                           "block_02_error": first["bypass"]["relative_l2_error"],
                           "block_04_error": second["bypass"]["relative_l2_error"]})
    comparison = {"identity": comparison_identity, "paired": paired,
                  "block_04_lower_error_count": sum(v["block_04_error"] < v["block_02_error"] for v in paired),
                  "n_comparisons": len(paired), "stages": [
                      {"stage": a["stage"], "block_02_mean_error": a["mean_relative_l2_error"],
                       "block_04_mean_error": b["mean_relative_l2_error"],
                       "block_02_max_error": a["max_relative_l2_error"], "block_04_max_error": b["max_relative_l2_error"]}
                      for a, b in zip(baseline_summary["stages"], summary["stages"])]}
    with comparison_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(comparison, indent=2) + "\n")
print("Block 4 has lower error than block 2 in", comparison["block_04_lower_error_count"], "/15 paired comparisons.")
for stage in comparison["stages"]:
    print(f"  {stage['stage']}: block 2={100*stage['block_02_mean_error']:.3f}%, block 4={100*stage['block_04_mean_error']:.3f}%")


✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-xVOYtJCQaAZz2s1gUjlxbe
✓ Created objects.
└── 🔨 Created function paired_bypass.

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]


Loading weights:   1%|          | 8/750 [00:00<00:09, 79.07it/s]


Loading weights:   2%|▏         | 16/750 [00:00<00:10, 72.57it/s]


Loading weights:   3%|▎         | 24/750 [00:00<00:12, 60.17it/s]


Loading weights:   4%|▍         | 31/750 [00:00<00:13, 52.91it/s]


Loading weights:   5%|▌         | 41/750 [00:00<00:11, 63.81it/s]


Loading weights:   6%|▋         | 48/750 [00:00<00:11, 62.98it/s]


Loading weights:   7%|▋         | 55/750 [00:00<00:11, 60.65it/s]


Loading weights:   8%|▊         | 62/750 [00:01<00:12, 56.83it/s]


Loading weights:   9%|▉         | 70/750 [00:01<00:11, 58.06it/s]


Loading weights:  10%|█         | 76/750 [00:01<00:11, 57.99it/s]


Loading weights:  11%|█         | 83

### Reading the block-4 results

**Block 4 is the more promising candidate of the two tested so far.** Skipping it causes less velocity prediction error than skipping block 2 in 14 of the 15 matched prompt/stage comparisons. This is a provisional ranking from five prompts, not a decision to remove it.

The model predicts a velocity at each denoising step, which the scheduler uses to update the noisy image representation. Relative L2 error measures the size of the prediction difference divided by the size of the original teacher prediction. For example, 10% means that the difference vector has one tenth of the teacher vector's norm. It does **not** mean that an image loses 10% of its quality. Smaller error is better for this diagnostic.

| Stage | Block 2 mean error | Block 4 mean error | Block 4 worst prompt |
|---|---:|---:|---:|
| Early | 16.41% | 10.81% | 17.80% |
| Middle | 4.15% | 2.69% | 3.44% |
| Final | 18.96% | 5.43% | 6.73% |

The means average five prompts at each stage. Block 4 has lower average error at all three measured stages, especially the final stage. Its early-stage error remains noticeable. The one comparison where it performs worse is the real-world prompt at the early stage: 17.80% for block 4 versus 16.84% for block 2. One prompt per capability does not establish a capability-wide result.

Velocity cosine measures whether the two prediction vectors point in similar directions. Block 4's mean cosine is 0.9943 early, 0.9996 in the middle and 0.9985 at the final stage. Values close to 1 indicate similar direction, but can coexist with substantial relative error because cosine alone does not capture changes in magnitude. “Passed” means the experiment completed and its checks succeeded; it does not mean a quality threshold was met.

**Why this comparison is fair:** both candidates used exactly the same saved teacher inputs and reference velocities. All five block-4 reports confirm teacher-target reuse. The saved intact replay controls match the teacher predictions exactly in all 15 cases. Each bypass constructed its own prefix cache, reflecting the skipped block. Predictions have the expected shapes and contain finite values; the skipped cache entry is empty and all retained entries are populated. A cache-only rerun made no Modal call and changed no saved artifact.

The printed “4/5 teacher PNG matches” refers to the existing teacher images from the block-2 run compared with the older activation pilot. Block 4 generated no teacher images. The previously documented real-world image mismatch remains unresolved, but both block comparisons use the same saved teacher run.

**What this supports:** prioritize block 4 over block 2 for further candidate assessment. Hidden-state similarity originally ranked block 2 first, whereas this output intervention favors block 4; a small local update can still have a larger downstream effect.

Identity simply passes the input through unchanged. These results measure the consequence of skipping the block. A learned bridge would try to reproduce its missing update, for example with `h + W h`; no bridge has been fitted here. Ease of learning that update can change the ranking. Full denoising trajectories, finished-image quality, other candidate blocks and interactions between replacements remain untested, so neither block is approved for removal.

Reports and predictions are saved in `data/phase3/bypass/block_04_pilot/`; the matched comparison is `comparison_with_block_02.json`. The shared teacher tensors remain in `data/phase3/bypass/teacher_targets/` under their frozen target key.


### Step 11 — Test block 5 against the two measured candidates

We skip block 5 temporarily using identity and compare its velocity predictions with the intact teacher at the same early, middle and final inputs for five pilot prompts. The saved teacher inputs, reference predictions and replay controls are reused without regenerating them. The bypass builds its own prefix cache, so downstream conditioning reflects the skipped block.

We save the predictions, prefix caches and reports locally and on Modal, plus matched comparisons with blocks 2 and 4 locally. Every completed rerun reads the saved results after checking their identities and checksums. Relative prediction error helps rank candidates for further assessment; image quality and learned bridge performance remain separate questions. This step fits no bridge and changes no model weights.


In [1]:
# Paired velocity comparisons for one block; persist shared teacher inputs and all predictions.
import base64
import hashlib
import json
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
shortlist_bytes = (PHASE3_DIR / "bypass_shortlist.json").read_bytes()
shortlist = json.loads(shortlist_bytes)
full_manifest = json.loads((PHASE3_DIR / "activations/full/request.json").read_text(encoding="utf-8"))
pilot_manifest = json.loads((PHASE3_DIR / "activations/pilot/request.json").read_text(encoding="utf-8"))
model = full_manifest["request"]
if shortlist["request"]["activation_request_key"] != full_manifest["request_key"]:
    raise RuntimeError("Shortlist and activation run identify different measurements.")
layer = shortlist["candidate_layers"][2]
rows = pilot_manifest["request"]["pilot_rows"]
if layer != 5 or len(rows) != 5:
    raise RuntimeError("This third bypass pilot expects block 5 and the five validated pilot prompts.")
source_reports = {}
for row in rows:
    path = PHASE3_DIR / "activations/pilot" / f"{row['eval_id']}.json"
    source_reports[row["eval_id"]] = json.loads(path.read_text(encoding="utf-8"))
    if hashlib.sha256(path.read_bytes()).hexdigest() != model["pilot_artifacts"][row["eval_id"]]["report_sha256"]:
        raise RuntimeError("A pilot activation report changed after the full pass.")

target_request = {
    "version": "phase3-paired-teacher-targets-v1",
    **{key: model[key] for key in ("model_repo", "model_revision", "config_sha256", "dtype",
                                  "noise_device", "num_layers", "runtime", "scheduler_class", "scheduler_config")},
    "rows": rows, "probe_sha256": model["probe_sha256"],
    "stages": ["early", "middle", "final"], "step_rule": "0, (n - 1) // 2, n - 1",
    "transformer_source_sha256": next(iter(source_reports.values()))["runtime"]["transformer_source_sha256"],
    "teacher_image_sha256": {row["eval_id"]: source_reports[row["eval_id"]]["image_sha256"] for row in rows},
    "cache_policy": "save teacher prefix KV tensors; rebuild a separate bypass prefix cache at the early step",
    "control": "replay intact teacher on captured inputs before bypassing; save control predictions and errors",
}
target_key = hashlib.sha256(json.dumps(target_request, sort_keys=True).encode()).hexdigest()

baseline_dir = PHASE3_DIR / "bypass/block_02_pilot"
baseline_manifest = json.loads((baseline_dir / "request.json").read_text(encoding="utf-8"))
baseline_summary_bytes = (baseline_dir / "summary.json").read_bytes()
baseline_summary = json.loads(baseline_summary_bytes)
if baseline_manifest["request"]["target_key"] != target_key:
    raise RuntimeError("Block 5 must use precisely the same teacher inputs as block 2.")
baseline_reports = {}
for row in rows:
    path = baseline_dir / f"{row['eval_id']}.json"
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != baseline_summary["report_hashes"][row["eval_id"]]:
        raise RuntimeError("Block-2 comparison source changed.")
    report = json.loads(raw)
    if report["status"] != "passed" or report["request_key"] != baseline_manifest["request_key"]:
        raise RuntimeError("Block-2 comparison source is invalid.")
    baseline_reports[row["eval_id"]] = report
    for extension, digest in [("pt", report["targets_sha256"]), ("png", report["teacher_image_sha256"])]:
        path = PHASE3_DIR / "bypass/teacher_targets" / target_key / f"{row['eval_id']}.{extension}"
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError("Shared local teacher cache is missing or changed; no regeneration allowed.")


block4_dir = PHASE3_DIR / "bypass/block_04_pilot"
block4_manifest = json.loads((block4_dir / "request.json").read_text(encoding="utf-8"))
block4_summary_bytes = (block4_dir / "summary.json").read_bytes()
block4_summary = json.loads(block4_summary_bytes)
if block4_manifest["request"]["target_key"] != target_key or block4_summary["request_key"] != block4_manifest["request_key"]:
    raise RuntimeError("Block-4 comparison source identifies different inputs.")
block4_reports = {}
for row in rows:
    path = block4_dir / f"{row['eval_id']}.json"
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != block4_summary["report_hashes"][row["eval_id"]]:
        raise RuntimeError("Block-4 comparison source changed.")
    report = json.loads(raw)
    if report["status"] != "passed" or report["request_key"] != block4_manifest["request_key"]:
        raise RuntimeError("Block-4 comparison source is invalid.")
    block4_reports[row["eval_id"]] = report

request = {"version": "phase3-block-bypass-pilot-v1", "require_cached_teacher": True, "target_request": target_request,
           "target_key": target_key, "bypassed_layer": layer,
           "shortlist_sha256": hashlib.sha256(shortlist_bytes).hexdigest(),
           "metric_definition": "velocity relative L2 error, RMSE and flattened cosine in FP32; epsilon=1e-12"}
request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = PHASE3_DIR / f"bypass/block_{layer:02d}_pilot"
targets_dir = PHASE3_DIR / "bypass/teacher_targets" / target_key
run_dir.mkdir(parents=True, exist_ok=True)
targets_dir.mkdir(parents=True, exist_ok=True)
manifest_path = run_dir / "request.json"
manifest = {"request": request, "request_key": request_key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Saved bypass pilot uses different inputs; preserve and inspect it.")
else:
    with manifest_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(manifest, indent=2) + "\n")

def validate_report(report, row):
    if report.get("request_key") != request_key or report.get("row") != row:
        raise RuntimeError("Bypass report belongs to different inputs.")
    if report.get("teacher_targets_reused") is not True:
        raise RuntimeError("Block 5 unexpectedly regenerated teacher targets.")
    if report["status"] != "passed":
        raise RuntimeError("Saved bypass run failed; no automatic retry: " + report["error"])
    if [m["stage"] for m in report["measurements"]] != target_request["stages"]:
        raise RuntimeError("Bypass report does not cover all three denoising stages.")

def verify_files(report, row):
    paths = [(run_dir / f"{row['eval_id']}.pt", report["outputs_sha256"]),
             (targets_dir / f"{row['eval_id']}.pt", report["targets_sha256"]),
             (targets_dir / f"{row['eval_id']}.png", report["teacher_image_sha256"])]
    for path, digest in paths:
        if not path.is_file():
            return False
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Saved bypass artifact checksum differs: {path}")
    return True

reports = {}
missing_rows = []
for row in rows:
    path = run_dir / f"{row['eval_id']}.json"
    if path.exists():
        report = json.loads(path.read_text(encoding="utf-8"))
        validate_report(report, row)
        if verify_files(report, row):
            reports[row["eval_id"]] = report
            continue
    missing_rows.append(row)

if missing_rows:
    import modal
    runtime = target_request["runtime"]
    results_volume = modal.Volume.from_name(runtime["results_volume"])
    remote_run = f"/bypass/runs/{request_key}"
    remote_targets = f"/bypass/targets/{target_key}"

    def receive_report(report, row):
        path = run_dir / f"{row['eval_id']}.json"
        if report.get("request_key") != request_key or report.get("row") != row:
            raise RuntimeError("Remote bypass report belongs to different inputs.")
        if report["status"] == "passed":
            files = [(f"{remote_run}/{row['eval_id']}.pt", run_dir / f"{row['eval_id']}.pt", report["outputs_sha256"]),
                     (f"{remote_targets}/{row['eval_id']}.pt", targets_dir / f"{row['eval_id']}.pt", report["targets_sha256"]),
                     (f"{remote_targets}/{row['eval_id']}.png", targets_dir / f"{row['eval_id']}.png", report["teacher_image_sha256"])]
            for remote, local, digest in files:
                if local.exists():
                    if hashlib.sha256(local.read_bytes()).hexdigest() != digest:
                        raise RuntimeError(f"Existing local artifact differs: {local}")
                    continue
                temporary = local.with_suffix(".tmp")
                with temporary.open("wb") as handle:
                    for chunk in results_volume.read_file(remote):
                        handle.write(chunk)
                if hashlib.sha256(temporary.read_bytes()).hexdigest() != digest:
                    raise RuntimeError("Downloaded bypass artifact checksum differs.")
                temporary.replace(local)
        if path.exists():
            if json.loads(path.read_text(encoding="utf-8")) != report:
                raise RuntimeError("Local bypass report differs from remote backup.")
        else:
            temporary = path.with_suffix(".tmp")
            temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
            temporary.replace(path)
        validate_report(report, row)
        return report

    try:
        remote_names = {Path(entry.path).name for entry in results_volume.listdir(remote_run, recursive=False)}
    except (FileNotFoundError, modal.exception.NotFoundError):
        remote_names = set()
    pending = []
    for row in missing_rows:
        if f"{row['eval_id']}.json" in remote_names:
            report = json.loads(b"".join(results_volume.read_file(f"{remote_run}/{row['eval_id']}.json")))
            reports[row["eval_id"]] = receive_report(report, row)
        else:
            if (run_dir / f"{row['eval_id']}.json").exists() or (run_dir / f"{row['eval_id']}.pt").exists():
                raise RuntimeError("Incomplete local bypass artifacts have no remote result backup.")
            pending.append(row)

    if pending:
        app = modal.App("nihonga-phase3-block5-bypass")
        weights_volume = modal.Volume.from_name(runtime["weights_volume"])
        image = (modal.Image.debian_slim(python_version=runtime["python_version"])
                 .apt_install("git").pip_install(*runtime["packages"])
                 .env({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}))

        @app.function(image=image, gpu=runtime["gpu_requested"], timeout=1800, max_containers=1,
                      serialized=True, volumes={"/root/.cache/huggingface": weights_volume.with_mount_options(read_only=True),
                                                "/phase3": results_volume})
        def paired_bypass(expected, key, jobs):
            import hashlib
            import inspect
            import json
            import math
            import time
            from datetime import datetime, timezone
            from pathlib import Path
            import torch
            from diffusers import QwenImage21Pipeline
            from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

            target_config = expected["target_request"]
            root = Path("/phase3/bypass/runs") / key
            shared = Path("/phase3/bypass/targets") / expected["target_key"]
            root.mkdir(parents=True, exist_ok=True)
            shared.mkdir(parents=True, exist_ok=True)

            def save_json(path, value):
                temporary = path.with_suffix(".tmp")
                temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
                temporary.replace(path)

            def save_tensors(path, value):
                temporary = path.with_suffix(".tmp")
                torch.save(value, temporary)
                temporary.replace(path)

            def cpu(value):
                if isinstance(value, torch.Tensor):
                    return value.detach().cpu().clone()
                if isinstance(value, dict):
                    return {k: cpu(v) for k, v in value.items()}
                if isinstance(value, (list, tuple)):
                    return type(value)(cpu(v) for v in value)
                if value is None or isinstance(value, (str, int, float, bool)):
                    return value
                raise TypeError(f"Unsupported captured value: {type(value).__name__}")

            def gpu(value):
                if isinstance(value, torch.Tensor):
                    return value.to("cuda")
                if isinstance(value, dict):
                    return {k: gpu(v) for k, v in value.items()}
                if isinstance(value, (list, tuple)):
                    return type(value)(gpu(v) for v in value)
                return value

            def cache_tensors(cache):
                return [{"k": cpu(item.k), "v": cpu(item.v)} for item in cache.layer_caches]

            def restore_cache(values):
                cache = QwenImage21KVCache(len(values))
                for item, saved in zip(cache.layer_caches, values):
                    if saved["k"] is not None:
                        item.store(gpu(saved["k"]), gpu(saved["v"]))
                return cache

            def metrics(prediction, reference):
                x, y = prediction.float(), reference.float()
                error = x - y
                norm = torch.linalg.vector_norm(y).clamp_min(1e-12)
                return {"relative_l2_error": float(torch.linalg.vector_norm(error) / norm),
                        "rmse": float(error.square().mean().sqrt()),
                        "cosine": float((x * y).sum() / (torch.linalg.vector_norm(x) * norm).clamp_min(1e-12)),
                        "teacher_rms": float(y.square().mean().sqrt())}

            save_json(root / "request.json", {"request": expected, "request_key": key})
            shared_manifest = {"request": target_config, "request_key": expected["target_key"]}
            if not (shared / "request.json").is_file() or json.loads((shared / "request.json").read_text()) != shared_manifest:
                raise RuntimeError("Shared teacher manifest is missing or changed; regeneration is disabled.")
            results_volume.commit()
            pipe = None
            load_error = None
            try:
                if "H100" not in torch.cuda.get_device_name(0):
                    raise RuntimeError("Allocated GPU differs from requested H100.")
                pipe = QwenImage21Pipeline.from_pretrained(target_config["model_repo"],
                    revision=target_config["model_revision"], dtype=getattr(torch, target_config["dtype"]),
                    local_files_only=True).to("cuda")
                pipe.set_progress_bar_config(disable=True)
                pipe.transformer.eval()
                source = Path(inspect.getfile(type(pipe.transformer)))
                if hashlib.sha256(source.read_bytes()).hexdigest() != target_config["transformer_source_sha256"]:
                    raise RuntimeError("Transformer implementation differs from the activation pilot.")
                if (len(pipe.transformer.transformer_blocks) != target_config["num_layers"]
                        or not pipe.transformer.config.causal_condition
                        or type(pipe.scheduler).__name__ != target_config["scheduler_class"]
                        or json.loads(json.dumps(dict(pipe.scheduler.config), default=str)) != target_config["scheduler_config"]):
                    raise RuntimeError("Loaded architecture or scheduler differs from the saved baseline.")
            except Exception as error:
                load_error = f"{type(error).__name__}: {error}"

            for row in jobs:
                report_path = root / f"{row['eval_id']}.json"
                if report_path.exists():
                    yield json.loads(report_path.read_text(encoding="utf-8"))
                    continue
                report = {"request_key": key, "row": row, "bypassed_layer": expected["bypassed_layer"],
                          "recorded_utc": datetime.now(timezone.utc).isoformat(), "status": "failed"}
                hooks = []
                block = None
                original_forward = None
                had_own_forward = False
                try:
                    if load_error:
                        raise RuntimeError(load_error)
                    target_path = shared / f"{row['eval_id']}.pt"
                    target_meta_path = shared / f"{row['eval_id']}.json"
                    teacher_png = shared / f"{row['eval_id']}.png"
                    if target_meta_path.exists():
                        target_meta = json.loads(target_meta_path.read_text(encoding="utf-8"))
                        if (target_meta["target_key"] != expected["target_key"] or target_meta["row"] != row
                                or hashlib.sha256(target_path.read_bytes()).hexdigest() != target_meta["targets_sha256"]
                                or hashlib.sha256(teacher_png.read_bytes()).hexdigest() != target_meta["image_sha256"]):
                            raise RuntimeError("Shared teacher artifacts differ from their saved identity.")
                        bundle = torch.load(target_path, map_location="cpu", weights_only=True)
                        report["teacher_targets_reused"] = True
                    else:
                        raise RuntimeError("Saved teacher targets are required; teacher regeneration is disabled.")

                    block = pipe.transformer.transformer_blocks[expected["bypassed_layer"]]
                    had_own_forward = "forward" in block.__dict__
                    original_forward = block.forward
                    bypass_calls = {"count": 0}
                    def identity_forward(*args, **kwargs):
                        bypass_calls["count"] += 1
                        return kwargs["hidden_states"] if "hidden_states" in kwargs else args[0]
                    block.forward = identity_forward
                    student_cache = QwenImage21KVCache(target_config["num_layers"])
                    predictions = []
                    measurements = []
                    with torch.inference_mode(), pipe.transformer.cache_context("cond"):
                        for item, reference, control in zip(bundle["inputs"], bundle["teacher_velocities"], bundle["control_metrics"]):
                            kwargs = gpu(item["kwargs"])
                            kwargs["kv_cache"] = student_cache
                            prediction = pipe.transformer(**kwargs)[0][:, -item["target_tokens"]:]
                            predictions.append(cpu(prediction))
                            measured = metrics(prediction, gpu(reference))
                            if not all(math.isfinite(value) for value in measured.values()):
                                raise RuntimeError("Bypass metrics contain nonfinite values.")
                            measurements.append({"stage": item["stage"], "step_index": item["step_index"],
                                "sigma": item["sigma"], "target_tokens": item["target_tokens"],
                                "bypass": measured, "intact_replay_control": control})
                    if bypass_calls["count"] != 3 or student_cache.get_layer(expected["bypassed_layer"]).k is not None:
                        raise RuntimeError("The selected block was not consistently bypassed.")
                    outputs_path = root / f"{row['eval_id']}.pt"
                    save_tensors(outputs_path, {"row": row, "request_key": key, "bypassed_layer": expected["bypassed_layer"],
                        "predictions": predictions, "bypass_prefix_cache": cache_tensors(student_cache)})
                    report.update({"status": "passed", "measurements": measurements,
                        "targets_sha256": target_meta["targets_sha256"], "teacher_image_sha256": target_meta["image_sha256"],
                        "outputs_sha256": hashlib.sha256(outputs_path.read_bytes()).hexdigest(),
                        "teacher_png_matches_activation": target_meta["image_sha256"] == target_config["teacher_image_sha256"][row["eval_id"]]})
                except Exception as error:
                    report["error"] = f"{type(error).__name__}: {error}"
                finally:
                    for hook in hooks:
                        hook.remove()
                    if block is not None and original_forward is not None:
                        if had_own_forward:
                            block.forward = original_forward
                        else:
                            del block.forward
                save_json(report_path, report)
                results_volume.commit()
                print(f"Block {expected['bypassed_layer']} bypass: {row['dimension']} ({report['status']}).")
                yield report
                if report["status"] != "passed":
                    break

        with modal.enable_output(), app.run():
            by_id = {row["eval_id"]: row for row in rows}
            for report in paired_bypass.remote_gen(request, request_key, pending):
                row = by_id[report["row"]["eval_id"]]
                reports[row["eval_id"]] = receive_report(report, row)
                print(f"Saved paired inputs, predictions and report: {len(reports)}/5 prompts.")
else:
    print("Reusing all five verified local bypass results; no Modal call.")

if len(reports) != 5:
    raise RuntimeError("The bypass pilot is incomplete; completed prompt artifacts remain saved.")
summary_path = run_dir / "summary.json"
report_hashes = {row["eval_id"]: hashlib.sha256((run_dir / f"{row['eval_id']}.json").read_bytes()).hexdigest() for row in rows}
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary["request_key"] != request_key or summary["report_hashes"] != report_hashes:
        raise RuntimeError("Saved bypass summary differs from its source reports.")
else:
    from statistics import fmean
    stages = []
    for stage in target_request["stages"]:
        values = [m for report in reports.values() for m in report["measurements"] if m["stage"] == stage]
        stages.append({"stage": stage, "n_prompts": len(values),
            "mean_relative_l2_error": fmean(m["bypass"]["relative_l2_error"] for m in values),
            "max_relative_l2_error": max(m["bypass"]["relative_l2_error"] for m in values),
            "mean_velocity_cosine": fmean(m["bypass"]["cosine"] for m in values),
            "max_intact_replay_relative_error": max(m["intact_replay_control"]["relative_l2_error"] for m in values)})
    summary = {"request_key": request_key, "bypassed_layer": layer, "report_hashes": report_hashes,
               "n_prompts": 5, "n_comparisons": 15, "stages": stages,
               "teacher_png_matches_activation_count": sum(r["teacher_png_matches_activation"] for r in reports.values()),
               "scope": "paired velocity pilot on teacher trajectories; no bypassed full-trajectory image evaluation"}
    temporary = summary_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    temporary.replace(summary_path)
print(f"Block {layer} bypass pilot: 5 prompts, 15 paired velocity comparisons.")
for stage in summary["stages"]:
    print(f"  {stage['stage']:6s}: mean error={100*stage['mean_relative_l2_error']:.3f}%, "
          f"max error={100*stage['max_relative_l2_error']:.3f}%, cosine={stage['mean_velocity_cosine']:.6f}, "
          f"intact replay max error={100*stage['max_intact_replay_relative_error']:.6f}%")
print("Teacher PNG matches prior activation PNG:", summary["teacher_png_matches_activation_count"], "/5")
print(f"Saved reports and bypass tensors: {run_dir}")
print(f"Saved reusable teacher inputs, prefix caches, control predictions and velocities: {targets_dir}")
print("These percentages measure velocity error, not image-quality loss; the removal decision remains pending.")

comparison_path = run_dir / "comparison_with_block_02.json"
comparison_identity = {"target_key": target_key, "block_02_summary_sha256": hashlib.sha256(baseline_summary_bytes).hexdigest(),
                       "block_05_report_hashes": report_hashes, "block_05_request_key": request_key}
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    if comparison["identity"] != comparison_identity:
        raise RuntimeError("Saved comparison differs from its source results.")
else:
    paired = []
    for row in rows:
        a, b = baseline_reports[row["eval_id"]], reports[row["eval_id"]]
        if a["targets_sha256"] != b["targets_sha256"]:
            raise RuntimeError("The two candidates used different teacher targets.")
        for first, second in zip(a["measurements"], b["measurements"]):
            if (first["stage"] != second["stage"] or first["step_index"] != second["step_index"]
                    or first["intact_replay_control"] != second["intact_replay_control"]):
                raise RuntimeError("Candidate comparisons differ in steps or controls.")
            paired.append({"eval_id": row["eval_id"], "dimension": row["dimension"], "stage": first["stage"],
                           "block_02_error": first["bypass"]["relative_l2_error"],
                           "block_05_error": second["bypass"]["relative_l2_error"]})
    comparison = {"identity": comparison_identity, "paired": paired,
                  "block_05_lower_error_count": sum(v["block_05_error"] < v["block_02_error"] for v in paired),
                  "n_comparisons": len(paired), "stages": [
                      {"stage": a["stage"], "block_02_mean_error": a["mean_relative_l2_error"],
                       "block_05_mean_error": b["mean_relative_l2_error"],
                       "block_02_max_error": a["max_relative_l2_error"], "block_05_max_error": b["max_relative_l2_error"]}
                      for a, b in zip(baseline_summary["stages"], summary["stages"])]}
    with comparison_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(comparison, indent=2) + "\n")
print("Block 5 has lower error than block 2 in", comparison["block_05_lower_error_count"], "/15 paired comparisons.")
for stage in comparison["stages"]:
    print(f"  {stage['stage']}: block 2={100*stage['block_02_mean_error']:.3f}%, block 5={100*stage['block_05_mean_error']:.3f}%")


comparison_path = run_dir / "comparison_with_block_04.json"
comparison_identity = {"target_key": target_key, "block_04_summary_sha256": hashlib.sha256(block4_summary_bytes).hexdigest(),
                       "block_05_report_hashes": report_hashes, "block_05_request_key": request_key}
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    if comparison["identity"] != comparison_identity:
        raise RuntimeError("Saved comparison differs from its source results.")
else:
    paired = []
    for row in rows:
        a, b = block4_reports[row["eval_id"]], reports[row["eval_id"]]
        if a["targets_sha256"] != b["targets_sha256"]:
            raise RuntimeError("The two candidates used different teacher targets.")
        for first, second in zip(a["measurements"], b["measurements"]):
            if (first["stage"] != second["stage"] or first["step_index"] != second["step_index"]
                    or first["intact_replay_control"] != second["intact_replay_control"]):
                raise RuntimeError("Candidate comparisons differ in steps or controls.")
            paired.append({"eval_id": row["eval_id"], "dimension": row["dimension"], "stage": first["stage"],
                           "block_04_error": first["bypass"]["relative_l2_error"],
                           "block_05_error": second["bypass"]["relative_l2_error"]})
    comparison = {"identity": comparison_identity, "paired": paired,
                  "block_05_lower_error_count": sum(v["block_05_error"] < v["block_04_error"] for v in paired),
                  "n_comparisons": len(paired), "stages": [
                      {"stage": a["stage"], "block_04_mean_error": a["mean_relative_l2_error"],
                       "block_05_mean_error": b["mean_relative_l2_error"],
                       "block_04_max_error": a["max_relative_l2_error"], "block_05_max_error": b["max_relative_l2_error"]}
                      for a, b in zip(block4_summary["stages"], summary["stages"])]}
    with comparison_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(comparison, indent=2) + "\n")
print("Block 5 has lower error than block 4 in", comparison["block_05_lower_error_count"], "/15 paired comparisons.")
for stage in comparison["stages"]:
    print(f"  {stage['stage']}: block 4={100*stage['block_04_mean_error']:.3f}%, block 5={100*stage['block_05_mean_error']:.3f}%")


✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-N923t4nAYBFwblr6g57Nfs
✓ Created objects.
└── 🔨 Created function paired_bypass.

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Loading checkpoint shards:  50%|█████     | 1/2 [00:01<00:01,  1.53s/it]


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.07s/it]

Loading pipeline components...:  80%|████████  | 4/5 [00:03<00:00,  1.28it/s]


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]


Loading weights:   2%|▏         | 12/750 [00:00<00:06, 115.86it/s]


Loading weights:   3%|▎         | 24/750 [00:00<00:08, 89.57it/s] 


Loading weights:   5%|▌         | 38/750 [00:00<00:06, 101.98it/s]


Loading weights:   7%|▋         | 56/750 [00:00<00:05, 121.37it/s]


Loading weights:  10%|█         | 76/750 [00:00<00:04, 139.45it/s]


Loading weights:  12%|█▏        | 92/750 [00:00<00:04, 138.95it/s]


Loadi

### Reading the block-5 results

The table compares the average velocity prediction error for each candidate on the same five prompts and saved teacher inputs. Lower is better for this diagnostic.

| Stage | Block 2 mean error | Block 4 mean error | Block 5 mean error | Block 5 worst prompt |
|---|---:|---:|---:|---:|
| Early | 16.41% | 10.81% | 12.74% | 17.00% |
| Middle | 4.15% | 2.69% | 2.81% | 3.61% |
| Final | 18.96% | 5.43% | 3.69% | 4.53% |

The model's velocity prediction determines how the scheduler updates the noisy image representation. Relative L2 error is the norm of the prediction difference divided by the norm of the teacher prediction. A value of 10% means the difference vector is one tenth the teacher vector's size; it is not a percentage of image-quality loss. The worst-prompt column shows the largest observed error at that stage, so the average does not conceal a difficult case.

Block 5 has lower error than block 2 in **15/15** matched prompt/stage comparisons.

Block 5 has lower error than block 4 in **8/15** matched prompt/stage comparisons.

It has higher error for: aesthetics at early (9.37% versus 4.75%); alignment at early (10.41% versus 9.40%); alignment at middle (3.61% versus 3.03%); creative at early (16.00% versus 15.03%); creative at middle (2.55% versus 2.28%); real_world at middle (2.29% versus 2.00%); visual_quality at early (17.00% versus 7.07%).

The lowest mean error comes from block 4 at the early stage, block 4 at the middle stage, block 5 at the final stage. Stage-dependent differences matter; a candidate should not be chosen from one averaged number alone.

**What this supports:** blocks 4 and 5 are both more promising than block 2 under this pilot diagnostic. Block 4 has lower mean error early and in the middle; block 5 has lower mean error at the final stage. Neither is the universal winner. Their individual bypass results also do not establish that skipping both together would be acceptable.

The printed cosine measures directional similarity between velocity vectors: a value near 1 means they point in similar directions. Block 5's means are 0.9917 early, 0.9996 in the middle and 0.9993 at the final stage. Cosine alone does not measure changes in magnitude, which is why the relative error also matters.

**What the checks establish:** all five reports passed and confirm reuse of the teacher targets. All 15 saved intact replay predictions match their teacher references exactly. The bypass predictions have matching shapes and finite values. Block 5's prefix-cache entry is empty, and all retained blocks have populated entries. A verified rerun used local caches without model imports, Modal calls, summary recomputation or artifact changes. “Passed” describes successful execution and checks, not acceptance of the candidate's image quality.

Identity passes the input through unchanged, so this experiment tests the consequence of omitting block 5. A learned bridge would approximate the missing update; fitting such a bridge may change the candidate ranking. These predictions are measured on teacher trajectories at three sampled steps, without allowing a bypassed model to evolve its own noisy inputs. Error accumulated over a complete generation, finished-image quality, speed savings and interactions between multiple replacements remain unknown. Five prompts provide pilot evidence, not a reliable estimate for an entire capability.

The “4/5 teacher PNG matches” line still refers to the shared teacher images generated in the block-2 run compared with the older activation pilot. No new teacher images were generated for block 5. Both paired comparisons use the same saved teacher references despite the previously documented real-world image mismatch.

Predictions, caches and reports are saved in `data/phase3/bypass/block_05_pilot/`. The files `comparison_with_block_02.json` and `comparison_with_block_04.json` preserve all 15 matched errors and the source identities. No candidate is approved for removal from these measurements alone.


### Step 12 — Test block 3, the remaining initial candidate

We temporarily skip block 3 using identity and measure velocity errors at the early, middle and final denoising stages for the same five pilot prompts. Saved teacher inputs, reference velocities and replay controls are reused; regenerating teacher targets is disabled. The bypass builds its own prefix cache so conditioning reflects the skipped block.

Predictions, caches and per-prompt reports are saved locally and on Modal. Matched comparisons with blocks 2, 4 and 5 are saved locally with their source identities. Completed reruns read cached results after checksum validation, without another GPU call or summary calculation. This diagnostic completes the initial four-candidate pilot batch. It does not fit a bridge, evaluate bypassed full-generation images or approve any removal.


In [1]:
# Paired velocity comparisons for one block; persist shared teacher inputs and all predictions.
import base64
import hashlib
import json
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
shortlist_bytes = (PHASE3_DIR / "bypass_shortlist.json").read_bytes()
shortlist = json.loads(shortlist_bytes)
full_manifest = json.loads((PHASE3_DIR / "activations/full/request.json").read_text(encoding="utf-8"))
pilot_manifest = json.loads((PHASE3_DIR / "activations/pilot/request.json").read_text(encoding="utf-8"))
model = full_manifest["request"]
if shortlist["request"]["activation_request_key"] != full_manifest["request_key"]:
    raise RuntimeError("Shortlist and activation run identify different measurements.")
layer = shortlist["candidate_layers"][3]
rows = pilot_manifest["request"]["pilot_rows"]
if layer != 3 or len(rows) != 5:
    raise RuntimeError("This fourth bypass pilot expects block 3 and the five validated pilot prompts.")
source_reports = {}
for row in rows:
    path = PHASE3_DIR / "activations/pilot" / f"{row['eval_id']}.json"
    source_reports[row["eval_id"]] = json.loads(path.read_text(encoding="utf-8"))
    if hashlib.sha256(path.read_bytes()).hexdigest() != model["pilot_artifacts"][row["eval_id"]]["report_sha256"]:
        raise RuntimeError("A pilot activation report changed after the full pass.")

target_request = {
    "version": "phase3-paired-teacher-targets-v1",
    **{key: model[key] for key in ("model_repo", "model_revision", "config_sha256", "dtype",
                                  "noise_device", "num_layers", "runtime", "scheduler_class", "scheduler_config")},
    "rows": rows, "probe_sha256": model["probe_sha256"],
    "stages": ["early", "middle", "final"], "step_rule": "0, (n - 1) // 2, n - 1",
    "transformer_source_sha256": next(iter(source_reports.values()))["runtime"]["transformer_source_sha256"],
    "teacher_image_sha256": {row["eval_id"]: source_reports[row["eval_id"]]["image_sha256"] for row in rows},
    "cache_policy": "save teacher prefix KV tensors; rebuild a separate bypass prefix cache at the early step",
    "control": "replay intact teacher on captured inputs before bypassing; save control predictions and errors",
}
target_key = hashlib.sha256(json.dumps(target_request, sort_keys=True).encode()).hexdigest()

baseline_dir = PHASE3_DIR / "bypass/block_02_pilot"
baseline_manifest = json.loads((baseline_dir / "request.json").read_text(encoding="utf-8"))
baseline_summary_bytes = (baseline_dir / "summary.json").read_bytes()
baseline_summary = json.loads(baseline_summary_bytes)
if baseline_manifest["request"]["target_key"] != target_key:
    raise RuntimeError("Block 3 must use precisely the same teacher inputs as block 2.")
baseline_reports = {}
for row in rows:
    path = baseline_dir / f"{row['eval_id']}.json"
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != baseline_summary["report_hashes"][row["eval_id"]]:
        raise RuntimeError("Block-2 comparison source changed.")
    report = json.loads(raw)
    if report["status"] != "passed" or report["request_key"] != baseline_manifest["request_key"]:
        raise RuntimeError("Block-2 comparison source is invalid.")
    baseline_reports[row["eval_id"]] = report
    for extension, digest in [("pt", report["targets_sha256"]), ("png", report["teacher_image_sha256"])]:
        path = PHASE3_DIR / "bypass/teacher_targets" / target_key / f"{row['eval_id']}.{extension}"
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError("Shared local teacher cache is missing or changed; no regeneration allowed.")


block4_dir = PHASE3_DIR / "bypass/block_04_pilot"
block4_manifest = json.loads((block4_dir / "request.json").read_text(encoding="utf-8"))
block4_summary_bytes = (block4_dir / "summary.json").read_bytes()
block4_summary = json.loads(block4_summary_bytes)
if block4_manifest["request"]["target_key"] != target_key or block4_summary["request_key"] != block4_manifest["request_key"]:
    raise RuntimeError("Block-4 comparison source identifies different inputs.")
block4_reports = {}
for row in rows:
    path = block4_dir / f"{row['eval_id']}.json"
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != block4_summary["report_hashes"][row["eval_id"]]:
        raise RuntimeError("Block-4 comparison source changed.")
    report = json.loads(raw)
    if report["status"] != "passed" or report["request_key"] != block4_manifest["request_key"]:
        raise RuntimeError("Block-4 comparison source is invalid.")
    block4_reports[row["eval_id"]] = report


block5_dir = PHASE3_DIR / "bypass/block_05_pilot"
block5_manifest = json.loads((block5_dir / "request.json").read_text(encoding="utf-8"))
block5_summary_bytes = (block5_dir / "summary.json").read_bytes()
block5_summary = json.loads(block5_summary_bytes)
if block5_manifest["request"]["target_key"] != target_key or block5_summary["request_key"] != block5_manifest["request_key"]:
    raise RuntimeError("Block-5 comparison source identifies different inputs.")
block5_reports = {}
for row in rows:
    path = block5_dir / f"{row['eval_id']}.json"
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != block5_summary["report_hashes"][row["eval_id"]]:
        raise RuntimeError("Block-5 comparison source changed.")
    report = json.loads(raw)
    if report["status"] != "passed" or report["request_key"] != block5_manifest["request_key"]:
        raise RuntimeError("Block-5 comparison source is invalid.")
    block5_reports[row["eval_id"]] = report

request = {"version": "phase3-block-bypass-pilot-v1", "require_cached_teacher": True, "target_request": target_request,
           "target_key": target_key, "bypassed_layer": layer,
           "shortlist_sha256": hashlib.sha256(shortlist_bytes).hexdigest(),
           "metric_definition": "velocity relative L2 error, RMSE and flattened cosine in FP32; epsilon=1e-12"}
request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = PHASE3_DIR / f"bypass/block_{layer:02d}_pilot"
targets_dir = PHASE3_DIR / "bypass/teacher_targets" / target_key
run_dir.mkdir(parents=True, exist_ok=True)
targets_dir.mkdir(parents=True, exist_ok=True)
manifest_path = run_dir / "request.json"
manifest = {"request": request, "request_key": request_key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Saved bypass pilot uses different inputs; preserve and inspect it.")
else:
    with manifest_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(manifest, indent=2) + "\n")

def validate_report(report, row):
    if report.get("request_key") != request_key or report.get("row") != row:
        raise RuntimeError("Bypass report belongs to different inputs.")
    if report.get("teacher_targets_reused") is not True:
        raise RuntimeError("Block 3 unexpectedly regenerated teacher targets.")
    if report["status"] != "passed":
        raise RuntimeError("Saved bypass run failed; no automatic retry: " + report["error"])
    if [m["stage"] for m in report["measurements"]] != target_request["stages"]:
        raise RuntimeError("Bypass report does not cover all three denoising stages.")

def verify_files(report, row):
    paths = [(run_dir / f"{row['eval_id']}.pt", report["outputs_sha256"]),
             (targets_dir / f"{row['eval_id']}.pt", report["targets_sha256"]),
             (targets_dir / f"{row['eval_id']}.png", report["teacher_image_sha256"])]
    for path, digest in paths:
        if not path.is_file():
            return False
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Saved bypass artifact checksum differs: {path}")
    return True

reports = {}
missing_rows = []
for row in rows:
    path = run_dir / f"{row['eval_id']}.json"
    if path.exists():
        report = json.loads(path.read_text(encoding="utf-8"))
        validate_report(report, row)
        if verify_files(report, row):
            reports[row["eval_id"]] = report
            continue
    missing_rows.append(row)

if missing_rows:
    import modal
    runtime = target_request["runtime"]
    results_volume = modal.Volume.from_name(runtime["results_volume"])
    remote_run = f"/bypass/runs/{request_key}"
    remote_targets = f"/bypass/targets/{target_key}"

    def receive_report(report, row):
        path = run_dir / f"{row['eval_id']}.json"
        if report.get("request_key") != request_key or report.get("row") != row:
            raise RuntimeError("Remote bypass report belongs to different inputs.")
        if report["status"] == "passed":
            files = [(f"{remote_run}/{row['eval_id']}.pt", run_dir / f"{row['eval_id']}.pt", report["outputs_sha256"]),
                     (f"{remote_targets}/{row['eval_id']}.pt", targets_dir / f"{row['eval_id']}.pt", report["targets_sha256"]),
                     (f"{remote_targets}/{row['eval_id']}.png", targets_dir / f"{row['eval_id']}.png", report["teacher_image_sha256"])]
            for remote, local, digest in files:
                if local.exists():
                    if hashlib.sha256(local.read_bytes()).hexdigest() != digest:
                        raise RuntimeError(f"Existing local artifact differs: {local}")
                    continue
                temporary = local.with_suffix(".tmp")
                with temporary.open("wb") as handle:
                    for chunk in results_volume.read_file(remote):
                        handle.write(chunk)
                if hashlib.sha256(temporary.read_bytes()).hexdigest() != digest:
                    raise RuntimeError("Downloaded bypass artifact checksum differs.")
                temporary.replace(local)
        if path.exists():
            if json.loads(path.read_text(encoding="utf-8")) != report:
                raise RuntimeError("Local bypass report differs from remote backup.")
        else:
            temporary = path.with_suffix(".tmp")
            temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
            temporary.replace(path)
        validate_report(report, row)
        return report

    try:
        remote_names = {Path(entry.path).name for entry in results_volume.listdir(remote_run, recursive=False)}
    except (FileNotFoundError, modal.exception.NotFoundError):
        remote_names = set()
    pending = []
    for row in missing_rows:
        if f"{row['eval_id']}.json" in remote_names:
            report = json.loads(b"".join(results_volume.read_file(f"{remote_run}/{row['eval_id']}.json")))
            reports[row["eval_id"]] = receive_report(report, row)
        else:
            if (run_dir / f"{row['eval_id']}.json").exists() or (run_dir / f"{row['eval_id']}.pt").exists():
                raise RuntimeError("Incomplete local bypass artifacts have no remote result backup.")
            pending.append(row)

    if pending:
        app = modal.App("nihonga-phase3-block3-bypass")
        weights_volume = modal.Volume.from_name(runtime["weights_volume"])
        image = (modal.Image.debian_slim(python_version=runtime["python_version"])
                 .apt_install("git").pip_install(*runtime["packages"])
                 .env({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}))

        @app.function(image=image, gpu=runtime["gpu_requested"], timeout=1800, max_containers=1,
                      serialized=True, volumes={"/root/.cache/huggingface": weights_volume.with_mount_options(read_only=True),
                                                "/phase3": results_volume})
        def paired_bypass(expected, key, jobs):
            import hashlib
            import inspect
            import json
            import math
            import time
            from datetime import datetime, timezone
            from pathlib import Path
            import torch
            from diffusers import QwenImage21Pipeline
            from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

            target_config = expected["target_request"]
            root = Path("/phase3/bypass/runs") / key
            shared = Path("/phase3/bypass/targets") / expected["target_key"]
            root.mkdir(parents=True, exist_ok=True)
            shared.mkdir(parents=True, exist_ok=True)

            def save_json(path, value):
                temporary = path.with_suffix(".tmp")
                temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
                temporary.replace(path)

            def save_tensors(path, value):
                temporary = path.with_suffix(".tmp")
                torch.save(value, temporary)
                temporary.replace(path)

            def cpu(value):
                if isinstance(value, torch.Tensor):
                    return value.detach().cpu().clone()
                if isinstance(value, dict):
                    return {k: cpu(v) for k, v in value.items()}
                if isinstance(value, (list, tuple)):
                    return type(value)(cpu(v) for v in value)
                if value is None or isinstance(value, (str, int, float, bool)):
                    return value
                raise TypeError(f"Unsupported captured value: {type(value).__name__}")

            def gpu(value):
                if isinstance(value, torch.Tensor):
                    return value.to("cuda")
                if isinstance(value, dict):
                    return {k: gpu(v) for k, v in value.items()}
                if isinstance(value, (list, tuple)):
                    return type(value)(gpu(v) for v in value)
                return value

            def cache_tensors(cache):
                return [{"k": cpu(item.k), "v": cpu(item.v)} for item in cache.layer_caches]

            def restore_cache(values):
                cache = QwenImage21KVCache(len(values))
                for item, saved in zip(cache.layer_caches, values):
                    if saved["k"] is not None:
                        item.store(gpu(saved["k"]), gpu(saved["v"]))
                return cache

            def metrics(prediction, reference):
                x, y = prediction.float(), reference.float()
                error = x - y
                norm = torch.linalg.vector_norm(y).clamp_min(1e-12)
                return {"relative_l2_error": float(torch.linalg.vector_norm(error) / norm),
                        "rmse": float(error.square().mean().sqrt()),
                        "cosine": float((x * y).sum() / (torch.linalg.vector_norm(x) * norm).clamp_min(1e-12)),
                        "teacher_rms": float(y.square().mean().sqrt())}

            save_json(root / "request.json", {"request": expected, "request_key": key})
            shared_manifest = {"request": target_config, "request_key": expected["target_key"]}
            if not (shared / "request.json").is_file() or json.loads((shared / "request.json").read_text()) != shared_manifest:
                raise RuntimeError("Shared teacher manifest is missing or changed; regeneration is disabled.")
            results_volume.commit()
            pipe = None
            load_error = None
            try:
                if "H100" not in torch.cuda.get_device_name(0):
                    raise RuntimeError("Allocated GPU differs from requested H100.")
                pipe = QwenImage21Pipeline.from_pretrained(target_config["model_repo"],
                    revision=target_config["model_revision"], dtype=getattr(torch, target_config["dtype"]),
                    local_files_only=True).to("cuda")
                pipe.set_progress_bar_config(disable=True)
                pipe.transformer.eval()
                source = Path(inspect.getfile(type(pipe.transformer)))
                if hashlib.sha256(source.read_bytes()).hexdigest() != target_config["transformer_source_sha256"]:
                    raise RuntimeError("Transformer implementation differs from the activation pilot.")
                if (len(pipe.transformer.transformer_blocks) != target_config["num_layers"]
                        or not pipe.transformer.config.causal_condition
                        or type(pipe.scheduler).__name__ != target_config["scheduler_class"]
                        or json.loads(json.dumps(dict(pipe.scheduler.config), default=str)) != target_config["scheduler_config"]):
                    raise RuntimeError("Loaded architecture or scheduler differs from the saved baseline.")
            except Exception as error:
                load_error = f"{type(error).__name__}: {error}"

            for row in jobs:
                report_path = root / f"{row['eval_id']}.json"
                if report_path.exists():
                    yield json.loads(report_path.read_text(encoding="utf-8"))
                    continue
                report = {"request_key": key, "row": row, "bypassed_layer": expected["bypassed_layer"],
                          "recorded_utc": datetime.now(timezone.utc).isoformat(), "status": "failed"}
                hooks = []
                block = None
                original_forward = None
                had_own_forward = False
                try:
                    if load_error:
                        raise RuntimeError(load_error)
                    target_path = shared / f"{row['eval_id']}.pt"
                    target_meta_path = shared / f"{row['eval_id']}.json"
                    teacher_png = shared / f"{row['eval_id']}.png"
                    if target_meta_path.exists():
                        target_meta = json.loads(target_meta_path.read_text(encoding="utf-8"))
                        if (target_meta["target_key"] != expected["target_key"] or target_meta["row"] != row
                                or hashlib.sha256(target_path.read_bytes()).hexdigest() != target_meta["targets_sha256"]
                                or hashlib.sha256(teacher_png.read_bytes()).hexdigest() != target_meta["image_sha256"]):
                            raise RuntimeError("Shared teacher artifacts differ from their saved identity.")
                        bundle = torch.load(target_path, map_location="cpu", weights_only=True)
                        report["teacher_targets_reused"] = True
                    else:
                        raise RuntimeError("Saved teacher targets are required; teacher regeneration is disabled.")

                    block = pipe.transformer.transformer_blocks[expected["bypassed_layer"]]
                    had_own_forward = "forward" in block.__dict__
                    original_forward = block.forward
                    bypass_calls = {"count": 0}
                    def identity_forward(*args, **kwargs):
                        bypass_calls["count"] += 1
                        return kwargs["hidden_states"] if "hidden_states" in kwargs else args[0]
                    block.forward = identity_forward
                    student_cache = QwenImage21KVCache(target_config["num_layers"])
                    predictions = []
                    measurements = []
                    with torch.inference_mode(), pipe.transformer.cache_context("cond"):
                        for item, reference, control in zip(bundle["inputs"], bundle["teacher_velocities"], bundle["control_metrics"]):
                            kwargs = gpu(item["kwargs"])
                            kwargs["kv_cache"] = student_cache
                            prediction = pipe.transformer(**kwargs)[0][:, -item["target_tokens"]:]
                            predictions.append(cpu(prediction))
                            measured = metrics(prediction, gpu(reference))
                            if not all(math.isfinite(value) for value in measured.values()):
                                raise RuntimeError("Bypass metrics contain nonfinite values.")
                            measurements.append({"stage": item["stage"], "step_index": item["step_index"],
                                "sigma": item["sigma"], "target_tokens": item["target_tokens"],
                                "bypass": measured, "intact_replay_control": control})
                    if bypass_calls["count"] != 3 or student_cache.get_layer(expected["bypassed_layer"]).k is not None:
                        raise RuntimeError("The selected block was not consistently bypassed.")
                    outputs_path = root / f"{row['eval_id']}.pt"
                    save_tensors(outputs_path, {"row": row, "request_key": key, "bypassed_layer": expected["bypassed_layer"],
                        "predictions": predictions, "bypass_prefix_cache": cache_tensors(student_cache)})
                    report.update({"status": "passed", "measurements": measurements,
                        "targets_sha256": target_meta["targets_sha256"], "teacher_image_sha256": target_meta["image_sha256"],
                        "outputs_sha256": hashlib.sha256(outputs_path.read_bytes()).hexdigest(),
                        "teacher_png_matches_activation": target_meta["image_sha256"] == target_config["teacher_image_sha256"][row["eval_id"]]})
                except Exception as error:
                    report["error"] = f"{type(error).__name__}: {error}"
                finally:
                    for hook in hooks:
                        hook.remove()
                    if block is not None and original_forward is not None:
                        if had_own_forward:
                            block.forward = original_forward
                        else:
                            del block.forward
                save_json(report_path, report)
                results_volume.commit()
                print(f"Block {expected['bypassed_layer']} bypass: {row['dimension']} ({report['status']}).")
                yield report
                if report["status"] != "passed":
                    break

        with modal.enable_output(), app.run():
            by_id = {row["eval_id"]: row for row in rows}
            for report in paired_bypass.remote_gen(request, request_key, pending):
                row = by_id[report["row"]["eval_id"]]
                reports[row["eval_id"]] = receive_report(report, row)
                print(f"Saved paired inputs, predictions and report: {len(reports)}/5 prompts.")
else:
    print("Reusing all five verified local bypass results; no Modal call.")

if len(reports) != 5:
    raise RuntimeError("The bypass pilot is incomplete; completed prompt artifacts remain saved.")
summary_path = run_dir / "summary.json"
report_hashes = {row["eval_id"]: hashlib.sha256((run_dir / f"{row['eval_id']}.json").read_bytes()).hexdigest() for row in rows}
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary["request_key"] != request_key or summary["report_hashes"] != report_hashes:
        raise RuntimeError("Saved bypass summary differs from its source reports.")
else:
    from statistics import fmean
    stages = []
    for stage in target_request["stages"]:
        values = [m for report in reports.values() for m in report["measurements"] if m["stage"] == stage]
        stages.append({"stage": stage, "n_prompts": len(values),
            "mean_relative_l2_error": fmean(m["bypass"]["relative_l2_error"] for m in values),
            "max_relative_l2_error": max(m["bypass"]["relative_l2_error"] for m in values),
            "mean_velocity_cosine": fmean(m["bypass"]["cosine"] for m in values),
            "max_intact_replay_relative_error": max(m["intact_replay_control"]["relative_l2_error"] for m in values)})
    summary = {"request_key": request_key, "bypassed_layer": layer, "report_hashes": report_hashes,
               "n_prompts": 5, "n_comparisons": 15, "stages": stages,
               "teacher_png_matches_activation_count": sum(r["teacher_png_matches_activation"] for r in reports.values()),
               "scope": "paired velocity pilot on teacher trajectories; no bypassed full-trajectory image evaluation"}
    temporary = summary_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    temporary.replace(summary_path)
print(f"Block {layer} bypass pilot: 5 prompts, 15 paired velocity comparisons.")
for stage in summary["stages"]:
    print(f"  {stage['stage']:6s}: mean error={100*stage['mean_relative_l2_error']:.3f}%, "
          f"max error={100*stage['max_relative_l2_error']:.3f}%, cosine={stage['mean_velocity_cosine']:.6f}, "
          f"intact replay max error={100*stage['max_intact_replay_relative_error']:.6f}%")
print("Teacher PNG matches prior activation PNG:", summary["teacher_png_matches_activation_count"], "/5")
print(f"Saved reports and bypass tensors: {run_dir}")
print(f"Saved reusable teacher inputs, prefix caches, control predictions and velocities: {targets_dir}")
print("These percentages measure velocity error, not image-quality loss; the removal decision remains pending.")

comparison_path = run_dir / "comparison_with_block_02.json"
comparison_identity = {"target_key": target_key, "block_02_summary_sha256": hashlib.sha256(baseline_summary_bytes).hexdigest(),
                       "block_03_report_hashes": report_hashes, "block_03_request_key": request_key}
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    if comparison["identity"] != comparison_identity:
        raise RuntimeError("Saved comparison differs from its source results.")
else:
    paired = []
    for row in rows:
        a, b = baseline_reports[row["eval_id"]], reports[row["eval_id"]]
        if a["targets_sha256"] != b["targets_sha256"]:
            raise RuntimeError("The two candidates used different teacher targets.")
        for first, second in zip(a["measurements"], b["measurements"]):
            if (first["stage"] != second["stage"] or first["step_index"] != second["step_index"]
                    or first["intact_replay_control"] != second["intact_replay_control"]):
                raise RuntimeError("Candidate comparisons differ in steps or controls.")
            paired.append({"eval_id": row["eval_id"], "dimension": row["dimension"], "stage": first["stage"],
                           "block_02_error": first["bypass"]["relative_l2_error"],
                           "block_03_error": second["bypass"]["relative_l2_error"]})
    comparison = {"identity": comparison_identity, "paired": paired,
                  "block_03_lower_error_count": sum(v["block_03_error"] < v["block_02_error"] for v in paired),
                  "n_comparisons": len(paired), "stages": [
                      {"stage": a["stage"], "block_02_mean_error": a["mean_relative_l2_error"],
                       "block_03_mean_error": b["mean_relative_l2_error"],
                       "block_02_max_error": a["max_relative_l2_error"], "block_03_max_error": b["max_relative_l2_error"]}
                      for a, b in zip(baseline_summary["stages"], summary["stages"])]}
    with comparison_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(comparison, indent=2) + "\n")
print("Block 3 has lower error than block 2 in", comparison["block_03_lower_error_count"], "/15 paired comparisons.")
for stage in comparison["stages"]:
    print(f"  {stage['stage']}: block 2={100*stage['block_02_mean_error']:.3f}%, block 3={100*stage['block_03_mean_error']:.3f}%")


comparison_path = run_dir / "comparison_with_block_04.json"
comparison_identity = {"target_key": target_key, "block_04_summary_sha256": hashlib.sha256(block4_summary_bytes).hexdigest(),
                       "block_03_report_hashes": report_hashes, "block_03_request_key": request_key}
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    if comparison["identity"] != comparison_identity:
        raise RuntimeError("Saved comparison differs from its source results.")
else:
    paired = []
    for row in rows:
        a, b = block4_reports[row["eval_id"]], reports[row["eval_id"]]
        if a["targets_sha256"] != b["targets_sha256"]:
            raise RuntimeError("The two candidates used different teacher targets.")
        for first, second in zip(a["measurements"], b["measurements"]):
            if (first["stage"] != second["stage"] or first["step_index"] != second["step_index"]
                    or first["intact_replay_control"] != second["intact_replay_control"]):
                raise RuntimeError("Candidate comparisons differ in steps or controls.")
            paired.append({"eval_id": row["eval_id"], "dimension": row["dimension"], "stage": first["stage"],
                           "block_04_error": first["bypass"]["relative_l2_error"],
                           "block_03_error": second["bypass"]["relative_l2_error"]})
    comparison = {"identity": comparison_identity, "paired": paired,
                  "block_03_lower_error_count": sum(v["block_03_error"] < v["block_04_error"] for v in paired),
                  "n_comparisons": len(paired), "stages": [
                      {"stage": a["stage"], "block_04_mean_error": a["mean_relative_l2_error"],
                       "block_03_mean_error": b["mean_relative_l2_error"],
                       "block_04_max_error": a["max_relative_l2_error"], "block_03_max_error": b["max_relative_l2_error"]}
                      for a, b in zip(block4_summary["stages"], summary["stages"])]}
    with comparison_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(comparison, indent=2) + "\n")
print("Block 3 has lower error than block 4 in", comparison["block_03_lower_error_count"], "/15 paired comparisons.")
for stage in comparison["stages"]:
    print(f"  {stage['stage']}: block 4={100*stage['block_04_mean_error']:.3f}%, block 3={100*stage['block_03_mean_error']:.3f}%")


comparison_path = run_dir / "comparison_with_block_05.json"
comparison_identity = {"target_key": target_key, "block_05_summary_sha256": hashlib.sha256(block5_summary_bytes).hexdigest(),
                       "block_03_report_hashes": report_hashes, "block_03_request_key": request_key}
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    if comparison["identity"] != comparison_identity:
        raise RuntimeError("Saved comparison differs from its source results.")
else:
    paired = []
    for row in rows:
        a, b = block5_reports[row["eval_id"]], reports[row["eval_id"]]
        if a["targets_sha256"] != b["targets_sha256"]:
            raise RuntimeError("The two candidates used different teacher targets.")
        for first, second in zip(a["measurements"], b["measurements"]):
            if (first["stage"] != second["stage"] or first["step_index"] != second["step_index"]
                    or first["intact_replay_control"] != second["intact_replay_control"]):
                raise RuntimeError("Candidate comparisons differ in steps or controls.")
            paired.append({"eval_id": row["eval_id"], "dimension": row["dimension"], "stage": first["stage"],
                           "block_05_error": first["bypass"]["relative_l2_error"],
                           "block_03_error": second["bypass"]["relative_l2_error"]})
    comparison = {"identity": comparison_identity, "paired": paired,
                  "block_03_lower_error_count": sum(v["block_03_error"] < v["block_05_error"] for v in paired),
                  "n_comparisons": len(paired), "stages": [
                      {"stage": a["stage"], "block_05_mean_error": a["mean_relative_l2_error"],
                       "block_03_mean_error": b["mean_relative_l2_error"],
                       "block_05_max_error": a["max_relative_l2_error"], "block_03_max_error": b["max_relative_l2_error"]}
                      for a, b in zip(block5_summary["stages"], summary["stages"])]}
    with comparison_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(comparison, indent=2) + "\n")
print("Block 3 has lower error than block 5 in", comparison["block_03_lower_error_count"], "/15 paired comparisons.")
for stage in comparison["stages"]:
    print(f"  {stage['stage']}: block 5={100*stage['block_05_mean_error']:.3f}%, block 3={100*stage['block_03_mean_error']:.3f}%")


✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-A7Z3TLDrxVqUsdWLC9g8wh
✓ Created objects.
└── 🔨 Created function paired_bypass.

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]


Loading weights:   1%|▏         | 10/750 [00:00<00:09, 82.17it/s]


Loading weights:   3%|▎         | 19/750 [00:00<00:11, 62.46it/s]


Loading weights:   4%|▎         | 28/750 [00:00<00:10, 68.16it/s]


Loading weights:   5%|▍         | 36/750 [00:00<00:10, 65.35it/s]


Loading weights:   6%|▌         | 43/750 [00:00<00:11, 59.17it/s]


Loading weights:   7%|▋         | 50/750 [00:00<00:11, 58.40it/s]


Loading weights:   8%|▊         | 61/750 [00:00<00:09, 69.38it/s]


Loading weights:   9%|▉         | 70/750 [00:01<00:09, 72.99it/s]


Loading weights:  10%|█         | 78/750 [00:01<00:11, 60.92it/s]


Loading weights:  11%|█▏        | 85/750 [00:01<00:10, 62.12it/s]


Loading weights:  12%|█▏        | 9

### Reading the block-3 results and the completed candidate pilot

We have now tested each of the four initial candidates individually on the same five teacher trajectories, at three sampled stages. The table reports mean relative velocity prediction error. Lower means that skipping the block changes the teacher's prediction less at those inputs.

| Stage | Block 2 | Block 3 | Block 4 | Block 5 | Block 3 worst prompt |
|---|---:|---:|---:|---:|---:|
| Early | 16.41% | 15.80% | 10.81% | 12.74% | 18.53% |
| Middle | 4.15% | 3.44% | 2.69% | 2.81% | 3.93% |
| Final | 18.96% | 11.85% | 5.43% | 3.69% | 13.79% |

The percentages describe the norm of the prediction difference relative to the teacher prediction, averaged over five prompts. They are not image-quality scores. The worst-prompt column is the largest observed block-3 error at that stage. The velocity explanation above the first bypass experiment describes how these predictions drive the scheduler's updates.

Block 3 has lower error than block 2 in **13/15** matched comparisons.

Block 3 has lower error than block 4 in **1/15** matched comparisons.

Block 3 has lower error than block 5 in **1/15** matched comparisons.

The lowest stage-wise mean errors come from block 4 at the early stage, block 4 at the middle stage, block 5 at the final stage.

**Interpretation:** blocks 4 and 5 remain the stronger candidates in this batch. Block 3 has higher error than each of them in 14 of 15 matched comparisons and has a higher mean error at all three sampled stages. Although block 3 improves on block 2 in 13 of 15 comparisons, that does not make it the preferred candidate. Block 4 gives the lowest early/middle mean error; block 5 gives the lowest final-stage mean error. This supports prioritizing blocks 4 and 5 for further assessment while keeping all four candidates' measurements.

A ranking can differ by stage and prompt. We should retain the per-prompt results and consider difficult cases rather than selecting a block from a single global average. The four interventions were tested separately; the results do not establish that skipping any combination is acceptable.

For block 3, the mean velocity cosines are 0.9878 at early, 0.9994 at middle, 0.9931 at final. Cosine measures directional agreement. Values near 1 can still coexist with changes in prediction magnitude, so relative error is also needed.

**Validation:** all five block-3 reports passed and confirm teacher-target reuse. The saved intact replay controls match the teacher predictions exactly in all 15 cases. The bypass prediction shapes match their references and contain finite values. Block 3's prefix-cache entry is empty, while all retained entries are populated. A cache-only rerun made no Modal call, imported no model code, recalculated no summary and changed no saved artifact. “Passed” describes experiment execution and validation, not approval of image quality.

The printed “4/5 teacher PNG matches” is the existing comparison of the shared teacher images from the block-2 run with the older activation pilot. No teacher images were regenerated here. The previously documented real-world image mismatch remains unresolved, but all four candidates use the same saved reference predictions.

**What remains uncertain:** identity simply omits the missing block's update. It tells us how sensitive the full model's prediction is to that omission at teacher inputs. A learned bridge would approximate the update and may alter the ranking. No bridge was trained in this pilot. Full generation can accumulate errors after it departs from the teacher trajectory; finished-image quality, speed savings, generalization beyond five prompts and interactions between replacements remain untested. These results support candidate prioritization, not an irreversible removal decision.

Block-3 predictions, caches, reports and the three paired comparisons are saved in `data/phase3/bypass/block_03_pilot/`. The earlier candidate directories and shared teacher targets remain available for further analysis.


### Step 13 — Compare full-generation images for blocks 4 and 5

We generate two independent variants per pilot prompt: block 4 bypassed throughout generation, and block 5 bypassed throughout generation. Each starts from the original CPU seed and uses the frozen model revision, resolution, scheduler, 40 steps and conditioning settings. Each variant evolves its own latent and builds its own prefix cache. The saved intact teacher images are reused; no teacher regeneration is allowed.

We save all ten full-resolution images and reports locally and on Modal. Locally we derive five side-by-side sheets (teacher, skip block 4, skip block 5) and pixel-similarity metrics. Completed reruns read verified caches without GPU calls or recalculating metrics. RGB differences quantify image changes, not aesthetic quality or prompt adherence; the accompanying explanation and visual comparisons support manual review. Timing is recorded for provenance but this pilot provides no matched teacher timing benchmark.


In [1]:
import hashlib
import json
from pathlib import Path

root = Path('data/phase3')
source_dir = root / 'bypass/block_02_pilot'
source_manifest = json.loads((source_dir / 'request.json').read_text(encoding='utf-8'))
target = source_manifest['request']['target_request']
target_key = source_manifest['request']['target_key']
rows = target['rows']
teacher_dir = root / 'bypass/teacher_targets' / target_key
teacher_hashes = {}
candidate_sources = {}
for layer in (2, 4, 5):
    directory = root / f'bypass/block_{layer:02d}_pilot'
    manifest = json.loads((directory / 'request.json').read_text(encoding='utf-8'))
    summary_bytes = (directory / 'summary.json').read_bytes()
    summary = json.loads(summary_bytes)
    if manifest['request']['target_key'] != target_key or summary['request_key'] != manifest['request_key']:
        raise RuntimeError('Candidate pilot provenance differs from teacher targets.')
    candidate_sources[str(layer)] = hashlib.sha256(summary_bytes).hexdigest()
    for row in rows:
        raw = (directory / f"{row['eval_id']}.json").read_bytes()
        if hashlib.sha256(raw).hexdigest() != summary['report_hashes'][row['eval_id']]:
            raise RuntimeError('Candidate pilot report changed.')
        report = json.loads(raw)
        if report['status'] != 'passed':
            raise RuntimeError('Candidate pilot did not pass.')
        name = row['eval_id']
        if name in teacher_hashes and teacher_hashes[name] != report['teacher_image_sha256']:
            raise RuntimeError('Candidates reference different teacher images.')
        teacher_hashes[name] = report['teacher_image_sha256']
        if hashlib.sha256((teacher_dir / f'{name}.png').read_bytes()).hexdigest() != teacher_hashes[name]:
            raise RuntimeError('Saved teacher image changed or is missing; no regeneration allowed.')

request = {'version': 'phase3-bypass-full-images-v1', 'target_request': target, 'target_key': target_key,
           'bypassed_layers': [4, 5], 'teacher_image_sha256': teacher_hashes,
           'candidate_summary_sha256': candidate_sources,
           'scope': 'independent single-block bypasses over all 40 steps; own latents and prefix caches; no bridge',
           'image_metrics': 'RGB8 float64 MAE, RMSE and PSNR with range 255; differences are not quality scores',
           'gallery': 'teacher, block 4, block 5; thumbnails height 320; preserve aspect ratios'}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = root / 'bypass/full_images_pilot'
run_dir.mkdir(parents=True, exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding='utf-8')) != manifest:
        raise RuntimeError('Image comparison inputs changed; preserve existing results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')

jobs = [{'row': row, 'layer': layer} for row in rows for layer in request['bypassed_layers']]
def stem(job):
    return f"{job['row']['eval_id']}-block-{job['layer']:02d}"
def validate(report, job):
    if report.get('request_key') != key or report.get('job') != job:
        raise RuntimeError('Image report has different inputs.')
    if report['status'] != 'passed':
        raise RuntimeError('Saved image generation failed; no automatic retry: ' + report['error'])
    if report['bypass_calls'] != job['row']['num_inference_steps']:
        raise RuntimeError('Not all steps bypassed the selected block.')
    if report['teacher_image_sha256'] != teacher_hashes[job['row']['eval_id']]:
        raise RuntimeError('Image comparison references a different teacher.')

reports = {}
missing = []
for job in jobs:
    path = run_dir / f'{stem(job)}.json'
    image_path = run_dir / f'{stem(job)}.png'
    if path.exists():
        report = json.loads(path.read_text(encoding='utf-8'))
        validate(report, job)
        if image_path.exists():
            if hashlib.sha256(image_path.read_bytes()).hexdigest() != report['image_sha256']:
                raise RuntimeError('Saved bypass image changed.')
            reports[stem(job)] = report
            continue
    missing.append(job)

if missing:
    import modal
    runtime = target['runtime']
    volume = modal.Volume.from_name(runtime['results_volume'])
    remote_dir = f'/bypass/full_images/{key}'
    def receive(report, job):
        name = stem(job)
        if report.get('request_key') != key or report.get('job') != job:
            raise RuntimeError('Remote image result has different inputs.')
        local_report = run_dir / f'{name}.json'
        if report['status'] == 'passed':
            local_image = run_dir / f'{name}.png'
            if not local_image.exists():
                temporary = local_image.with_suffix('.tmp')
                with temporary.open('wb') as handle:
                    for chunk in volume.read_file(f'{remote_dir}/{name}.png'):
                        handle.write(chunk)
                if hashlib.sha256(temporary.read_bytes()).hexdigest() != report['image_sha256']:
                    raise RuntimeError('Downloaded image checksum differs.')
                temporary.replace(local_image)
            elif hashlib.sha256(local_image.read_bytes()).hexdigest() != report['image_sha256']:
                raise RuntimeError('Existing image differs from remote result.')
        if local_report.exists():
            if json.loads(local_report.read_text(encoding='utf-8')) != report:
                raise RuntimeError('Local report differs from remote backup.')
        else:
            with local_report.open('x', encoding='utf-8') as handle:
                handle.write(json.dumps(report, indent=2) + '\n')
        validate(report, job)
        return report
    try:
        names = {Path(entry.path).name for entry in volume.listdir(remote_dir, recursive=False)}
    except (FileNotFoundError, modal.exception.NotFoundError):
        names = set()
    pending = []
    for job in missing:
        name = stem(job)
        if f'{name}.json' in names:
            report = json.loads(b''.join(volume.read_file(f'{remote_dir}/{name}.json')))
            reports[name] = receive(report, job)
        else:
            if (run_dir / f'{name}.json').exists() or (run_dir / f'{name}.png').exists() or f'{name}.png' in names:
                raise RuntimeError('Incomplete image artifacts have no report backup; preserve and inspect.')
            pending.append(job)
    if pending:
        app = modal.App('nihonga-phase3-bypass-images')
        weights = modal.Volume.from_name(runtime['weights_volume'])
        image = (modal.Image.debian_slim(python_version=runtime['python_version'])
                 .apt_install('git').pip_install(*runtime['packages'])
                 .env({'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}))
        @app.function(image=image, gpu=runtime['gpu_requested'], serialized=True, timeout=1800, max_containers=1,
                      volumes={'/root/.cache/huggingface': weights.with_mount_options(read_only=True), '/phase3': volume})
        def generate_bypassed_images(expected, request_key, tasks):
            import hashlib
            import inspect
            import json
            import time
            from datetime import datetime, timezone
            from pathlib import Path
            import torch
            from diffusers import QwenImage21Pipeline
            from PIL import Image

            config = expected['target_request']
            directory = Path('/phase3/bypass/full_images') / request_key
            shared = Path('/phase3/bypass/targets') / expected['target_key']
            directory.mkdir(parents=True, exist_ok=True)
            def save_json(path, value):
                temporary = path.with_suffix('.tmp')
                temporary.write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')
                temporary.replace(path)
            saved_manifest = {'request': expected, 'request_key': request_key}
            manifest_path = directory / 'request.json'
            if manifest_path.exists():
                if json.loads(manifest_path.read_text()) != saved_manifest:
                    raise RuntimeError('Remote image request differs.')
            else:
                save_json(manifest_path, saved_manifest)
                volume.commit()
            load_error = None
            try:
                if 'H100' not in torch.cuda.get_device_name(0):
                    raise RuntimeError('Allocated GPU differs from H100.')
                teacher_manifest = {'request': config, 'request_key': expected['target_key']}
                if json.loads((shared / 'request.json').read_text()) != teacher_manifest:
                    raise RuntimeError('Shared teacher identity changed.')
                pipe = QwenImage21Pipeline.from_pretrained(config['model_repo'], revision=config['model_revision'],
                    dtype=getattr(torch, config['dtype']), local_files_only=True).to('cuda')
                pipe.set_progress_bar_config(disable=True)
                pipe.transformer.eval()
                source = Path(inspect.getfile(type(pipe.transformer)))
                if hashlib.sha256(source.read_bytes()).hexdigest() != config['transformer_source_sha256']:
                    raise RuntimeError('Transformer source differs from the pilot.')
                if (len(pipe.transformer.transformer_blocks) != config['num_layers'] or not pipe.transformer.config.causal_condition
                        or type(pipe.scheduler).__name__ != config['scheduler_class']
                        or json.loads(json.dumps(dict(pipe.scheduler.config), default=str)) != config['scheduler_config']):
                    raise RuntimeError('Architecture or scheduler differs from the baseline.')
            except Exception as error:
                load_error = f'{type(error).__name__}: {error}'
            for job in tasks:
                row, layer = job['row'], job['layer']
                name = f"{row['eval_id']}-block-{layer:02d}"
                report_path = directory / f'{name}.json'
                if report_path.exists():
                    yield json.loads(report_path.read_text())
                    continue
                report = {'request_key': request_key, 'job': job, 'status': 'failed',
                          'recorded_utc': datetime.now(timezone.utc).isoformat()}
                block = None
                original = None
                own_forward = False
                try:
                    if load_error:
                        raise RuntimeError(load_error)
                    teacher_path = shared / f"{row['eval_id']}.png"
                    digest = hashlib.sha256(teacher_path.read_bytes()).hexdigest()
                    if digest != expected['teacher_image_sha256'][row['eval_id']]:
                        raise RuntimeError('Remote teacher image differs; regeneration is disabled.')
                    block = pipe.transformer.transformer_blocks[layer]
                    own_forward = 'forward' in block.__dict__
                    original = block.forward
                    calls = {'count': 0}
                    def identity_forward(*args, **kwargs):
                        calls['count'] += 1
                        return kwargs['hidden_states'] if 'hidden_states' in kwargs else args[0]
                    block.forward = identity_forward
                    torch.cuda.synchronize()
                    torch.cuda.reset_peak_memory_stats()
                    started = time.perf_counter()
                    with torch.inference_mode():
                        generated = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                            num_inference_steps=row['num_inference_steps'], true_cfg_scale=row['true_cfg_scale'],
                            generator=torch.Generator(config['noise_device']).manual_seed(row['seed']),
                            use_kv_cache=True).images[0]
                    torch.cuda.synchronize()
                    elapsed = time.perf_counter() - started
                    if calls['count'] != row['num_inference_steps'] or generated.size != (row['width'], row['height']):
                        raise RuntimeError('Unexpected bypass call count or image dimensions.')
                    image_path = directory / f'{name}.png'
                    temporary = image_path.with_suffix('.tmp')
                    generated.save(temporary, format='PNG')
                    temporary.replace(image_path)
                    with Image.open(teacher_path) as teacher:
                        if teacher.size != generated.size:
                            raise RuntimeError('Teacher image dimensions differ.')
                    report.update(status='passed', bypass_calls=calls['count'], image_sha256=hashlib.sha256(image_path.read_bytes()).hexdigest(),
                                  teacher_image_sha256=digest, image_size=list(generated.size), gpu=torch.cuda.get_device_name(0),
                                  generation_seconds=elapsed, peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,
                                  peak_reserved_gib=torch.cuda.max_memory_reserved()/2**30,
                                  teacher_regenerated=False, scope='own full trajectory; individual identity bypass')
                except Exception as error:
                    report['error'] = f'{type(error).__name__}: {error}'
                finally:
                    if block is not None and original is not None:
                        if own_forward:
                            block.forward = original
                        else:
                            del block.forward
                save_json(report_path, report)
                volume.commit()
                print(f"Full image: {row['dimension']}, block {layer} ({report['status']}).")
                yield report
                if report['status'] != 'passed':
                    break
        with modal.enable_output(), app.run():
            by_name = {stem(job): job for job in jobs}
            for report in generate_bypassed_images.remote_gen(request, key, pending):
                job = report['job']
                reports[stem(job)] = receive(report, by_name[stem(job)])
                print(f'Saved bypass images: {len(reports)}/10.')
else:
    print('Reusing ten verified local bypass images; no Modal call.')
if len(reports) != 10:
    raise RuntimeError('Image comparison is incomplete; completed artifacts remain saved.')

summary_path = run_dir / 'summary.json'
report_hashes = {stem(job): hashlib.sha256((run_dir / f'{stem(job)}.json').read_bytes()).hexdigest() for job in jobs}
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    if summary['request_key'] != key or summary['report_hashes'] != report_hashes:
        raise RuntimeError('Saved image summary differs from its sources.')
    for name, digest in summary['gallery_sha256'].items():
        if hashlib.sha256((run_dir / name).read_bytes()).hexdigest() != digest:
            raise RuntimeError('Saved gallery changed.')
else:
    import math
    import numpy as np
    from PIL import Image, ImageDraw
    measurements = []
    galleries = {}
    for row in rows:
        name = row['eval_id']
        with Image.open(teacher_dir / f'{name}.png') as im:
            teacher = im.convert('RGB')
        reference = np.asarray(teacher, dtype=np.float64)
        panels = [teacher]
        for layer in (4, 5):
            job = {'row': row, 'layer': layer}
            with Image.open(run_dir / f'{stem(job)}.png') as im:
                predicted = im.convert('RGB')
            panels.append(predicted)
            delta = np.asarray(predicted, dtype=np.float64) - reference
            mse = float(np.mean(delta * delta))
            measurements.append({'eval_id': name, 'dimension': row['dimension'], 'layer': layer,
                                 'rgb8_mae': float(np.mean(np.abs(delta))), 'rgb8_rmse': math.sqrt(mse),
                                 'psnr_db': 10*math.log10(255**2/mse) if mse else None,
                                 'identical_pixels': bool(mse == 0), 'generation_seconds': reports[stem(job)]['generation_seconds']})
        width = round(row['width'] * 320 / row['height'])
        canvas = Image.new('RGB', (3*width + 32, 360), '#ffffff')
        draw = ImageDraw.Draw(canvas)
        for i, (label, panel) in enumerate(zip(('Teacher (32 blocks)', 'Skip block 4', 'Skip block 5'), panels)):
            x = i*(width+16)
            draw.text((x+8, 10), label, fill='#111111')
            canvas.paste(panel.resize((width,320), Image.Resampling.LANCZOS), (x,32))
        filename = f'{name}-comparison.png'
        path = run_dir / filename
        if path.exists():
            with Image.open(path) as saved:
                if saved.size != canvas.size or saved.convert('RGB').tobytes() != canvas.tobytes():
                    raise RuntimeError('Existing gallery differs from the derived comparison.')
        else:
            canvas.save(path)
        galleries[filename] = hashlib.sha256(path.read_bytes()).hexdigest()
    summary = {'request_key': key, 'report_hashes': report_hashes, 'gallery_sha256': galleries,
               'n_prompts': 5, 'n_generated_images': 10, 'teacher_regenerated': False, 'measurements': measurements,
               'interpretation': 'pixel differences measure similarity to saved teacher images; no quality scores or speedup claim'}
    with summary_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(summary, indent=2) + '\n')
print('Full trajectories: 10 bypass images, 5 cached teacher images, 5 saved comparison sheets.')
for m in summary['measurements']:
    psnr = f"{m['psnr_db']:.2f}" if m['psnr_db'] is not None else 'identical'
    print(f"  {m['dimension']:14s} block {m['layer']}: RGB MAE={m['rgb8_mae']:.2f}/255, RMSE={m['rgb8_rmse']:.2f}/255, PSNR={psnr} dB")
print(f'Saved full-resolution images and comparisons: {run_dir}')
print('Pixel similarity is not image quality. Review the teacher/block-4/block-5 sheets below before selecting a replacement.')


✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-MbLvhhGhb66BEuq3bQ8o0U
✓ Created objects.
└── 🔨 Created function generate_bypassed_images.

Loading pipeline components...:  40%|████      | 2/5 [00:01<00:02,  1.35it/s]


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]


Loading weights:   1%|          | 9/750 [00:00<00:09, 80.57it/s]


Loading weights:   2%|▏         | 18/750 [00:00<00:20, 35.08it/s]


Loading weights:   3%|▎         | 23/750 [00:00<00:19, 36.59it/s]


Loading weights:   4%|▍         | 30/750 [00:00<00:16, 44.15it/s]


Loading weights:   5%|▍         | 36/750 [00:00<00:15, 46.67it/s]


Loading weights:   6%|▌         | 42/750 [00:00<00:14, 48.86it/s]

Loading weights:   6%|▋         | 48/750 [00:01<00:18, 38.33it/s]


Loading weights:   7%|▋         | 53/750 [00:01<00:18, 37.53it/s]

Loading weights:  18%|█▊        | 134/750 [00:01<00:03, 202.75it/s]


Loading weights:  21%|██        | 159/750 [00:02<00:05, 99.98it/s] 


Loading weights

### Reading the full-generation image comparison

**The main finding is a prompt-adherence failure when block 4 is omitted: the astronaut becomes an ordinary skateboarder.** Block 5 preserves the astronaut in that example. Block 4 had lower mean velocity error at the early and middle teacher inputs, yet that diagnostic did not predict this semantic failure. Candidate selection therefore needs finished-image checks as well as numerical prediction comparisons.

Ten images were generated: five prompts with block 4 skipped and the same five with block 5 skipped. Each bypass evolved its own latent over all 40 steps, starting from the original seed and using the frozen generation settings. The five intact teacher images were reused. These runs test identity bypass, not a trained replacement bridge.

**How to read the pixel metrics:** RGB MAE is the average absolute difference between corresponding color-channel values on a 0–255 scale. RMSE weights larger differences more heavily. PSNR is derived from squared pixel error using a maximum channel value of 255; higher PSNR means closer pixel agreement. None measures aesthetic quality or prompt adherence. A changed pose, background or object position can cause large pixel differences even when an image satisfies its prompt.

| Prompt capability | Block 4 MAE /255 | Block 5 MAE /255 | Block 4 PSNR | Block 5 PSNR |
|---|---:|---:|---:|---:|
| aesthetics | 7.36 | 8.68 | 26.89 dB | 26.19 dB |
| alignment | 34.34 | 30.35 | 13.90 dB | 14.44 dB |
| creative | 11.78 | 15.95 | 22.30 dB | 21.06 dB |
| real_world | 21.71 | 20.10 | 15.42 dB | 14.85 dB |
| visual_quality | 21.13 | 22.99 | 18.03 dB | 17.77 dB |

Block 4 is closer in both displayed pixel metrics for the meadow, creature and books. Block 5 is closer for the skateboarder. For the kettle, MAE favors block 5 while PSNR favors block 4, illustrating that similarity metrics can disagree. A count of metric wins does not override the observed astronaut failure.

**Provisional interpretation:** prioritize block 5 for a learned-bridge pilot, retaining block 4 as a comparator. This preference reflects the observed prompt-adherence failure for block 4, not a claim that block 5 has proven equivalent quality. Block 5 also changes generated content, and five prompts with one seed each provide limited evidence. Neither identity bypass is accepted as the final replacement. A learned bridge must approximate the missing update and be evaluated again; its performance may change this ranking.

**Validation and limits:** all ten image reports passed, with 40 identity calls each. Saved full-resolution PNGs decode correctly and match the requested dimensions. Model revision, transformer implementation, scheduler, dtype and H100 GPU were checked. Teacher images were not regenerated. A cache-only rerun used saved reports, metrics and galleries without a Modal call or any artifact change. Timings are recorded but are not a matched speedup benchmark against the teacher.

The shared teacher images came from an earlier run. The earlier real-world teacher repeatability check had a mean RGB-channel difference of about 0.187/255 against the older activation image; its cause remains unresolved. This run has no newly generated intact control, so it cannot quantify how much of its pixel difference comes from cross-run variation. The galleries and qualitative observations should be read with that limitation; the pixel metrics alone do not establish causality or image quality.

The observations below come from manual inspection of comparison thumbnails, without a blinded study or automatic quality judge. Full-resolution originals remain saved for closer review. Each sheet uses **teacher / skip block 4 / skip block 5**, left to right.

**aesthetics** — Prompt: Generate an image of an alpine flower meadow. Style: candid photo, natural light.

All three images retain the flower-meadow scene. Block 4 appears closer to the teacher composition; block 5 changes prominent flower stems and placement. This is a similarity observation, not a rated aesthetic preference.

![Teacher and the two individual bypasses for aesthetics](data/phase3/bypass/full_images_pilot/internal-p1-e0d8dd3b84cd-comparison.png)

**alignment** — Prompt: An astronaut skateboarding down a ramp at a skate park, digital painting

The teacher depicts an astronaut skateboarding. With block 4 skipped, the rider wears ordinary clothing rather than a spacesuit: the astronaut requirement is lost. With block 5 skipped, the spacesuit and helmet remain, although the pose and background change. This is a concrete prompt-adherence difference in this one example.

![Teacher and the two individual bypasses for alignment](data/phase3/bypass/full_images_pilot/internal-p1-5bec387a9a76-comparison.png)

**creative** — Prompt: 一只小型狐狸状生物，带有水晶鹿角，3D渲染

English rendering: a small fox-like creature with crystal antlers, 3D render.

All three retain an animal-like creature with crystal antlers. Block 4 changes facial proportions. Block 5 changes the creature more visibly, including fur color, proportions and pose. Preserving the broad concept does not establish fidelity to the teacher or overall creative quality.

![Teacher and the two individual bypasses for creative](data/phase3/bypass/full_images_pilot/internal-p1-4a1cdb170e44-comparison.png)

**real_world** — Prompt: A movie still of an electric kettle on a white background.

All three show a kettle against a white background, but the design changes substantially. Block 4 produces a taller metal design with a water-level window; block 5 produces a cream-colored design. The prompt does not specify the teacher kettle design, so these differences alone are not proof of worse prompt adherence.

![Teacher and the two individual bypasses for real_world](data/phase3/bypass/full_images_pilot/internal-p1-425ec4e55730-comparison.png)

**visual_quality** — Prompt: 一张电影剧照，一堆旧书

English rendering: a movie still of a pile of old books.

All three depict old books. Both bypasses alter stack arrangement, book placement and surface details. Thumbnail inspection does not establish which is aesthetically better or rule out defects at full resolution.

![Teacher and the two individual bypasses for visual_quality](data/phase3/bypass/full_images_pilot/internal-p1-6177a6812fbc-comparison.png)

All ten full-resolution images, reports, metrics, comparison sheets and the saved qualitative observations (`visual_review.json`) are in `data/phase3/bypass/full_images_pilot/`. The reference teacher images remain in the shared teacher-target directory.


### Candidate policy for learned replacements

Blocks **2, 3, 4 and 5 all remain candidates for learned bridges**. The identity experiments quantify the effect of discarding each block's update; they do not measure how well a cheap learned function could reproduce that update. The astronaut failure for block 4 rules out accepting its tested identity bypass, but does not disqualify a learned replacement. The earlier preference for block 5 is a diagnostic observation, not an exclusion rule.

Other blocks are not permanently excluded either; these four form our initial measured comparison batch. Every candidate must earn acceptance through held-out prediction and finished-image checks after bridge fitting, with its compute savings measured. Successfully approximating a block is a question to test, not a guarantee.

Bridge definitions and independent initializations are recorded in `phase4-insert-bridge-modules.ipynb`. Training uses the Phase 1 training split, with validation for model selection. The Phase 3 evaluation prompts are not bridge-training examples.
